# SynerPlant Figure Extraction v1.2 — queue-aligned, provenance-first

This notebook is a downstream **candidate-generation and figure-reconciliation layer**. It does not write to the master database. It processes only figure items already placed in the upstream `figure_digitization_queue.csv` and/or `HUMAN_REVIEW_QUEUE.v4_3.csv`, then reconciles them against an immutable snapshot of `TFM_ML_DATABASE_V3` before digitization.

Scientific source hierarchy: **machine-readable supplement / native table / explicit PDF text > calibrated figure digitization > unresolved/manual review**. An existing exact value is never intentionally replaced by a pixel estimate. Items already represented by supported figure provenance are routed to `RECONCILE_FIRST` instead of automatic digitization.

The automatic extractor remains candidate-only. Unresolved panels can be sent to **WebPlotDigitizer**; the notebook exports a reproducible work package and imports WPD CSV results only when the operator, WPD version, source CSV hash and saved WPD project hash are recorded. Final database eligibility still requires explicit candidate-level review and semantic mapping.


In [ ]:
# ============================================================
# 1. Dependencies: validate installed versions; do not silently upgrade
# ============================================================
import importlib.util
import importlib.metadata
import subprocess
import sys

REQUIRED = {
    "cv2": "opencv-python>=4.10",
    "fitz": "PyMuPDF>=1.26",
    "numpy": "numpy>=1.26",
    "pandas": "pandas>=2.2",
    "PIL": "Pillow>=10",
    "matplotlib": "matplotlib>=3.8",
    "packaging": "packaging>=23",
    "openpyxl": "openpyxl>=3.1",
}
AUTO_INSTALL_MISSING = True
missing = [spec for mod, spec in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    if not AUTO_INSTALL_MISSING:
        raise RuntimeError("Missing dependencies: " + ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()

from packaging.requirements import Requirement

def check_dependency_versions(required):
    aliases = {"opencv-python": ("opencv-python", "opencv-python-headless", "opencv-contrib-python", "opencv-contrib-python-headless")}
    bad, versions = [], {}
    for spec in required.values():
        req = Requirement(spec)
        installed = None
        for dist in aliases.get(req.name, (req.name,)):
            try:
                installed = importlib.metadata.version(dist)
                break
            except importlib.metadata.PackageNotFoundError:
                continue
        versions[req.name] = installed
        if installed is None or not req.specifier.contains(installed, prereleases=True):
            bad.append(f"{req.name}: installed={installed}; required={req.specifier}")
    return versions, bad

DEPENDENCY_VERSIONS, incompatible = check_dependency_versions(REQUIRED)
if incompatible:
    print("\n".join(incompatible))
    raise RuntimeError("Dependency versions do not match. Update the listed packages, restart the kernel, and rerun. No automatic upgrades were performed.")
print("Python:", sys.version)
print("Dependency versions:", DEPENDENCY_VERSIONS)


In [ ]:
# ============================================================
# 2. Imports + project paths
# ============================================================
import base64
import bisect
import csv
import gc
import hashlib
import io
import json
import math
import os
import shutil
import platform
import re
import tempfile
import time
import traceback
import zlib
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import cv2
import fitz
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display

# Environment override is useful for testing / moving the corpus; the Windows path remains the default.
PROJECT_ROOT_DISPLAY = os.environ.get("SYNERPLANT_PROJECT_ROOT", r"G:/Мой диск/TFM_CORPUS_NORMALIZED")
PROJECT_ROOT = Path(PROJECT_ROOT_DISPLAY)

# Keep the same Windows long-path behavior as the Docling notebook.
def windows_extended_path_text(value):
    import ntpath
    value = ntpath.abspath(str(value))
    slash = chr(92)
    prefix = slash * 2 + "?" + slash
    if value.startswith(prefix):
        return value
    if value.startswith(slash * 2):
        return prefix + "UNC" + slash + value[2:]
    return prefix + value

if os.name == "nt":
    PROJECT_ROOT = Path(windows_extended_path_text(PROJECT_ROOT))

EXTRACTED_DIR = PROJECT_ROOT / "04_extracted_json"
EXPORTS_DIR = PROJECT_ROOT / "05_exports"
LOGS_DIR = PROJECT_ROOT / "06_pipeline_logs"

print("Computer:", platform.node())
print("System:", platform.system())
print("Python:", sys.version.split()[0])
print("Project:", PROJECT_ROOT)

if not PROJECT_ROOT.exists():
    raise RuntimeError(f"Project root is not visible: {PROJECT_ROOT}")
for p in [EXPORTS_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# 3. Configuration
# ============================================================

# Immutable upstream source layer.
SOURCE_PIPELINE_VERSION = "docling_local_sourceqa_v4_3_rerun_380"

# Independent downstream figure run. Start a new version if any scientific setting changes.
FIGURE_PIPELINE_VERSION = "figure_extraction_v1_2_queue_aligned"
FIGURE_EXTRACTOR_REVISION = "figure_project_20260927_v1_2_queue_aligned"
ENGINE_SOURCE_SHA256 = "00ae6721de74857f789b9ba3be8a68b4b0a5b6674542a9151ad0fa8ff52b124d"
DATABASE_SPEC = "TFM_ML_DATABASE_V3_v3.4.0_corpus_v1.5"

# Master database reference used only for duplicate/gap reconciliation.
# The embedded snapshot is self-contained and immutable for this figure-generation version.
MASTER_DB_REFERENCE_MODE = "EMBEDDED_SNAPSHOT"  # EMBEDDED_SNAPSHOT or EXTERNAL_XLSX
MASTER_DB_REFERENCE_XLSX = ""  # optional fresh export of TFM_ML_DATABASE_V3.xlsx; use a NEW figure version if changed
MASTER_DB_SNAPSHOT_EXPORTED_UTC = "2026-09-27"
MASTER_DB_SNAPSHOT_XLSX_SHA256 = "ee04544c65c8405b5665cbbd0d64b157d0fdabaf67d3a8c6df921c40cce7c4e2"
MASTER_DB_SNAPSHOT_CSV_SHA256 = {"measurements": "f6c8dcd211b5f2c8db9926c07ee2ee347efee5f1ac272e39247f912a1f65c650", "provenance_qa": "8f104ebca5220634592536f349b0b5fc5d4860262286077c891b86d4841cc28c", "extraction_queue_relevant": "1a84c723875b492f8f21cf57ac878644cc68d804568d5d606317a68b6674b21c"}

# BATCH = process selected queue items; AUDIT_EXISTING = read-only planning/checkpoint audit.
EXECUTION_MODE = "BATCH"

# QUEUE = all queue-aligned items. PILOT = only listed study IDs.
RUN_MODE = "QUEUE"
PILOT_STUDY_IDS = [
    "D_366d799918128863",
    "D_48f99795a500094e",
]

# Queue contract. Inventory-wide processing is intentionally disabled by default.
DISCOVERY_POLICY = "QUEUE_ALIGNED"
USE_HUMAN_REVIEW_QUEUE = True
ALLOW_LIKELY_QUANTITATIVE_FALLBACK = False
INCLUDE_RECONCILE_FIRST = False  # existing supported values: inspect rather than re-digitize automatically

# Classes to retain. Class A is sorted first, then B/D, C, E.
INCLUDE_SCREENING_CLASSES = {"A", "B", "C", "D", "E"}
PRIORITIZE_CLASS_A = True
MAX_FIGURES = None

# Resume / selective retry.
FORCE_REPROCESS = False
REPROCESS_ITEM_IDS: List[str] = []
REPROCESS_VERIFIED = False
HASH_POLICY = "ONCE"
MAX_STAGE_RETRIES = 1
FAIL_FAST = False

# Conservative image analysis.
AUTO_PANEL_SPLIT = True
MAX_PANELS = 24
AUTO_VECTOR_BAR_DIGITIZATION = True
AUTO_VECTOR_LINE_DIGITIZATION = True
AUTO_RASTER_BAR_DIGITIZATION = True
MIN_AXIS_CALIBRATION_R2 = 0.995
MIN_BAR_COUNT = 1
MAX_BAR_COUNT = 60

# Duplicate screening against article text/native tables remains a warning layer.
ENABLE_DUPLICATE_SCREEN = True
MAX_DUPCHECK_CELLS_PER_FILE = 50_000
MAX_DUPCHECK_VALUES_PER_STUDY = 120_000
DUPLICATE_REL_TOL = 0.02
DUPLICATE_ABS_TOL = 0.02

# Source PDF / OCR recovery.
RECOVER_FROM_SOURCE_PDF = True
PDF_RENDER_DPI = 288
MAX_WORKING_PIXELS = 6_000_000
MAX_OCR_ATLAS_PIXELS = 8_000_000
ENABLE_LOCAL_OCR = True
OCR_BACKEND = "AUTO"
TESSERACT_EXE = ""
OCR_LANGUAGE = "eng"
OCR_TIMEOUT_SECONDS = 90
OCR_MIN_CONFIDENCE = 0.35
OCR_PREPROCESSING = "AXIS_COLUMNS_ATLAS_V1"
MIN_AUTO_TICKS = 3
DECIMAL_COMMA = "AUTO"
AUTO_RASTER_LINE_DIGITIZATION = True
AUTO_SCATTER_DIGITIZATION = True
AUTO_ERRORBAR_ENDPOINTS = True
MAX_CURVE_POINTS_PER_SERIES = 200

# WebPlotDigitizer evidence contract. WPD imports are candidates until reviewed.
REQUIRE_WPD_PROJECT_FILE = True
WPD_TOOL_NAME = "WebPlotDigitizer"

# Master DB safety.
WRITE_BACK_TO_MASTER = False
assert WRITE_BACK_TO_MASTER is False

SOURCE_RUN_DIR = EXTRACTED_DIR / SOURCE_PIPELINE_VERSION
FIGURE_OUTPUT_DIR = EXPORTS_DIR / FIGURE_PIPELINE_VERSION
FIGURE_LOGS_DIR = LOGS_DIR / FIGURE_PIPELINE_VERSION
CHECKPOINT_DIR = FIGURE_LOGS_DIR / "checkpoints"
BATCH_REPORT_DIR = FIGURE_LOGS_DIR / "batch_reports"
HASH_CACHE_PATH = FIGURE_LOGS_DIR / "file_hash_cache.csv"
MANUAL_REVIEW_PATH = FIGURE_OUTPUT_DIR / "MANUAL_REVIEW_OVERRIDES.csv"
VERIFIED_EXPORT_PATH = FIGURE_OUTPUT_DIR / "VERIFIED_FIGURE_CANDIDATES.csv"
QUEUE_ALIGNMENT_PATH = FIGURE_OUTPUT_DIR / "QUEUE_ALIGNMENT.csv"

for p in [FIGURE_OUTPUT_DIR, FIGURE_LOGS_DIR, CHECKPOINT_DIR, BATCH_REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if not SOURCE_RUN_DIR.exists():
    raise RuntimeError(f"Docling source run is not visible: {SOURCE_RUN_DIR}")

print("Source layer:", SOURCE_RUN_DIR)
print("Figure pipeline:", FIGURE_PIPELINE_VERSION)
print("Figure output:", FIGURE_OUTPUT_DIR)
print("Execution mode:", EXECUTION_MODE)
print("Run mode:", RUN_MODE)
print("Discovery:", DISCOVERY_POLICY)
print("Force reprocess:", FORCE_REPROCESS)

CALIBRATION_DIR = FIGURE_OUTPUT_DIR / "manual_calibrations"
EDITOR_DIR = FIGURE_OUTPUT_DIR / "manual_editors"
OCR_CACHE_DIR = FIGURE_LOGS_DIR / "ocr_cache"
WPD_ROOT = FIGURE_OUTPUT_DIR / "webplotdigitizer"
WPD_INPUT_DIR = WPD_ROOT / "input"
WPD_RESULTS_DIR = WPD_ROOT / "results"
for p in [CALIBRATION_DIR, EDITOR_DIR, OCR_CACHE_DIR, WPD_ROOT, WPD_INPUT_DIR, WPD_RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)


In [ ]:
# Local OCR selection. No article/image is sent to an external service.
import shutil
import platform


def _package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def _install_missing_package(module, specification):
    if importlib.util.find_spec(module) is None:
        if not AUTO_INSTALL_MISSING:
            raise RuntimeError(f"Missing {specification}. Install it in this kernel's environment.")
        subprocess.check_call([sys.executable, "-m", "pip", "install", specification])
        importlib.invalidate_caches()
    from packaging.requirements import Requirement
    requirement = Requirement(specification)
    installed = _package_version(requirement.name)
    if installed is None or not requirement.specifier.contains(installed, prereleases=True):
        raise RuntimeError(f"Installed {requirement.name}={installed} does not match {requirement.specifier}. "
                           "Use a separate compatible environment; the notebook does not silently upgrade it.")


def select_local_ocr():
    requested = OCR_BACKEND.upper()
    if not ENABLE_LOCAL_OCR or requested == "OFF":
        return "OFF", "", {}
    if requested not in {"AUTO", "TESSERACT", "RAPIDOCR"}:
        raise ValueError("OCR_BACKEND must be AUTO, TESSERACT, RAPIDOCR or OFF")
    candidates = [TESSERACT_EXE, shutil.which("tesseract")]
    if os.name == "nt":
        for env_name in ["ProgramFiles", "ProgramFiles(x86)", "LOCALAPPDATA"]:
            base = os.environ.get(env_name)
            if base:
                candidates.extend([str(Path(base)/"Tesseract-OCR"/"tesseract.exe"),
                                   str(Path(base)/"Programs"/"Tesseract-OCR"/"tesseract.exe")])
    executable = next((str(p) for p in candidates if p and Path(p).is_file()), "")
    if requested == "TESSERACT" and not executable:
        raise RuntimeError("Tesseract executable not found. Set TESSERACT_EXE or use OCR_BACKEND='RAPIDOCR'.")
    if executable and requested in {"AUTO", "TESSERACT"}:
        _install_missing_package("pytesseract", "pytesseract>=0.3.13,<0.4")
        banner = subprocess.check_output([executable, "--version"], stderr=subprocess.STDOUT, timeout=15).decode(errors="replace").splitlines()[0]
        languages = subprocess.check_output([executable, "--list-langs"], stderr=subprocess.STDOUT, timeout=15).decode(errors="replace")
        missing_languages = [x for x in OCR_LANGUAGE.split("+") if x not in languages.splitlines()]
        if missing_languages:
            raise RuntimeError(f"Tesseract language data not installed: {missing_languages}")
        return "TESSERACT", executable, {"pytesseract": _package_version("pytesseract"), "tesseract": banner}
    _install_missing_package("rapidocr", "rapidocr>=3,<4")
    _install_missing_package("onnxruntime", "onnxruntime>=1.22,<2")
    return "RAPIDOCR", "", {"rapidocr": _package_version("rapidocr"), "onnxruntime": _package_version("onnxruntime")}


if EXECUTION_MODE == "AUDIT_EXISTING":
    # Audit never installs an OCR package or loads a model.
    spec_path = FIGURE_LOGS_DIR / "RUN_SPEC.json"
    if not spec_path.is_file():
        raise RuntimeError("No RUN_SPEC.json exists yet. Use BATCH for the first run.")
    _prior_spec = json.loads(spec_path.read_text(encoding="utf-8"))
    OCR_EFFECTIVE_BACKEND = _prior_spec["settings"]["OCR_EFFECTIVE_BACKEND"]
    TESSERACT_COMMAND = ""
    OCR_DEPENDENCY_VERSIONS = _prior_spec.get("ocr_versions", {})
else:
    OCR_EFFECTIVE_BACKEND, TESSERACT_COMMAND, OCR_DEPENDENCY_VERSIONS = select_local_ocr()

if EXECUTION_MODE != "AUDIT_EXISTING":
    _versions_after_ocr_setup, _incompatible_after_ocr_setup = check_dependency_versions(REQUIRED)
    if _incompatible_after_ocr_setup or _versions_after_ocr_setup != DEPENDENCY_VERSIONS:
        raise RuntimeError("OCR dependency installation changed the core environment. Restart the kernel "
                           "and rerun from the first cell before processing any figures.")

print("Local OCR backend:", OCR_EFFECTIVE_BACKEND)
print("OCR versions:", OCR_DEPENDENCY_VERSIONS)
print("First RapidOCR use may download public OCR model weights. Source figures stay local.")


In [ ]:
# ============================================================
# 4. Scientific / staging contract
# ============================================================

STAGE_ORDER = [
    "VALIDATE_ASSET",
    "PANEL_SEGMENTATION",
    "AXIS_TEXT_RECOVERY",
    "GEOMETRY_ANALYSIS",
    "DIGITIZATION",
    "RECONCILIATION",
    "SEMANTIC_STAGING",
    "QA_FINAL",
]

FINAL_REVIEW_STATUSES = {
    "VERIFIED",
    "ACCEPTED",
    "EXCLUDED_DUPLICATE",
    "NOT_QUANTITATIVE",
    "UNUSABLE_IMAGE",
}

CANDIDATE_COLUMNS = [
    "figure_item_id", "study_id", "source_file_id", "figure_index", "figure_label", "page_no",
    "panel_id", "candidate_index", "digitization_method", "figure_type_candidate",
    "series_candidate", "x_value_candidate", "x_position_pdf", "x_position_px",
    "value_candidate", "y_position_pdf", "y_position_px",
    "uncertainty_candidate", "uncertainty_type_candidate",
    "outcome_raw_candidate", "unit_raw_candidate", "target_entity_candidate",
    "confidence_geometry", "axis_calibration_r2_x", "axis_calibration_r2_y",
    "possible_duplicate", "duplicate_evidence", "verification_status",
    "eligible_for_database", "notes",
]

NUMERIC_TOKEN_RE = re.compile(r"^[+\-−]?\d+(?:[.,]\d+)?(?:[eE][+\-]?\d+)?$")
FIGURE_LABEL_RE = re.compile(r"\b(?:Fig(?:ure)?\.?)[\s\u00a0]*([A-Za-z0-9]+)", re.I)
PANEL_LABEL_RE = re.compile(r"\(([A-H])\)")
CONTRAST_HINT_RE = re.compile(r"\b(difference|contrast|fold\s*change|log2?\s*fc|logfc|ratio|relative\s+change|delta|change\s+vs)\b", re.I)
HEATMAP_HINT_RE = re.compile(r"\b(heatmap|heat\s+map|hierarchical\s+cluster|clustering)\b", re.I)
ORDINATION_HINT_RE = re.compile(r"\b(PCA|PCoA|PLS-DA|OPLS-DA|ordination|score\s+plot|loading\s+plot)\b", re.I)
BOX_HINT_RE = re.compile(r"\b(box\s*plot|boxplot|violin)\b", re.I)
BAR_HINT_RE = re.compile(r"\b(bar\s*(?:chart|plot|graph)|histogram)\b", re.I)
LINE_HINT_RE = re.compile(r"\b(time\s*course|time-series|time series|curve|kinetic|dynamics?)\b", re.I)
SCATTER_HINT_RE = re.compile(r"\b(scatter|correlation|regression)\b", re.I)
PHOTO_HINT_RE = re.compile(r"\b(photograph|representative\s+image|micrograph|microscopy|phenotype\s+image)\b", re.I)

SOURCE_PRIORITY = [
    "SUPPLEMENT_MACHINE_READABLE",
    "SUPPLEMENT_TABLE",
    "MAIN_TEXT_TABLE",
    "EXPLICIT_PROSE_VALUE",
    "FIGURE_DIGITIZED",
    "FIGURE_DIGITIZATION_WEBPLOTDIGITIZER",
]

print("Stages:", " -> ".join(STAGE_ORDER))
print("Master write enabled:", WRITE_BACK_TO_MASTER)

CANDIDATE_COLUMNS += [
    "candidate_id", "point_role", "value_axis", "axis_scale_x", "axis_scale_y",
    "calibration_source_x", "calibration_source_y", "working_image_sha256", "source_pdf_sha256",
    "coordinate_system", "error_lower_candidate", "error_upper_candidate",
    "error_low_endpoint_candidate", "error_high_endpoint_candidate", "digitization_precision_candidate",
    "evidence_path", "geometry_bbox_px", "category_raw_candidate", "external_file_sha256", "external_row",
    "external_tool", "external_tool_version", "external_project_sha256", "external_operator",
    "external_axis_type", "external_calibration_notes"
]


In [ ]:
# ============================================================
# 5. Atomic writes, local resume mirror, hashes, locks
# ============================================================

def clean_str(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    return str(value).strip()


def safe_name(value: str, max_len: int = 120) -> str:
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", clean_str(value)).strip("._") or "item"
    return value[:max_len]


def jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if pd.isna(obj) if not isinstance(obj, (str, bytes)) else False:
        return None
    return obj


def read_json_if_exists(path: Path) -> Optional[Dict[str, Any]]:
    if not path.is_file():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


_resume_project_key = hashlib.sha256(PROJECT_ROOT_DISPLAY.encode("utf-8")).hexdigest()[:16]
LOCAL_RESUME_DIR = (
    Path(os.environ.get("LOCALAPPDATA") or (Path.home() / ".cache"))
    / "SynerPlant" / "figure_resume" / _resume_project_key / FIGURE_PIPELINE_VERSION
)


def _resume_local_path(path: Path) -> Path:
    try:
        relative = Path(path).relative_to(PROJECT_ROOT).as_posix()
    except Exception:
        relative = str(path)
    key = hashlib.sha256(relative.encode("utf-8")).hexdigest()
    return LOCAL_RESUME_DIR / key[:2] / (key[2:] + Path(path).suffix)


def _atomic_write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", newline="", dir=path.parent,
            prefix=".figure_resume_", suffix=".tmp", delete=False,
        ) as f:
            temp_path = Path(f.name)
            f.write(text)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp_path, path)
    finally:
        if temp_path is not None:
            try:
                temp_path.unlink(missing_ok=True)
            except OSError:
                pass


def _write_resume_text(path: Path, text: str):
    local_path = _resume_local_path(path)
    local_error = None
    try:
        _atomic_write_text(local_path, text)
    except OSError as e:
        local_error = e
    try:
        _atomic_write_text(path, text)
    except OSError:
        if local_error is not None:
            raise
        print(f"RESUME WARNING: Drive copy unavailable; local state saved: {local_path}")
    else:
        if local_error is not None:
            print(f"RESUME WARNING: local mirror unavailable ({local_error!r}); Drive copy saved: {path}")


def write_json_atomic(path: Path, payload: Any, mirror: bool = False):
    text = json.dumps(jsonable(payload), ensure_ascii=False, indent=2)
    if mirror:
        _write_resume_text(path, text)
    else:
        _atomic_write_text(path, text)


def _resume_timestamp(record: Dict[str, Any]) -> float:
    value = record.get("_resume_saved_at_utc") or record.get("completed_at_utc") or record.get("updated_at")
    try:
        dt = datetime.fromisoformat(str(value))
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.timestamp()
    except Exception:
        return 0.0


def read_checkpoint(path: Path) -> Optional[Dict[str, Any]]:
    candidates = []
    for candidate in (_resume_local_path(path), path):
        cp = read_json_if_exists(candidate)
        if isinstance(cp, dict) and cp:
            candidates.append((candidate, cp))
    if not candidates:
        return None
    selected_path, cp = max(candidates, key=lambda x: _resume_timestamp(x[1]))
    if selected_path != _resume_local_path(path):
        try:
            _atomic_write_text(_resume_local_path(path), json.dumps(cp, ensure_ascii=False, indent=2))
        except OSError as e:
            print(f"RESUME WARNING: cannot mirror {path.name}: {e!r}")
    return cp


def write_checkpoint(path: Path, payload: Dict[str, Any]):
    body = {**payload, "_resume_saved_at_utc": datetime.now(timezone.utc).isoformat()}
    _write_resume_text(path, json.dumps(jsonable(body), ensure_ascii=False, indent=2))


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


_HASH_MEMORY = None

def load_hash_cache() -> pd.DataFrame:
    for path in (HASH_CACHE_PATH, _resume_local_path(HASH_CACHE_PATH)):
        try:
            df = pd.read_csv(path, dtype=str, keep_default_na=False)
            if {"path", "size_bytes", "modified_time_ns", "sha256"}.issubset(df.columns):
                return df
        except Exception:
            pass
    return pd.DataFrame(columns=["path", "size_bytes", "modified_time_ns", "sha256"])


def cached_sha256(path: Path) -> str:
    global _HASH_MEMORY
    if HASH_POLICY.upper() == "NONE" or not path.is_file():
        return ""
    if _HASH_MEMORY is None:
        _HASH_MEMORY = {
            (r["path"], r["size_bytes"], r["modified_time_ns"]): r["sha256"]
            for r in load_hash_cache().to_dict("records")
        }
    st = path.stat()
    key = (str(path), str(st.st_size), str(st.st_mtime_ns))
    if HASH_POLICY.upper() == "ONCE" and _HASH_MEMORY.get(key):
        return _HASH_MEMORY[key]
    digest = sha256_file(path)
    _HASH_MEMORY[key] = digest
    rows = [{"path": k[0], "size_bytes": k[1], "modified_time_ns": k[2], "sha256": v} for k, v in _HASH_MEMORY.items()]
    _write_resume_text(HASH_CACHE_PATH, pd.DataFrame(rows).to_csv(index=False))
    return digest


def file_signature(path: Optional[Path]) -> Dict[str, Any]:
    if path is None or not path.is_file():
        return {"exists": False}
    st = path.stat()
    out = {"exists": True, "name": path.name, "size_bytes": int(st.st_size), "modified_time_ns": int(st.st_mtime_ns)}
    if HASH_POLICY.upper() != "NONE":
        out["sha256"] = cached_sha256(path)
    return out


def pipeline_fingerprint() -> str:
    settings = {
        "revision": FIGURE_EXTRACTOR_REVISION,
        "source_pipeline": SOURCE_PIPELINE_VERSION,
        "panel_split": AUTO_PANEL_SPLIT,
        "max_panels": MAX_PANELS,
        "vector_bar": AUTO_VECTOR_BAR_DIGITIZATION,
        "vector_line": AUTO_VECTOR_LINE_DIGITIZATION,
        "raster_bar": AUTO_RASTER_BAR_DIGITIZATION,
        "min_r2": MIN_AXIS_CALIBRATION_R2,
        "duplicate_screen": ENABLE_DUPLICATE_SCREEN,
        "duplicate_rel_tol": DUPLICATE_REL_TOL,
        "duplicate_abs_tol": DUPLICATE_ABS_TOL,
    }
    return hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()


@contextmanager
def project_run_lock():
    lock_path = LOCAL_RESUME_DIR / "RUN.lock"
    lock_path.parent.mkdir(parents=True, exist_ok=True)
    handle = lock_path.open("a+b")
    if handle.tell() == 0:
        handle.write(b"0")
        handle.flush()
    handle.seek(0)
    locked = False
    try:
        try:
            if os.name == "nt":
                import msvcrt
                msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
            locked = True
        except OSError as e:
            raise RuntimeError("Another SynerPlant figure extractor is running for this project on this computer.") from e
        yield
    finally:
        if locked:
            handle.seek(0)
            if os.name == "nt":
                import msvcrt
                msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                import fcntl
                fcntl.flock(handle.fileno(), fcntl.LOCK_UN)
        handle.close()


print("Local resume mirror:", LOCAL_RESUME_DIR)


In [ ]:
# Reproducible effective settings: run selection is deliberately not a scientific setting.
_EXTRACT_SETTING_NAMES = [
    "SOURCE_PIPELINE_VERSION", "FIGURE_EXTRACTOR_REVISION", "ENGINE_SOURCE_SHA256",
    "DISCOVERY_POLICY", "USE_HUMAN_REVIEW_QUEUE", "ALLOW_LIKELY_QUANTITATIVE_FALLBACK",
    "INCLUDE_RECONCILE_FIRST", "MASTER_DB_REFERENCE_MODE", "MASTER_DB_SNAPSHOT_XLSX_SHA256",
    "MASTER_DB_SNAPSHOT_CSV_SHA256", "AUTO_PANEL_SPLIT", "MAX_PANELS", "RECOVER_FROM_SOURCE_PDF",
    "PDF_RENDER_DPI", "MAX_WORKING_PIXELS", "MAX_OCR_ATLAS_PIXELS", "ENABLE_LOCAL_OCR",
    "OCR_EFFECTIVE_BACKEND", "OCR_LANGUAGE", "OCR_TIMEOUT_SECONDS", "OCR_MIN_CONFIDENCE",
    "OCR_PREPROCESSING", "MIN_AUTO_TICKS", "DECIMAL_COMMA", "MIN_AXIS_CALIBRATION_R2",
    "AUTO_VECTOR_BAR_DIGITIZATION", "AUTO_VECTOR_LINE_DIGITIZATION", "AUTO_RASTER_BAR_DIGITIZATION",
    "AUTO_RASTER_LINE_DIGITIZATION", "AUTO_SCATTER_DIGITIZATION", "AUTO_ERRORBAR_ENDPOINTS",
    "MAX_CURVE_POINTS_PER_SERIES", "MIN_BAR_COUNT", "MAX_BAR_COUNT", "ENABLE_DUPLICATE_SCREEN",
    "MAX_DUPCHECK_CELLS_PER_FILE", "MAX_DUPCHECK_VALUES_PER_STUDY", "DUPLICATE_REL_TOL", "DUPLICATE_ABS_TOL",
    "REQUIRE_WPD_PROJECT_FILE"
]


def extraction_spec():
    return {"settings": {k: globals()[k] for k in _EXTRACT_SETTING_NAMES},
            "dependencies": DEPENDENCY_VERSIONS, "ocr_versions": OCR_DEPENDENCY_VERSIONS,
            "python": platform.python_version(), "platform": platform.system(),
            "contract": "QUEUE_ALIGNED_PROVENANCE_FIRST_CANDIDATES_ONLY_V1_2"}


def pipeline_fingerprint():
    return hashlib.sha256(json.dumps(extraction_spec(), sort_keys=True).encode()).hexdigest()


def freeze_run_spec():
    target = FIGURE_LOGS_DIR / "RUN_SPEC.json"
    current = extraction_spec()
    if target.is_file():
        old = json.loads(target.read_text(encoding="utf-8"))
        if old != current:
            raise RuntimeError("RUN_SPEC_MISMATCH: extraction settings, master-reference snapshot, package versions or platform changed. "
                               "Restore the saved environment, or choose a NEW FIGURE_PIPELINE_VERSION; "
                               "do not mix different methods in an existing run.")
    elif EXECUTION_MODE != "AUDIT_EXISTING":
        write_json_atomic(target, current)
    print("Frozen settings:", target)
    print("Effective fingerprint:", pipeline_fingerprint())


freeze_run_spec()


In [ ]:
# ============================================================
# 5b. Immutable master-database reference for gap / duplicate reconciliation
# ============================================================
# Export source: TFM_ML_DATABASE_V3.xlsx, SHA-256 ee04544c65c8405b5665cbbd0d64b157d0fdabaf67d3a8c6df921c40cce7c4e2
# Snapshot counts: measurements=231, provenance_qa=58, relevant extraction_queue rows=96.
# This snapshot is a READ-ONLY reference. It is persisted into the run log for audit.

_EMBEDDED_MASTER_B64 = {
    "measurements": (
    'eNrtXd1u4ziavd+nIApoTDdGpRb/9NOFWSBVSaYGWz9Bku4Z7I2g2HIswJY8kpyq7NVezvUs9h32ti/mBeoBZt6hn2RJSnISV2wqlCPTKmJ6UnZiSzqHR4cf'
    '+X2k5nFULPN4HqdlmIyt+PMizpPmXZTP+T/Zshxl8zjMo09hGs3j1S+KMkrHUT5++NtJNE9mt1aWX0epVbKDLbKEHY9925rH7FfF2CpiK7WWaVL9lr+w2LHK'
    '8CoqksJa5NlNnEbpKOZnv2EXNElGUZlkKT9juSysv0bNqzQr48JaLsZRGY/DqPw36vjhyV/OHBgenb93cJjGZbiYZmVW3KblNC6TETtnGYcOtFYftda+ZH2I'
    'S3DvS/yiNhzHWvsYzsegzJcxmMXRxEIETEE0KeMcHM1i9pHoFUjm83icsK/ObsEij19+iN7MLOjayLcsx0aeBR3ryz/m2Qy8+YjA/Le//R2Bgv2EG37LOQVf'
    'fgUXJwLG2fnHX8Lzk4uf311ehGfIuvj57Ozj+eXJsXX88efX707CN29P3vwHe/smWnBKwTieJGlcgElyzYQAbqLZkr2LCnbAVyD9A3TAYhalZfFjmcdRyaVh'
    'W8hB7ksneImcS+j8BNFPjvOfa8zDzcyjjczD52D+U8R/MqpXADY1ArUDyBsB99UGLz5koEiuUyHwtATjZDKJ85gp3wJn/+7YDrVftCEbrZPNCAnzeMZumps4'
    'HE2j9Jrxjjfyjh7hHVRfAzcFOI5ugWPJT7G9SfhRfOsl8Ri/Fmf4O/a/5gjN2SZ5Nq/P9yQmTz4vZskoKVl75sxwcuYGTNkj1uJFzGSclYC1UHRVZLNlGQNm'
    'bIC3md2O3DKP0mKR5JUHfcUr2cLr5f2vAq7br3jdenTrOirC+HP95lFCsW/TAyO0KLN5VEazcJSl4+WoFF7/Fa90C68X9RHAvSN8xWyb07QgmPgHo1hce+8k'
    'myVRzlxhlGf8RmW9+oh36veYvHOD+kvWqfgSGEfziOH6/jvQfP0Ha8PxLBY73CSsywirL1mfptksfik6DCC+wzkF49qJ3//yY2XGKydmZut4Vv3z7oT3Xz7m'
    'rad/+qO7mVkCknQcL2L2g13GXUhTvALzJE3myzmgX3VqII5G01fs09GIOWHGYIxBKpqBRxrxuBXxaDPxaCPxaC/EI98mlkVslwyBeLyZeLyReGwU35l4spl4'
    'spF4shfioS0ibBwcNu+kvcXftQDZq8UTxImH1ILoCdRfHjGqt3SgbGyCQJnxIKBmMeJHKwC63xKt2Gzl2+ts7se3XWQ7TMeujX1t6Wzlxut07suNqa+5OFtZ'
    '7Dqb+7FYhG2PjUWYRHWkk7Z3zruhB92nczJlastjK89c53E/nsk1iW2kLZWt/HKdSmwkuc5jK6dc53FPwSiyMbZYTOoFGtLptndK11r70r4GVX49qIJkt2yS'
    'HbHZyi/X2dyTXzpCnJ4daMtmK8tcZ3Oflqkpj60sc53H/VgmFvNV0NWQS6+9X3rW2pf25JdQTEJBqGHv47X3y3U29+SXxK5CzIBqS2crw1ynE5sJo8fZbGWb'
    '62zuxzYJzw46tq/TnX6MIAnfO8z8jkP2EqMx8TzPwSR2SZ3w2/SH+jb/JSmW7NSfklmZpNegGGU546Z6F1bvHmWumo21CDg+umTU8Hy/ZYnPA/jbf/8PffBa'
    'lKuIa21FyYcMLNNRnJdRkpa3IMtBepfaS1JwGV2xC0Cv7qX8RzzTWLIGK3gZQhHnN9VHi2zJmh+U/BsSDtG+OKQ1h/jgKcT7otAVFGLbPXwZkn1x6AkOke0f'
    'vg7pPu3wnOnQ8w+eQ1eRQ2y6lBWH3r44bLoUPjg5cA79fXHY9Cnk8O/lYF8ceoPhEDr79MNzdi+zyODQOVQdpsDd9Cn04AlE+yKQDoRAvC8C3YEQSPZFoDcQ'
    'Auk+PfB8AAS6yoO7F6fslFNwlWTzqCjA96evLXD9wwurmGZZGU74H8P6j9bDd+ITdS8CsVgm4FrX7H+PMYY3M3Z0E+d8xjKbALeuZwSLOAdxWua3vFgdXBz/'
    'eHGyYlFChbdXKs558h1pQoX6MGFnqgg0oSLYKxVMFVgPIpCjbLU704SnCRVwr1Sc83WZUBMq1Of5LzggMFnn44fWXMCKDOTyVVD+Ri7IZi6g/1UZvGDgfj/c'
    'kgf1yfoPmQ1OZ9kn3ov/Hpzmy6RkeMUvGAnsXTjKliziWEzjNJtl17cP4hDYDGhrFqrPVj+fyMaLR+kQWbjqeBupebGdG/VJ+O/AOWvpwsq5Ilg4NolG8SoH'
    'KX4Z5aNpUsajcskCtvyeLKi4RRgjfHXcE5ngJwX16QATXMz+G4OrWzBOrhOeW0xEpjRKo9ltkRSqmqHKXcvO7h1fh3tHfXJ8J/cO5UvOdL131Ce9O907ge73'
    'jq/cBe/s3gkalvZ79wTKTOzk7nHFoEXPuwerx6wd7h5P+7sHq1e4nN78eDq3JjfhZG6NprMszxbT29ksnMyWGbtXRmKXCLFMvYraHdvjkYk15pU3RZKlMwZp'
    '7d1jHNGtEyVViLphioRKwKuHrP/8v8xaFEmYtYRONUOuHqT+639PGPRpEsZtsWNfM/CkA/jjCvy4LXhENAOvXsNx9qfw6KqwFkkYsX/awQ80Q+/2ZHfn3O50'
    '073Xi91x6EQ3v/N78jsOHusGPujJ7zh4hPQCT5ze/O7cgraDNYMPlUeNCvGdZm5PkDL4p8Z3RDOvJ1gZ+uEHeIR0AP/kAA9qBl59LvHpAR60dfM7tye/O9dw'
    'PEu8XvxOBHi6Wb3fk9+d6+h3QU9+d67hgJY6vfkdhx9oJnyqnoFWCPA0G9lQ9brNJwd4mnV0VL3iUiHA02xQR0kH8E8O8HQTvXqlo8oMnqeb8N2eDE/HCI96'
    'vRieiPA0G9dQvyfDO9fR8IKeDE/HCM91ejM8Dt/Xxe8RcvgiMwaDv6rwrl49Ujr3KU6up2Xr4oU4HfPSv7vNECzM2t7yeRYfOqtSBrHvwrE48f2tw7ek8P98'
    '+aqpO6/AFoAf+cuv7NDg2hLPbNiUrm8wo42Y4W4xi22mRZ6+E+bz92++As0P/eVXdmwpaC8QoLHFX9Wgm1c1aFEO8C5Or8tpVXQwu/f66wIE7IBxdFtYgQ1r'
    'O8PWaM7/u4+PneNOwXAzvEqncMOCgzzmJQf31xvY23ESGc6qbf9ctW0ua9oHgJHNgkNHLOrDX7XnnvBSGd5xftsoWaBl71fo7r9+gJTdUljsKOdrBNXdArW6'
    'b2sNV3dsLeLrPPtUTsN5li+mVXlRdbuuGpXYvgVtvVTsSaE+kHFbi2owQ548Z/93NWpdXwqZK/kB4E1SfgiW4fTFXetphDbYiBap2jGEolH9QCcls45PArSD'
    'H/OlGqxhA33aFUIZXEU7JrxpeYGNPlDRFqjqdhzYRLugAmIp1E52jPisf/VQDG1al0ghK9sxEUGUqxPazVEUVrZjXpjp8IhRJyW7MqCdwmOhYp08ypPBVbRj'
    'DMWuyzpB9bdAVbdjT3SxfIs2jUQcSKF2i475NI1jQ30aFzlSxMpujKrIAmuEdnMQRZTdGIvJCk8rN0ZIBrRTcCy6WZ3aFcvgKroxFcMAqNG0DCJboKq7MUTC'
    'jqtxuzYqplKsHaNj4VBIo84WuVLIyn5MxVyFRvNuaHMQRZXtmNa17BrJ2JfB7GDGfOF89ZgObVo1kMFVNGNfDAKQRpMy2NkCtYMZ892fxd7iGqkYQynWTmaM'
    'bYrEnJtGUsZIilnZjT2Blmg0UY43R1Guqh1TMXLXa6oCExnODn4MWbhYVyNo065UhlfRkB0RUeg0uYjdLVA7GTKyeBJEJxV7Uqid/JjY1OXZS42CY+xLIatP'
    'VoistKPR1AzeHEd56oUV+uVAiCPD2WnmWLsECIEyvKqZPO2yHwRtgdqhsIK1KrMmgrWSMZZi7eTHPl/HwPxYo4CREClkRT+uJ8q1qqwgm8MoX9WPdUyCEFeG'
    's1N4rFsGhHgyuKp1btqlP4i/BWoHO+YFuI4daDXII4EUa9dCNyeoF5ro0r7UkWJWnzyuKt00unHp5jgqUJ491jENQpEMaKcAWbscCMUyvKrZPN0SIJRsQdql'
    '8JjPHuulYSpF2s2OMR8OQLGRjzat60oxK4fHgRjrUZ20vDGOgo6yHeuYCKG+DGin2gr9siA0kAFW9GP9UiCuswWquiFjJOaPHa0mLFwoxdrNkinfSxDyR8vr'
    '075IilnRkusSVajDfXvs0/AShmdvTz6EHy7CN5fn76zj0KdXvocDl8aTCfSunNVa1Mf/0KxFFXsnJ6PCEk+lDhf1+3CUpaV4knbMXo0jRsycXd8V+1sZF9Xa'
    '1GmU38RFWT9EG0EwBtPbcZ4tsjQZgdFyViY3UZlkabP9ILbm1+A6mrHjg2iUjEH81yV/gBE7D7j+7W9/hw/XTz7tw6w9msWOL1fLWVkD3d9+uLxdxCBlKog/'
    'L9hxk/Ixelo15Yu6Le9WznKM/HwO+JSU04dNPMrGMfjd69HvbHA5rR98BMZZXDy4mNktSMYMXjJhV8o+xg7GiJ2DqGAq4+t0L04shhAkJUjuP1FpGucxFw07'
    'FjtiHrHv5uwATJ9Mx+wy2EfKDBTjH4vYBukfMN9WnLkYu7YZv3x27oh/8Xty/3kbqz+8AmVSFMsYFNGc/Sq95tczjotRnlxVYn0fl9NsXPxwfyUpuXTcnyBm'
    'ev29Q9nPddEevVaULOpNsrzjGrpkibtdspFR7MpmFQWL+/NY+A14LJR47NgotlHssaJiSX8WS74BiyVCsSx23CjaI2OzK9G+VRQt7U+034DNEonNXhnBNoI9'
    'UxSs259gv4GxF6G1yyLjsjLRvu00Z+D1p1v0DegWbTdaI9o70SrPGfjGandvtWbOoIXNKgo26G/OwPsG5gw8iceaaHalWMU5A+j0p9hvYM4AE5NJaKtYxQkD'
    '2F/uS+zSPvSgAEvHXyYuWGlWcc4A9pf8wsE34LKBVLOvTWgQnr47+qVzncHpLLrJ0iwZN8KdNL/YrXJhtUu6UG6+LBlUqWZbfWzfauW4arW6JvnVQq0dCgz6'
    '0irma0gGqFWOq9IqdM3EVhtnVS4t6M9WPTpMW/XkiQMz2FopVb2koC+lItv1hqhUjkue4jLh6kqq6oUE/ZlqMFBTDRpTpWYKq4VU1UsIeuz/vYH2/42rYtP/'
    'S5TatW6gR18dqK3WUiVbAgAj1pVYO9QL9BitwoFGq1A+XWWmAO6MVblSoMcQwB9oCODLpwDMzOpKquolAv1J1XWHKVXXraUamCRAC6mq1wb0FwA4dJgBgEOl'
    'AyszXXUnVfWSABMA7CwA8EwAsK165fxj9/0Gcia+NLYW1b8rhWbFPJvdlrFCV1/555d/XD+mrg2/1qErr/0REt+MkOSq67JhwM41h93D1Bx25YNy0yevjE59'
    'xf+OFefYXnCIiuPXLe1aX5u+dSW5Dkv2dy65Oo9+cJJz5RkdM0ZeKa7DevudKy44UMUFteIgNOluueI6LJjfdSB3kFFcM2wwWpNorfM69x3LzSXiMbKHJzlx'
    '4fyEhE9Am9k8qey6rFTfsegCT+xhf3iiExdeiY6aTT3aeJ36YvMdaw55NvIPUXPiwvkJMV+Cu1FzZvTQaK7DcvEdaw77B9q5igvnJ0Q23JzVPTIxXaO5Dgu+'
    'd+1zq9WGh+ZzzXJCtK1E2/jcSnMdFmzvehBB7YNMPvDrrnpWakYQWwT35uiyc5KVHcNiVxbNoiIOoxHTTlLeWuzKk+xzMmb/hnH6X7fzp/exd10sEy94iz4i'
    'ME9SobZ5JTom8zJO0haf0GCZqldvzUqg2ehSqsgOCdjn0iOxkTccPXI08g0pzHh3ZZHK6dnn80ccDMkfmw1SIDblAnI9qudun1GPcFB6hGbDnvZ6VM/sPp8e'
    'KR6SHik2/thej+p532f0RzQof0Ty3LDRo9Bj19zw80mSDCqEJIG0NMbMMjaK7JA2fi490maH6EHokTZ7QZudydpZpHJK+fn80R/UkMZvyrm2lDiY+uiVINXz'
    'zc83BzmkMQ1ZjWmQZ9I0cj2q56KfT49OMCQ9OoF0TGM2vbkTpHqi+vkECeGQBAnrHhthI8htgjw6+0vnRDY7hhUVoyy/YpcXsosVUtytPhFEzTNFBqHQCg8/'
    'I6S2v2UmKDYarTXaIbXdh0Ix8QY1Cq/w8DNSs4ylnYkqp7p7cVAvsNGA+vgKjyiPFBNGZhWpVKDque9eBMra0RuSQAUe0cXzBy9t0qfp4Bt5qqfCe+nfA2j7'
    '7oD6d4GnKi8PAjONJNenemq8D30S6g5KnxUeoU+0bZ2XqbWsBNo1V95LF+9Dm9IBdfECj9CoY9NtaxFNL9+ItEP6vBcbDYjtD8hFORwxiGcCNbsCtDFR5Wx6'
    'L/KEpHkezTD0KfDwM7o29M3CWrk+1ZPr/czTe8MaxAs8YhDv2IiaUbxUn+rJ9n5m6endTiuDmKWn9QYsTKfITILK9amee+9Dn67DF84MR58VHuGf/rbnIZhy'
    'Thj+cQe5eHYM63oZJdEomz2TRCl6WP3enG67SmWf2n/VMarL4OG2ft6MkxqZdkjH9yBSH9qOOzyRClhVTh6ZDenbmKlyTr4HkRLPRmh4IhWwhJMi2/XNIje5'
    'StUT832o1LnbJ3BIKnXqXQSZpUIjUrlI1dPz/VipN0wr9eot4JDZTqGFSNVz9H2MnLANvQGOnDisatc4x3T3MpF2zdP3oFMXNw88HpROBSzR42MbEbPSXarT'
    'Dqn6HlSKnODusXIDkmmFqxrkB2ZP9TZ+qpyy70GmEAY2GqBMK1z1CqYte3Oa1P1Kp+qp+z50SqmNBjivX+GqxlDY1Dm3kKl6Br8Xma4tBB2MTO8tCCVGpnKZ'
    'qifye5BpEDzcAmcgKhWwKi/1sOnyt4j04uNx52Q+O4ZVLGuBxuE4KebLctd+atNqSmqZJowRdsp10W38gwaPSqbNXty+WaAsF2OHlH0fUkQ2RYcqRX7t0j3r'
    'TF7+zheV8/L9mKLnHq4pevXz47c8qOq12ftmJUX15Hs/puiQwzVFp352y5bnV5m80EqJ6hl20z237J5N3yyXoXoOvacBCzngAQuR7iRremYhxK558n4ssa6M'
    'O0hLrMrfHF68YWZypGLskAzvQ4rY9g5WivzapfM4Jt9954vK+e5+lOjAw1WiI396lcnBrJSontHuR4n1k/4OUonNc/3wltqKyEixkaJ61rqnwbN3wINn+SNP'
    'TaS4UqJ6Yrqf8bOLD3f87Mof5vct1+/+P4EYd8w='
),
    "provenance_qa": (
    'eNrdXMly28x23ucpulR169r1AxRmkGacKogAB4mTOMpahAWCTQISJmEgRa3u8q5zK5VXyNaLvICzT97hf5KcboCjZVu2af+/4wUFAg2wz3em75xuOIyCJfZN'
    '38ITZ8ZgP3GS9SRZh3hzDGfjII3g+txx8d5XOig/Ds3F9jgxpy7e3bNIo+232MY42XyxsOtOItPf3fmQBgn87mMSmVbiBP7Ew4kdzDZngoixAn/uzDDMllni'
    'yJk7lkkHxomZpHF+Ds8m0/Xu2EyYB3Piw6NjJg1nZkLP/ZPMFSfdXmc06Rn9YXPQn3QFxsNmDNP1QPTJIgrSkCGjjJsuxzN8oMdaY25KuBvitt4eT/r2/W0c'
    'tq1ehU8HyxFTHTabA+NmwAgM+dfGCQrtIAnitZ/YOHbiN0hzQSJsoreIVwpCEX14j7iCoKJCoYDqQoeclwslPjsvcuR8GSXYR6Fr+kmMVjjCKJ/lDIU4QkmE'
    'zYRMuMB09eqE/P6kqb0zeozG1Bu1OtMfdrud3sDQyZwETlBYrsQK3IDn3vDCG4673SEx0C6axudQkFhO/RogeI4ZEGtAFJGzauA6ZoRmpgfmgl79BfnYigIA'
    '5jWRf2m6KY6J6P2CAWL7iBfQK+E1s/sr7X8vnDEw4WFT603a2qAxMr5B4mqjpnxGXvErhWWq1NyRQqStBP5vrRESigWJA6mkgiKVkdYk5/hCkZwCFZfKRJ1+'
    'jOZR4KE5+AHCj6BXh8wmZpDn+I6XwhVniXMjON+p/Oz7dd4yBvWO3md2v7rD4Kuk51k+s/uNjZdKb3i0PF8S3QoSspELrhchnuOQ10Jts+KSKwszBoktm8QB'
    '8p3nNrZ+ausmQ3mIIF4Y+Ieq1poAgqEx/PLRtru1h8akczmcuU8PevP2Xihe6GpdaTSERZP+XNNoGe0Bk1s2Ty17IzN9egyuGcxRGCRmHDugvJljr2dRcGfG'
    'XuADCAyKgxm5MMX+U0BPmP4MNVkzWoC+fQzGnyaRA7OEJ/oxxOGUmAMy3ThArhNDCPsO3esCL31K+Yw+gauiMJNUVeVECSvSJv7NV0ZsO8q7edxzr/VLXanG'
    'D0v5Xrce0rZhqtd78Y+VqCWcSezISUykFGTURLb5+9//jWeQZUYgWoRkMITNWQhyqwCZYejmET1GoCQId/gezppRAogEyIEbF/QyeEqEVIgkawBlTswK4Ecx'
    'BrRm+485CUqfDIunwErZj5Bjx00cf4FiK4ggFJoJkn7/2z9UpGsDaiESHPUK3xz2jkQSf7BIIplIFQSxUWwHQYKmTuCBS1C5DmUqI2Xf7WE+0fpkcko/SE41'
    'l1PKQkBC8hPI8iaXtnrBoLkbQMqOf5tHqUPC+V9QBJfiMuKLz4e5s1MJLf9goeUsoy/Pqx6D/uc/Awb9778b9FMPqF67jYk2fU7XJ5ExC+ZA5swJWBSQSnpJ'
    '6/eNAZHn7qn0xI2Waak3DMKSINwU1aRphFHDfphcVKbmQSDfhPI+z2j8G60qqyJzJqsCZGbgssBZEszOQUEks0fBCliYk9gkpTVBJtecotj0QrjfCtzUg8j1'
    'iqIdQ2oQGImRyyh2HvMxkOtMyybUZTeBSUur1BttA4iophPVfR4PacApb3gR8PiNk+HzCBThE6AIDK8/Nbyq2IhKjV7vnuPZqa36Sf3q8e7W7tYfFpfPgyIQ'
    'UK6EYpE5E4rqHiZ7WLjBolqB/D7FboyGgwoL2S5d2AmDwiiYpRa4xuYUtYPNWWF7Fth6nGb3rqAuYFeEK9As9wKgtNZFozbsDPsw634K0Z8aPTIjbzOn7TTO'
    'tz8NlzEklQR5kC8gY6e+6U2BvgVp7K5REqA8eTk+jICPBKqQwufUICrKFyiVPoExM7VUKvFFHhBVgF9OR/duUXi8HY35p+sodNqj4WI0bhZHXac06XruzvNE'
    'VsmyqlYzEMdQIkXAFODvh/9aIK9JMinNjJBlFXoNmLJdRgscAY/MEif4I4cCkn03LKxMjmOMZ1ksmgaPTEZEI5zlURx/TxrdwgJcm2fyevBjJDaR6OV4qFu6'
    'TTjYWW1PSJDDAthJUXogfATCUFz2Tzr+DD+WEXcOgJ4fgQlQA9DnyjkBkoExO9gAE5Bo2DMmeqPWGDRuIZZ12mCb18NGDwDRmFZHN3rawGDahqH3wXrbQ60J'
    'A0YNY/wyvIRT4lXc4kW4xlkfNO4SsjGnGXqFHeqwJD8hF/sLcGuClL0OAytI1m5+soymZhRT58kKFzKob1CzO7Ia9IrPDIv8TGZc24uvfwZ80inh2yvvJFre'
    'aQMGdTs6g/rko6VrFIoYkgCJmxBnEgxhIwR0wRQhwEChkyH2G4oTGGpGM4SjKAA0UzDBCGXu/HMtTP4CRMJXQcRvIZKPXZKI5gVRaAeQLNY5aQ8hPIJtMFAh'
    'RRiCsum62A0sG3tgJC7ARaqkrDhaBlA+OgAsuGT5i6b3M6BTTgqdeNg8aGgag2qayKDbHpVTu9BIKqLUMpM8Z5I7KI6s6mdgoJ4WA5aXtjCoBIYeDoOIOM+W'
    'IbPLmIUqL4kCF6K4j1nIsuBiMbGxvINE7KF30alDFgQYBQY1tRsZPFTrXqnwp3sFp4BIiJn9dPRKhdSfdoQxqVDAPKnxvSD7HfAO2sJEqyCakaBqumDfPo2C'
    'U5xAHQvJBxMLXmK0N2Myg5kznwPX8S2cd4Fy8aDkxYvsjlwwz7zHtNDFjyZ4C4b74A8wssix0Ja9FL6kN5K0mXjLk56lJZOh3rO44vLmXng3Dxr1UktrOaNE'
    'd9zF45pb3B1xxS7hORHJKtvHmtEaxSCoOaNdXyoV8KiYOj34vuODnjwaHM5j/JBmALzChUVho6RcY8/oMtNapsbXDKVxQZpYgbftjtL+2Rfp49YD9olME2C3'
    '1uim2YdJxiFgDAZINROZK0S64KBTy0xjQvl9H5PONKWHaNe6RisI91MIftH6fGFGkBFmn9LLRCrOSyW1JJsyx3ElCX+RRR7fwPAl67HV6jicg0Ou9K6aWJ4R'
    'SxrHc3G8vni3vvzIyQi/r/GswdN80yS0kOf+tfj73/4Bf0qoUh2ee80y2tDzKegyCcKQmDal5nBQhjJ9FgYObUyR20mZ9+19On1SlKdFVSwpMhg2r065z/So'
    'Phqap/NGXTab94vGenHVqqt1PB3Kneg+WnSHfKd0r7J7OAgsTxsUDKXDvLDHh7UpAUNBZIUi3lBBSoxmQYzLqFuPkUhQ2zBu2tYmzSp3XYYrz0YR9N//AfVv'
    'VvR/F0qWJSgqVoqcopgmr34OpeOhOUqXtbbceUrkcOyP7vzrEd8S1agbFR8X4aQSKtf7NQet85kqXX5xQKC9HhwIBJLGieOlRKoyGuEFVIcuulA4qD/uF+e2'
    'SeBYmDG7afFmFGdTRQaQ5+N9jE7U8+VLez3f/YKYXNg2Cbybu3RsJ+rd5SCqeNrDaDi7ad084pidul69J37c7Y1Rn2f7YlaFtTv9Ruu8a/Qgb3ZGkK59CPRB'
    'dI9mOLYiJwTA4oy7QIVB5AMz6wyGuzxgToH3mVnch2t1gWNltIw38f87ii5B4A7Wtp7pBZEhuTlMBKX4NNAvhEQbD7quMmwZXf0pukgel9dX6rvezhzytm6f'
    '9rf26wYa7kSuIHx4XywIaJEVpsWC9OG9WijBdyLgOGsC9VoVKDMwDawAA3EZ/y3PfY+4ypcWM2AEwxetPjuPw6ued3W1lBulWnprT2tJcqdUWu1VID3TwB4E'
    'kKGCN+C4EBwqub/XNSqOTD0eVAwZ2UdZ6ZiHkE17upzx0q1Fk8Yf99EyTpl4ADYpzaEsBDzqu2puQS3tdQL557QPI3Lt925mbinpTEJ70pYHiVp8wE9h72mw'
    'rj9NHKOU7hXcbPFg0aMWBSuoEveyACX1Ipf15bOixidJ3s34qQmFIM2ZM4f2+MtbYgMc9sP7PpRR/lvxe/qDn4iOtFs4aRo1o60f9Mc+Hr+NDh22Na3anCDW'
    '5nGLa45X3SveNEfGDJKs0rO5IwYEpAGAMEmzrFJizsb9tzRPQtokXA+DKYzHb0lrC+WtrTK6GLzdj5+ZmZAL7Yu3fnAQWwuon/UY834WMJ0pwXIXjZmD8Shb'
    'EM/qhE14/fquo56VZSSqTYPgfkOH4OfKm6Yn6Z4RVuCQZTzyQ05C6hS6HLPlgp/tm31WZ0ApNFDID1KaDk8ljyRtX02UoH49g08uW8ZLaY21ookb2MFeUt/0'
    'cl/JNGHNt8nRjLzXpHnmkrb/jLCm7ZNMH0jqrhSmOEZ0YRe7s/i0qrGCiFwkVBsvHSgJ0KZ9bcFcEiRKkKCTABFZv101rX6r/4P0Qh8NSrmUwJWkImr1z1v9'
    '/dYz2QOSbLd/5F3oCCfkLCBMAiiDvPMnyL8z0u4lSomIa0AZ4VfsxhVeM9kq2542iGYTwmxPqo5vBViYjBrdlwEMVfY4CeR6lHDxjVJX9flw6bZqgqDKbZEb'
    '8UaFPQIYHh3TUMVLkBglFcEJNsZuVutk23HoKpyDc3BhwB5gZNtGzEL0dmaZTadAYyJS3yXrMjV606emDsyYkD/4OoMHd0lsqqwtlJWENpAGQSTLuIlJXa1w'
    '0gCVSQHPzc3eXEByzcUxo8SxIHzRsu3VX8GuUrJ+p24dNv7r62/VnDgZdZoVIIUvUh4U21cLfsp1i57q1YziFR61B0/GWGuIPJte2Wzn2DtGgWuZkCC6/8wV'
    'OL5crfwLXaOpCzJ4iyAXtx2FBGIS++yCTYYNawVEK5RvmjHEi8r5xTkkYfhOu3QMCs3EXkEyh4q71Wgafap86kEsuNCpo1Y2DTorFGMPbCUrD4hdZQ3UXKd5'
    'FoxgdnTp3yTdFbA9HIUkCMy+oLln68xPcqVTVJsHvCk/Jok5iEiUBv4ICrJisl5sLgM/cGYxXTojfIlBtNOsdW8YVOvebJpWlGKaQCd4gRCr5LgaI7kICFaL'
    '7qCLaYRz5uvPtbnQK0pW95alD7r1X+ZkZwZtS2UNSVClT8qg6aZ3ogcWXW7ws+KH7hJElf4oiyc2tu6JihekRZRQ+jBPXZel/knHAgXKlJ/dGUaUcn54v+GQ'
    '05SU53BAmkEbKuKuM1shD/RQXyflVd8g3T4o5PeCFglzZElja2tk1W9nCSjbPUgfDbhbdAAk0Hh2HhNa9Wlr2+4+ylPhRBseFKR0+9EmS3I3ctV8aA97o0f/'
    'QdBdBfi0sLK16qCuLe7A+Q/jQK+LOmkSgtzsdmMhPB1CQQ9qEVLr5nuFUBqyEV6kLq0xwK6mGIJ9ksWDfPvMKkBwP2HsJw7Eha9CR++M2yfEJzjGhz6fIMSX'
    'AKHiFqFZsPJ/EYz6P8aC+iSI5ObD8wrD89IGHBJf2DT8c6NyYss5xGVnNjx5WukAGWI7f2JshEmFfx4Y4I1Xd8uOet0YNe7Uu+6oGbX022s/XN8N+DmXNGvF'
    'I2AqbhrTTZwEjDFwR0lBVnYOiIY3xVFsO2EGQejSAnRT9pzXOpti54+CQTg1DJR3jUWFEaVfCAbx1DCIFAaeY4q/EArSqVGQMhREIPQi/wsBIZ8aCDnzCp4R'
    'Sr8QDMqpYVB+yeCgnhoGNXMLBRiF8isBUTw1EEUKhKIyivwLwVA6NQyljDqUGNJs+TVgECddbVB/HgiI9p0HzF1YNb1zd9nRfPNGmV6vn9JmS2+u5eXSax4B'
    'cYXXu74KYFGTJeZMFqHe9enETJedQvm5a9aE7N7GlnH/zXjMIK3Zf6M16cEYTmRbhODcuP/TepfSEQM/LEue2dawZeNjYVDSxUVxKo05v1t6Wt/drGrT2mRS'
    'TRt3xeDmCDDNdZFu1A6qWAJcVZB50vDiDutbMnQHXpNsTWY2GDKoqmf7qXbW9YcBdlyx/ADIDstaChpf5JgzXi0dl7y/CGzCpKeNX4QahKjWQ9Rcq8N652k4'
    'YpXI6s8Hj+9Sw7vupP1KlZ0cl33mivaiaWd8LHIMfVdt1z/NN/jTtzdoWwlgI1uCotQ/z5fD9sLXEkfmAp/PHJAyoru9XlwD7m8sG2/g2jWyyVuwrkMWeKiu'
    'tj0+UEzWrgrm9J2Do40V24XX8seLWfmTSYPL8ec4ir7QRP1YKYNG54epBSaNukZN5NhWYN1nZgxecXakn535kmG0Eclk9+XH88AlvUb6znFm4lk7NUnJehJK'
    'WIAq2Rj9TzPwonwYF7q9RsvoHS/9HPegt6GBrbTazWFVKTXj1izVLk3Ok+xQWAmjC8fhh9UjNA936O2/+VJV8n18eYt1u2NPpDstNxv3QtOJflYiPsJGmGRb'
    'bV6CDNjZ4tKTLhviKBng+8C6W3Zv23xJXHPsJW5fD/UXIUNL/HcCx9BfpnbExg8paRYfocTL2zez2dCMTDDMn9v3OQJLnDQNrfoirIDEGEueNW2XbzeD4p0h'
    '6pWhV+ECvX4f3L0zI1d7fuUcuRjiXLaUqPG8CubIK6Wj9XPiYqqwi0b5spMJTGed0Bi0eXMqDiG/kEUCN1id287CzpY26BItxLPdPpqvRnQTI8G/QWtObH9y'
    'm97BFrTy/iwhzKM4nc+dx+PVgt3KVJxEqUWW2r9KUcbNUP/BiuoFdFkkJf//AdXXxa0giwxQqP9P2iIvDdMFrOxVpmduQSl5PZsw6RwMIo771c4lTZqdGjCk'
    'lyhNAh7Ra93234HMrOUYDVu8mlUv/KZ6PbNMxXBr0bHSHLJ0Rt6doNk+X5EmWmvKAs/IwvEWFZr0dnvks9U+x8pfzPuptdMRUDIANX4RTDLD3zdW1rWo8Q9e'
    'nM4ue0H6UMfc8PZWsYu2syxeHBODfZuGfL6izXpaU/E8xH+eJ6bJUqvc7jHIsNrCZ7lkWpnZbwHMcPujAKPP/RmI1Tc+S7cP8OQ/fOBURDz5z47Z/wGpSwuO'
),
    "extraction_queue_relevant": (
    'eNrdXNeW2zi2fb9fUe9X9ABgRN+XUaCyqBxfuEASFEkxiUHp6y8oqeyim+MZ97Kremot2yqF7cLeAM7Z54CUFxm6a9XSLLeu9x+iPDGpbrs+/f5pSAJaSyKf'
    'vr5q+iRNX58kJDzUPPa/pRnJ8rQWRhlNa3lskYxaOsn+p3gPAABrIlBqEGzENjlqy9nqEh5Ry5fsjKKzU28vuvW9py1qgH1M1+fLyUTXIfJ4r+7ens8B+nLx'
    '00uteDJURyr7dP3NE31Ub3Z7mqrP1Hqr3hiqtdZYU2s1BJDEAcwhsIDgD4j+AGD3Oir0GNXAO43laW/V82Rvshomo9ZuGsZXbwFtkA07SnlUY62hDdD6dVT8'
    'Lx8V/xjV+EhBw+y0xl5/XA/JRjKm11s+HLWGV/F0CoblUW06t1HgoddRCb98VEKtpSMo8MgSZFkGvEAloQa9G76B1SnHs2UUY4Q2ipwN1TjpOUe90TRIBejr'
    'kC+Nkxk2D69Dhr98yGLlkFu3XtDmewnuzWYHADnDkcOsO7h4O2fSPe77PzHkX78iJfbbeUmyZIwxVCBSFImvQX3ZmplAOW0OaGtHvS4e1UfuKmu5/v5yBXuv'
    'AvR1yA1J627kQUnl/3zEs6Wm9bROTVPV1py9rS3rQ/buqqeu/3gxie8aSbHTX2x3nyf0xXL3bubeSOZG4UtMQ8sN9//38ogVL2YU2r5rZulLnNCUJidqffmh'
    'GDLjJSg2xjIWichewAKtwdExGV7lZXd8W644KTHn9uKyzdVgOs7nzTanV4C+iiHtJwFoXf7qkvuhGvuExA4Xhf715ZiTMHNZRHRP9IWpEEduyGgnNCBu+GJH'
    'yVvp3mrGtMqS3MyYltY/MnrJ/pERw2e6koy8nBM3y2j4Y82USs3WaIFb/F4xhDUIJ/h29TbnjtHR9Xbe85Ro8xOaoc+nGWb0FdFQZB5LIrVtKBugBrnmSBsu'
    '2xIepiMrr/cJCAQnRme0arguXLYrQF81I1oyWQfp511nEFRqtu8HQr/Hr7IFPUSmd5rsNIj5K+D6VJsuWz+j2edbZxBWaqaeIEccH2rDSPFUvtVcBk0QtbqH'
    'yNuSxK//hGb859MMVWoGZqPdfMuGypmu2nP4gdVuhEN5aplEUv1O8hOaCZ9PM75Ss0PvbE75OjwGaW71Z1F+7FKw3O0kR3Hck9L4Cc3Ez6dZYW8V00SSTCUF'
    'SBIhUGZ5M4vEbpKBdCN15Za9PPmjDkKyqPFgBdUmVwH6qpngxirGG11f3IeCvmyG882n0kys1GzMjYy2AxDfsdMRGK7PkwEkZKVawKXSzAE/0gy3GyN/dH7V'
    'DH4+zaRKzQZ7aICJEshBR1UGdKUtbuq63uMhlw8cbvxDzUYGWdZt81Uz/vNpxgTCrCAONl6+djLZ6y+SZlA/rpbWZrS50JQz/KA742uAfeyrLMmgJx25+jcL'
    'Flv2v1SlVV/U9YZanxVK/BcootQQYiH90u5G3da0ZURKvJnli3AQqCN9ONiA5v7Ui1c1wD72VRGw4SeHDnveKn7P3KE0YzvMiszLp9EF15DEVkrrEOR4AaEz'
    'Ryt+pa4ycdfHGb9qOIFx1cQW00X6tlKaA9DJW41vK+UzScKWCZJxDZLlMRu1zYXmhMKYLoCp1tVx7mnuWOGCHt0ySWT8VZJ6w5S2pvhJJZEfDbbTxXEmnWNP'
    'H/eXln87toa7A1IaLbkr9Xpo/12DbZYTzYn1vxBP/m3XR8SPAGfbvb6QpB5/vlncwIwP5jCOnLpgDkVPvkXhM8C1l8PhQt0sil6kF1334fLba/dRvT5hY/r6'
    '499/WiTxEdR0tjdvi1YDZfX1YuJLy5E6ad2SRnY5TQfydvYMam9UEMUASZnzKVSQxUcIU8w5Z6fxYBYMBiexhzv5zjE6WeZJzZF2joRnCHujwlzSj0F2+Rwq'
    'yI+oNdtYPs7GeuzomrjIZOVIb/Hstrh2b7qr4vwZtd6osFfn9k7afwYVIHwGqqiV1ns2EegkplpLW+tz57BLY82cNWG+OK2egeqNCirspqR5/A9U+HfBCSqw'
    'sotun9XUcaWtnc78aavfktrp8SQeWuYx11QiTyu76G9G6MTaeuLOdZ3skyiMgisHEccmXhK4E/+XR8veqWygG6uDr6DLbrWGt2kSu9pquV+th8pq4mJ9EviV'
    'DfS3q2os2PuArbRiKUQX12KPaTEElk14kTuhv7bKfm8DHcHqBjo2L6PR2GXlWAzwtp2ZgZoKdQBBml4b22u/shn8RowpdCYjdafrAWULOvLdjFlwCJkWvIT/'
    'shZ/hx2HJLGyddLrimR42Peu+8GoK3epsRTHySHZT5ZwjA8yV9k6eaPYaBC3pnJb110vSDnEc1BAUPov1kmuLv37HU0c3zIxXocrL5yu4IiXk0miXPax3oyl'
    'aWUZ+0an1koWe3WmnR37aREQJFnBSPzvFUqSitIdGyZvm7aIKMK2UcSj6II6K+J2ttNDN5+FU3vL+4zu1tmZK8sSogpQSagnXtfTiLBQwAbwr2PmRNVahUh1'
    'y2Lszm7mvAQkzInP4kkS5+mLG8RRkr28soD4GyOIF0BmdNif/wUi+/eV1j0pGIYoIEUyFYIxC/U1OOl327dx14Ojzm2l9s8r1xVEqofWIDQv1nJVAfrqr5/Y'
    'R3Wq36tT/Wfqjt9CE1XSFPuntqHbbWWlKVvNbhzOw2AySFyJTo1+GJNKmm/96gP/ZpmLAIvi+85g0SSGikhFCVFABSgaItvBQfvaStx1kxqCUJ+HfVPjY7fR'
    '1tcQtvXh+FYBKlF74nXdYQNyzdxnW4dQDihccf6hlCk2fjPFoqcLKC8zLpZBMTAwYOkvwSvoDQKe21tSaOTAO0fWWrL8eQRjfnzhKkAlik+8rtMkRrzwvpSK'
    'uEuIIRKiABYWmKRAYuEk6o7lxmmws7ZCf5KrgS87G/cabDd2cwzQtgJUDicPvK7v84gNAb3vSixaogQhiAgSBahgwHZODQrnfmbYAI/MtiujRtAbLIZzyTGS'
    '3VXtz2lQASpxeuL/tBJlrrjwB/7Ip/wWloUXM2xbkC3M7KVBZbaFarA7I6PdaGqD64lZMTdSQ+i5WyhPt+50me26FaASyyde15sOicmepTM2EPi+a7K45MC0'
    'JdOCCLLAgCxTtmtwS7Nb7p3htpcJ8SAyB9AJJ+CY97YDcd3fyRWgErMnnqW4nCVmNkqXMG7wnaNkcWkANg2FKCbAkFi8YbK16amJHXkRdk8XoxEtpdnRGy2x'
    'vwx6ikf5YFEBKnF74ov95r/7fpMLS0IEwrKbhKlgEIliVtAhLujFFtkcpwsE/OUEQcCTIB9MB05DidxJBajE6YnX9SxxY5I57p8X4u8mBmtL3RIoNUUZmoZt'
    'GrxtsC2G3cxUXKeVdry+qM2O63x3yDu77WR06lz9RQWovMUeeF2/ES8KzWK+9MyhqZuW2bV+M7vCixhQMQlBsmVALCvQZMWcuNru1XPSFgXcHS2QNDl6A0FX'
    'udBI8lSGFaBvp20P7PPkSP/Zk/DfwpKvZHlaQjMMZf1WF3e7Ud/KfOOsGtgyuc7xvBr8kOUr9smS/zuwFCpZpmbCjU6Leawduvu1H99a/snqneSr3c+Cfur8'
    'iOUT+8oS/R1YipUsvX48jOdxbniLST9O2i2H6+WNcRth6TDT+uRHLJ/YV5bw78BSqmQpzgc0XS7ccZStlVRKT/poiDkdtoPhpgfETSXLtzXCA/9aIwisTOCR'
    '8N4R9d5AwgrlTQowkQUKoVGcJaFmV+7ejjEczTN0m4Orst1J09l0hwNg0QpQidsTr+sxpYlXWDJZFuD7RtPCtAAiA96QDB5QC1Ik1uBSEm694LRYbcdqdu3I'
    'gwkJ8eYWzmbaYK4ruwpQidkT/2bWIOaZw31fboVpIZbI4iNh/wDAqhjGLa2rGxYuLa0zsvb00OXD+sEewmg6iMNA2oEKUInbE1+0/TySfLn7Fgi/CF9k5V0X'
    'pVKMFBiGIgJJRETigUkYvWRmTg6zg8R1/Y117u9Vfcvvabift8IpIW2uAlQu6x74V0/GF91N5+pzHkmDKCQZfV+SRUrDQDCpyLNlJvAWKRqR54MlYecqpZl0'
    '2Wvywhh4E3uchdDHI5zU/QpQieQTz0w1RIoMOAAVFlhEgQPvy67wMpYpUUFUsEKgzbaUxXZfD5wnB28UIvnSHe0MIuLNqRl1VTpaRELUqwCVd98Dr+tF15zV'
    'CvL7kiqsi8kD3gYyM/3MZxMMWW2+HGKVitfp+HY9p6o3PNM+UqdzEavdZJdnFaBybf7A67rv5oyTpJPA5kzH9X1G8n0J3m8BYWmdWJZoEsmSaNFP2WUHa5gN'
    '7CUwB+7qJM/6zRlqoxnwbUfanKcVoBLBJ57FFfLutZBSWBRoilSClshTaAGxqMqPm97JtK/pGeAI7LmTbKviKadN/sa3uO4qrQCVKD3x39eu0vtyK5K3YhtU'
    'FkUFKZZtKpRtsnRA83yyrwP30mu3s8Z5vmk2B86An9d5I+TSClA5DTzwrymO5xQe8vB922DKPXsXQ5SISXlmpESDea75aB6MDw4fTNcDZTTLhF6v1Sb7VeJS'
    'laz8oAJUPkF/4B8rEWLdzM08MGjyrtlbwTVewDXYviQzXb+4/V5+Hem3fiNxtv3J4HIabdaXxSyrAfaxr8b48envWl1i0eqSMZeCn7lc5XeQwkXOhtS0AVJM'
    'KGLbwAbL2Ty8NjutXmPeW/huqwsGQagok0wXG0sPioNNBag0YU88M5L+/UD2foQMJPDezT1cZGsLmaaIAREEhCFvKTWYzftRdNKa1IyX+Syw14fcyLJNmkNu'
    '7Ax76wpQid4T/40e5JgUCL07vSJdixSItghlZilEkRqI0fPbwm5Mx61gb6drxWvc0HR/sZrmbJGPY9epAJXpPfC6PqI+yaLQDXXi+/TkMrOV6g4lGeeGVm6y'
    'AGSxILqnemTrWRSwz/5C+i8GyUznBZVkgOgPhP4kQ5HgRSTLPMCCRQRgSTxhruU82+HFDaDTRbz18yPRdC12zzNJ0Pez4s7XP4PKruWB1/V+lCch8SN74oaU'
    '+DOaUpKYDle4Ua5LUhJyX5UKqHWXiTK9ilNLkyRGFFr3axPoU7XYibIovYaPZhUJLfZDErDXfJqQ0GQf+0gxCzPBIwpMgzdEEVBiiyzzcmuRM8Zac9/yegFc'
    'Dvs64gOpcZJWmebMT3wFqCTmE1+KCBCK0sdwLNyFjTDhARJEGVuKybOot6nHGz/jORHmSjY9ZaujCLE/XTtavA2nR1ABKnF84v8U7ourUSSZ/xiqRReEGqJt'
    'sSLLoJhKMjLY3thd14cJj5pmxosK1EjreB23b3R3mdyIculXgMp744EvRUAIeOVjOBaGCmJiWgpB1JAkmRLG8ToUW7Yh40vCydw0y9JVW5GwpxhjQ9kMusMK'
    'UInjE8/MIvnlVctPkLvfOIqAAGXKQjWbE0VmKWyzpcJNwXVy6e5NZ7uVA4SUfBPOvMtxRfpZBai8Vh945lNiFntcVphR7hGrOCtiE0pDpwhF3HehiTOuXBBZ'
    'OYt2rMrhypGMY6GMM6nvs7eTjxHr3mFRbAHYNrAtqgBe5mvQPV/inmQLjXHdDZfdWX20uXGnfBfVV0r9cEQVoJJYTzzzn3lyiM7pwUWvzu2dGcqgiEIIA9NE'
    'AuWJxTZv0flDw7Ehk3bWU4/75riVpYJn9IfCun3mneE0whWg8iHRA8/WukMcjzj3sgh/DMMiMRPDligSTZnwAoUiK9cVSuJJPRPZwr0pwu1AUKT4g6Pjz7Oj'
    'Fk42FaASwyeeMQyvQUEPKB9Dr1htWJJZdYYVC1ICZLPYz6rSEHJOlfor+7TYeHtxXtd6qprsGvwpcUkFqLyfH3hdtxySGw6xqA8x/qAZLNwApqz8VngABbar'
    'AGGFk3ZdkqbVGfS7u+mm3mlEFEwnnmX2Igf7ztytAJUoPvFsBlnocdgEih/DrjBIzODYFgKGyeKAKUJUtCaW7fggrxumdTulWw1MbG+khfSwA3q/1aQVoO9a'
    'E3e8ri/u3o9rOi4LrSSc5Ibvpg61Jv367GMIF25JMpgJMG1DIJjH97b1ejEWcTYa9U6yewgFdT/jbdqXxhtWLoi2cq0AlQg/8Ww6BYQk5pEQYD5J4EVO+Bia'
    '96/4kGWCTYx4RbKoKbF5VQYgieqtU3ppaB1l2eh14sSv8+Ryyhuy5w4rQOW488Drukmij4up96s6ELJkGwkKIrzIS2wKtb7eijVWqKxbTeyrZn03X52RKV1G'
    '2qmVbsQKUHlHPvBvrxUvWtcy/KCdWVglBQnIBiIkgk1lLMs1aGiIJ026Dc/nXIuupA5X3t7H6uq4XHbTYVYBKrF84otGr00fhQv+pe3Qn2BYJHLeQpIlYIhN'
    'g0hIZMkjjbMzy4FSvUcvxDfWHTbc8851Xb4hRb3etAJUbh0+8LruPSrdL3EU0i+g6A7jj9mM96+x4CUgMPuKIeRNntWWNUjUwByGE7W7vmgJ2ndPErwm59t6'
    'EGiT6+q2rACVDzgf+O9uF3hcIg/wr23f/ATZwvFIMpRFKAOjiIEmKa5mZWMcSNnmoA3UeTQ5nRHfOlvKznHEobFdOxWg8kn1A//2zJNn88l/DMe77WEWhkLM'
    'ChNKiVA0U5f17lweEzlvQnMebhw6GU3OCDgTgixNs+sVoHId+sCX2gqIFz8m/ty/34EKhgyJzXKaIigGZdVHzC2H9f04GvBNt9n1B1l+lr0YZOEqyFfLRbcC'
    'VOL4xBc3Cn9hNbbwT+8LZOmU44EofLmbPMhCF7Ozl49hff+GBguzHG/xiiKzAIOKLyBzW4qSTDamEyjXCT+Mz0c5UPyhsaSbW+wuKkDlS/EeeJY3HVq49Y9J'
    'nPevUuABS+y2De2iac8LLKW0wDUQFt15cl23UH8zap28uiLN/Hye8GZwlCtA5fspHvjKa2ARlj+G6v1yX5MiImGkWAaxjOIet/nEHg8cdTHcGn64Cle2om6d'
    'Y9Qbzb2ztDhqFaDy2c0Df1+9PK/gfxbRqLhIAXxhk/p9hdJ8J673IzgoQdFGpsxTCdss48CrMwBSa6yMW645nlyvcBpHUdRMxjG65rdQrACVO0YPvK7viU8C'
    'I0rvND+EYGGFiMxbIpIEyNwpEYtznWtTDpI2bmTjuud22lxvNFvOCTAH07x7RsNbBahM8IFnRYpDXyZFyH15NsdfuGKcPHvYOSTcs8c58V3zyv6+ENO1XhK6'
    'L7pGNH3JztHLs3GUZjR46fVe4iTKqPm4t45kzplcGfvw5dEGf8lDiyYvv+Fo/Sf0LIyXLYrIkBTBxEAg9wsj6tGpRxbrmOHqSV8LyLQz3ozbU3VlxE1pFFSA'
    'Sno+8YUf+cW3Cv3n1FBhtUTTYh7fMAjBgoUhrcFNezux0Xo6DWLptphv6yZjpO29pH/Z8MfDtAJUbkg88K/uA3IiZp7sY5pm6H7WY8i8LUFAmFtiMZZRPAnH'
    'o3KrUzu1M7pKWzEmqnpAoZmsxrP23bF8DypRfOLflj98URyAj7GR6P6VXKZCAbZlxPNsYiCpQbz1s4F43jGTYGxX+ZwLZet6HIaLMz1YkXmoAJVYPvG6jgSO'
    'xsXNeGnqhpHvWlRno02iE7X0++Zkj48dqz+398fIULgwlotMw6TEgmzRIZuVDubcXouOp5+U61xeLfzpMhGRfDGWq0BfL0O3AlSS4RWvm7n7i293+Aluwj3d'
    'GpJNBMnGwMaiadfgjNdP0hbM3W68vHaCWbaVta4xhXy32YnWCagAlbg98UUfGGFFZ/R4vV4Yk+KCZfBBYanItqyOsyHzuQQVt6VSVhQF26267yrJMomXIzGg'
    '64E75m1hgBr9HZgPcAWofB/cA19kMBJElhu63NcOG3lQfme6/w8+iUx1'
),
}


def _snapshot_csv(name):
    raw = zlib.decompress(base64.b64decode(_EMBEDDED_MASTER_B64[name])).decode("utf-8")
    digest = hashlib.sha256(raw.encode("utf-8")).hexdigest()
    expected = MASTER_DB_SNAPSHOT_CSV_SHA256[name]
    if digest != expected:
        raise RuntimeError(f"Embedded master snapshot hash mismatch for {name}")
    return pd.read_csv(io.StringIO(raw), dtype=str, keep_default_na=False), raw


def load_master_reference():
    mode = MASTER_DB_REFERENCE_MODE.upper()
    if mode == "EMBEDDED_SNAPSHOT":
        out = {}
        raw_text = {}
        for name in _EMBEDDED_MASTER_B64:
            out[name], raw_text[name] = _snapshot_csv(name)
    elif mode == "EXTERNAL_XLSX":
        path = Path(MASTER_DB_REFERENCE_XLSX)
        if not path.is_file():
            raise RuntimeError("MASTER_DB_REFERENCE_XLSX does not exist")
        workbook_digest = sha256_file(path)
        if workbook_digest == MASTER_DB_SNAPSHOT_XLSX_SHA256:
            print("External workbook matches the embedded reference snapshot.")
        else:
            raise RuntimeError("The external master workbook differs from this frozen figure run. Export it, choose a NEW "
                               "FIGURE_PIPELINE_VERSION and rebuild the notebook snapshot before processing.")
        out = {
            "measurements": pd.read_excel(path, sheet_name="measurements", dtype=str).fillna(""),
            "provenance_qa": pd.read_excel(path, sheet_name="provenance_qa", dtype=str).fillna(""),
            "extraction_queue_relevant": pd.read_excel(path, sheet_name="extraction_queue", dtype=str).fillna(""),
        }
        raw_text = {k:v.to_csv(index=False, lineterminator="\n") for k,v in out.items()}
    else:
        raise ValueError("MASTER_DB_REFERENCE_MODE must be EMBEDDED_SNAPSHOT or EXTERNAL_XLSX")

    audit_dir = FIGURE_LOGS_DIR / "master_db_reference"
    if EXECUTION_MODE != "AUDIT_EXISTING":
        audit_dir.mkdir(parents=True, exist_ok=True)
        for name, text_value in raw_text.items():
            _write_resume_text(audit_dir / f"{name}.csv", text_value)
        write_json_atomic(audit_dir / "SNAPSHOT_META.json", {
            "database_spec": DATABASE_SPEC,
            "exported_utc": MASTER_DB_SNAPSHOT_EXPORTED_UTC,
            "source_workbook_sha256": MASTER_DB_SNAPSHOT_XLSX_SHA256,
            "csv_sha256": MASTER_DB_SNAPSHOT_CSV_SHA256,
            "counts": {k:int(len(v)) for k,v in out.items()},
        })
    return out


MASTER_REFERENCE = load_master_reference()
MASTER_MEASUREMENTS = MASTER_REFERENCE["measurements"].copy()
MASTER_PROVENANCE = MASTER_REFERENCE["provenance_qa"].copy()
MASTER_EXTRACTION_QUEUE = MASTER_REFERENCE["extraction_queue_relevant"].copy()

for frame in [MASTER_MEASUREMENTS, MASTER_PROVENANCE, MASTER_EXTRACTION_QUEUE]:
    frame.columns = [clean_str(c) for c in frame.columns]


def canonical_figure_label(value):
    m = FIGURE_LABEL_RE.search(clean_str(value))
    return ("FIGURE " + m.group(1).upper()) if m else ""


def figure_label_from_caption(caption):
    return canonical_figure_label(caption)


MASTER_PROVENANCE["_figure_norm"] = MASTER_PROVENANCE.get("source_figure", pd.Series(dtype=str)).map(canonical_figure_label)
_MASTER_MEAS_BY_PROV = {}
if "provenance_id" in MASTER_MEASUREMENTS:
    for pid, group in MASTER_MEASUREMENTS.groupby("provenance_id", dropna=False):
        _MASTER_MEAS_BY_PROV[clean_str(pid)] = group.copy()


def master_alignment_for_item(item):
    source_id = clean_str(item.get("source_file_id"))
    study_id = clean_str(item.get("study_id"))
    fig_label = figure_label_from_caption(item.get("caption", ""))
    prov = MASTER_PROVENANCE
    if source_id and "source_file_id" in prov:
        source_prov = prov[prov["source_file_id"].astype(str).eq(source_id)].copy()
    else:
        source_prov = prov.iloc[0:0].copy()
    exact = source_prov[source_prov["_figure_norm"].eq(fig_label)].copy() if fig_label else source_prov.iloc[0:0].copy()
    pending = exact[(exact.get("extraction_method", "").astype(str).str.upper().eq("FIGURE_DIGITIZATION_REQUIRED")) |
                    (exact.get("verification_status", "").astype(str).str.upper().str.contains("NEEDS_MANUAL_REVIEW", na=False))]
    supported = exact[exact.get("verification_status", "").astype(str).str.upper().isin({"SUPPORTED", "VERIFIED", "ACCEPTED"})]
    measurement_ids=[]
    for pid in supported.get("provenance_id", pd.Series(dtype=str)).astype(str):
        g=_MASTER_MEAS_BY_PROV.get(pid)
        if g is not None:
            measurement_ids.extend(g.get("measurement_id", pd.Series(dtype=str)).astype(str).tolist())
    measurement_ids=[x for x in measurement_ids if clean_str(x)]

    q = MASTER_EXTRACTION_QUEUE
    source_q = q[q.get("source_file_id", pd.Series(dtype=str)).astype(str).eq(source_id)].copy() if source_id and not q.empty else q.iloc[0:0].copy()
    queue_notes = " | ".join(sorted(set(clean_str(x) for x in source_q.get("notes", pd.Series(dtype=str)).tolist() if clean_str(x))))
    source_note_pending = bool(re.search(r"digitiz|graph-only|manual[_ ]review", queue_notes, re.I))

    if len(pending):
        gap_status="MASTER_EXPLICIT_FIGURE_DIGITIZATION_GAP"; decision="DIGITIZE_REQUIRED"
    elif measurement_ids:
        gap_status="MASTER_SUPPORTED_VALUES_FROM_SAME_FIGURE"; decision="RECONCILE_FIRST"
    elif source_note_pending:
        gap_status="MASTER_SOURCE_NOTE_DIGITIZATION_PENDING"; decision="DIGITIZE_REQUIRED"
    else:
        gap_status="UPSTREAM_QUEUE_UNRESOLVED"; decision="DIGITIZE_OR_RECONCILE"

    return {
        "source_figure_label": fig_label,
        "master_gap_status": gap_status,
        "digitization_decision": decision,
        "existing_figure_provenance_count": int(len(exact)),
        "existing_supported_measurement_count": int(len(measurement_ids)),
        "existing_supported_measurement_ids": ";".join(measurement_ids[:50]),
        "master_pending_figure_provenance_count": int(len(pending)),
        "master_source_digitization_note": queue_notes,
    }


def _master_reference_self_test():
    assert len(MASTER_MEASUREMENTS) == 231, len(MASTER_MEASUREMENTS)
    assert len(MASTER_PROVENANCE) == 58, len(MASTER_PROVENANCE)
    assert canonical_figure_label("Fig. 6") == "FIGURE 6"
    assert canonical_figure_label("Figure 1. Example") == "FIGURE 1"
    test={"source_file_id":"1bVkl82xZVW1zQrpiNVUgVWL8VPi9_Pml","study_id":"D_366d799918128863",
          "caption":"Figure 2. Seedling fresh weight"}
    a=master_alignment_for_item(test)
    assert a["digitization_decision"] == "DIGITIZE_REQUIRED", a
    assert a["master_pending_figure_provenance_count"] >= 1, a
    print("Master reference self-test: OK | measurements",len(MASTER_MEASUREMENTS),"| provenance",len(MASTER_PROVENANCE))


_master_reference_self_test()


In [ ]:
# ============================================================
# 6. Discover queue-aligned figure work
# ============================================================

def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.is_file():
        return pd.DataFrame()
    try:
        return pd.read_csv(path, keep_default_na=False)
    except Exception:
        return pd.DataFrame()


def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    return clean_str(value).lower() in {"1", "true", "yes", "y"}


def int_or_none(value):
    try:
        if clean_str(value) == "":
            return None
        return int(float(value))
    except Exception:
        return None


def resolve_source_path(source_meta):
    raw=clean_str(source_meta.get("source_path"))
    paths=[]
    if raw:paths.extend([Path(raw),PROJECT_ROOT/Path(raw),PROJECT_ROOT/Path(raw.replace(chr(92),"/"))])
    for p in paths:
        if p.is_file():return p
    name=clean_str(source_meta.get("source_file_name")) or raw.replace(chr(92),"/").split("/")[-1]
    if not name:return None
    hits=[]
    for folder in [PROJECT_ROOT/"01_fulltext",PROJECT_ROOT/"02_supplements"]:
        if folder.is_dir():hits.extend(p for p in folder.rglob(name) if p.is_file() and "_history" not in p.parts)
    unique=list(dict.fromkeys(hits))
    if len(unique)>1:raise ValueError(f"Ambiguous source basename: {name}; correct source_path in the input metadata")
    return unique[0] if unique else None


def locate_figure_asset(source_dir: Path, figure_index: int, image_file: str = "") -> Dict[str, Optional[Path]]:
    stem = f"figure_{figure_index:03d}"
    image_name = clean_str(image_file) or stem + ".png"
    preferred_dirs = [source_dir/"quality_v4_3"/"figures", source_dir/"parsed_v4_3"/"figures", source_dir/"parsed"/"figures"]
    def locate(name: str) -> Optional[Path]:
        for d in preferred_dirs:
            p=d/name
            if p.is_file():return p
        hits=[p for p in source_dir.rglob(name) if p.is_file() and "_history" not in p.parts]
        if not hits:return None
        hits.sort(key=lambda p:("quality_v4_3" not in p.parts,"parsed_v4_3" not in p.parts,len(p.parts)))
        return hits[0]
    return {
        "image":locate(image_name),"native_text":locate(stem+"__native_text.csv"),
        "vector_primitives":locate(stem+"__vector_primitives.json"),"auto_vector_bars":locate(stem+"__auto_vector_bars.csv"),
        "digitization_report":locate(stem+"__digitization_report.json"),
    }


def _study_human_review(study_dir: Path) -> pd.DataFrame:
    if not USE_HUMAN_REVIEW_QUEUE:return pd.DataFrame()
    frame=read_csv_if_exists(study_dir/"HUMAN_REVIEW_QUEUE.v4_3.csv")
    if frame.empty or "review_type" not in frame:return pd.DataFrame()
    return frame[frame["review_type"].astype(str).str.contains("FIGURE_DIGITIZATION",case=False,na=False)].copy()


def _priority_tier(screening_class, decision):
    cls=clean_str(screening_class).upper()
    if decision=="RECONCILE_FIRST":return "R_RECONCILE_FIRST"
    class_rank={"A":1,"B":2,"D":2,"C":3,"E":4}.get(cls,5)
    gap_rank=0 if decision=="DIGITIZE_REQUIRED" else 1
    return f"P{class_rank}_{'GAP' if gap_rank==0 else 'QUEUE'}"


def discover_figure_items() -> pd.DataFrame:
    rows=[]
    study_dirs=[p for p in SOURCE_RUN_DIR.iterdir() if p.is_dir() and p.name!="_history"]
    for discovery_order,study_dir in enumerate(sorted(study_dirs,key=lambda p:p.name),1):
        handoff_path=study_dir/"STUDY_HANDOFF.v4_3.json"
        handoff=read_json_if_exists(handoff_path)
        if not isinstance(handoff,dict):continue
        study_id=clean_str(handoff.get("study_id")) or study_dir.name
        screening_manifest=handoff.get("screening_manifest") or []
        manifest_class="";manifest_order=discovery_order
        if screening_manifest:
            manifest_class=clean_str(screening_manifest[0].get("screening_class"))
            try:manifest_order=int(float(screening_manifest[0].get("batch_order") or discovery_order))
            except Exception:manifest_order=discovery_order
        human=_study_human_review(study_dir)

        for source_rank_order,source in enumerate(handoff.get("sources") or [],1):
            if not source.get("current_generation_available"):continue
            source_meta=source.get("source_metadata") or {}
            source_file_id=clean_str(source.get("source_file_id"))
            source_folder=clean_str(source.get("source_folder"))
            source_dir=study_dir/Path(source_folder.replace(chr(92),"/"))
            source_role=clean_str(source_meta.get("role"))
            screening_class=clean_str(source_meta.get("screening_class")) or manifest_class
            if screening_class and screening_class not in INCLUDE_SCREENING_CLASSES:continue
            paths=source.get("paths") or {}
            inv_rel=clean_str(paths.get("figure_inventory.csv"));queue_rel=clean_str(paths.get("figure_digitization_queue.csv"))
            inv_path=(study_dir/Path(inv_rel.replace(chr(92),"/"))) if inv_rel else None
            queue_path=(study_dir/Path(queue_rel.replace(chr(92),"/"))) if queue_rel else None
            if inv_path is None or not inv_path.is_file():
                hits=[p for p in source_dir.rglob("figure_inventory.csv") if "_history" not in p.parts];inv_path=hits[0] if hits else None
            if queue_path is None or not queue_path.is_file():
                hits=[p for p in source_dir.rglob("figure_digitization_queue.csv") if "_history" not in p.parts];queue_path=hits[0] if hits else None
            inv_df=read_csv_if_exists(inv_path) if inv_path else pd.DataFrame();queue_df=read_csv_if_exists(queue_path) if queue_path else pd.DataFrame()
            inv_by_idx={}
            for rec in inv_df.to_dict("records"):
                idx=int_or_none(rec.get("figure_index"))
                if idx is not None:inv_by_idx[idx]=rec

            queue_by_idx={}
            for rec in queue_df.to_dict("records"):
                idx=int_or_none(rec.get("figure_index"))
                if idx is not None:queue_by_idx[idx]=rec
            human_by_idx={}
            if not human.empty:
                hs=human[human.get("source_file_id",pd.Series(dtype=str)).astype(str).eq(source_file_id)] if source_file_id else human.iloc[0:0]
                for rec in hs.to_dict("records"):
                    idx=int_or_none(rec.get("figure_index"))
                    if idx is not None:human_by_idx[idx]=rec

            if DISCOVERY_POLICY=="QUEUE_ALIGNED":
                indices=sorted(set(queue_by_idx)|set(human_by_idx))
                if not indices and ALLOW_LIKELY_QUANTITATIVE_FALLBACK:
                    indices=sorted(idx for idx,rec in inv_by_idx.items() if as_bool(rec.get("likely_quantitative")))
            elif DISCOVERY_POLICY=="LIKELY_QUANTITATIVE":
                indices=sorted(idx for idx,rec in inv_by_idx.items() if as_bool(rec.get("likely_quantitative")))
            else:
                raise ValueError("DISCOVERY_POLICY must be QUEUE_ALIGNED or LIKELY_QUANTITATIVE")

            for idx in indices:
                merged={**inv_by_idx.get(idx,{}),**queue_by_idx.get(idx,{})}
                h=human_by_idx.get(idx,{})
                assets=locate_figure_asset(source_dir,idx,merged.get("image_file",""))
                item_id=f"{study_id}::{source_file_id or safe_name(source_folder,24)}::FIG{idx:03d}"
                queue_sources=[]
                if idx in queue_by_idx:queue_sources.append("FIGURE_DIGITIZATION_QUEUE")
                if idx in human_by_idx:queue_sources.append("HUMAN_REVIEW_QUEUE")
                if not queue_sources:queue_sources.append("LIKELY_QUANTITATIVE_FALLBACK")
                queue_reason=" | ".join(x for x in [clean_str(queue_by_idx.get(idx,{}).get("reason")),clean_str(h.get("reason"))] if x)
                source_path_abs=resolve_source_path(source_meta)
                row={
                    "figure_item_id":item_id,"study_id":study_id,"study_dir":str(study_dir),"source_dir":str(source_dir),
                    "source_file_id":source_file_id,"source_file_name":clean_str(source_meta.get("source_file_name")),
                    "source_role":source_role,"screening_class":screening_class,"ml_role":clean_str(source_meta.get("ml_role")),
                    "dataset_role":clean_str(source_meta.get("dataset_role")),"manifest_order":manifest_order,"source_rank_order":source_rank_order,
                    "figure_index":idx,"page_no":int_or_none(merged.get("page_no")),"caption":clean_str(merged.get("caption")),
                    "nearby_page_text":clean_str(merged.get("nearby_page_text")),"bbox_l":merged.get("bbox_l",""),"bbox_t":merged.get("bbox_t",""),
                    "bbox_r":merged.get("bbox_r",""),"bbox_b":merged.get("bbox_b",""),"bbox_origin":clean_str(merged.get("bbox_origin")),
                    "work_priority_hint":clean_str(merged.get("work_priority_hint")),"upstream_auto_status":clean_str(merged.get("auto_status")),
                    "upstream_digitization_status":clean_str(merged.get("digitization_status")),"queue_source":"+".join(queue_sources),
                    "queue_reason":queue_reason,"image_path":str(assets["image"] or ""),"native_text_path":str(assets["native_text"] or ""),
                    "vector_primitives_path":str(assets["vector_primitives"] or ""),"auto_vector_bars_path":str(assets["auto_vector_bars"] or ""),
                    "digitization_report_path":str(assets["digitization_report"] or ""),"source_path_abs":str(source_path_abs or ""),
                    "source_sha256":clean_str(source_meta.get("sha256")),"handoff_path":str(handoff_path),
                    "source_pipeline_revision":clean_str(handoff.get("handoff_version")),
                }
                row.update(master_alignment_for_item(row))
                row["priority_tier"]=_priority_tier(screening_class,row["digitization_decision"])
                rows.append(row)
    return pd.DataFrame(rows)


def select_figure_items(items: pd.DataFrame) -> pd.DataFrame:
    if items.empty:return items.copy()
    df=items.copy()
    if RUN_MODE.upper()=="PILOT":
        wanted={clean_str(x) for x in PILOT_STUDY_IDS if clean_str(x)};df=df[df["study_id"].isin(wanted)].copy()
    elif RUN_MODE.upper() not in {"QUEUE","ALL_READY"}:
        raise ValueError("RUN_MODE must be PILOT or QUEUE")
    if not INCLUDE_RECONCILE_FIRST:
        df=df[~df["digitization_decision"].eq("RECONCILE_FIRST")].copy()
    class_rank={"A":0,"B":1,"D":1,"C":2,"E":3}
    decision_rank={"DIGITIZE_REQUIRED":0,"DIGITIZE_OR_RECONCILE":1,"RECONCILE_FIRST":9}
    df["_class_rank"]=df["screening_class"].map(class_rank).fillna(9)
    df["_decision_rank"]=df["digitization_decision"].map(decision_rank).fillna(5)
    df["_role_rank"]=df["source_role"].map({"FULLTEXT":0,"SUPPLEMENT":1}).fillna(9)
    df=df.sort_values(["_class_rank","_decision_rank","manifest_order","study_id","_role_rank","source_rank_order","figure_index"],kind="stable")
    df=df.drop(columns=["_class_rank","_decision_rank","_role_rank"])
    if MAX_FIGURES is not None:df=df.head(int(MAX_FIGURES))
    return df.reset_index(drop=True)


DISCOVERED_ITEMS=discover_figure_items()
SELECTED_ITEMS=select_figure_items(DISCOVERED_ITEMS)
ALIGNMENT_COLUMNS=["figure_item_id","study_id","screening_class","source_file_id","source_role","figure_index","source_figure_label",
    "page_no","queue_source","queue_reason","master_gap_status","digitization_decision","priority_tier",
    "existing_figure_provenance_count","existing_supported_measurement_count","existing_supported_measurement_ids",
    "master_pending_figure_provenance_count","master_source_digitization_note","caption","image_path"]
QUEUE_ALIGNMENT_DF=DISCOVERED_ITEMS.reindex(columns=ALIGNMENT_COLUMNS)
if EXECUTION_MODE!="AUDIT_EXISTING":_write_resume_text(QUEUE_ALIGNMENT_PATH,QUEUE_ALIGNMENT_DF.to_csv(index=False))
print("Queue-aligned figure items:",len(DISCOVERED_ITEMS))
print("Reconcile-first items:",int(DISCOVERED_ITEMS.get("digitization_decision",pd.Series(dtype=str)).eq("RECONCILE_FIRST").sum()))
print("Selected for figure processing:",len(SELECTED_ITEMS))
print("Queue alignment:",QUEUE_ALIGNMENT_PATH)
if not QUEUE_ALIGNMENT_DF.empty:
    display(QUEUE_ALIGNMENT_DF.head(30))


In [ ]:
# ============================================================
# 7. Input signatures + manual-review protection
# ============================================================

def path_or_none(value) -> Optional[Path]:
    value = clean_str(value)
    return Path(value) if value else None


def item_input_signature(item: Dict[str, Any]) -> Dict[str, Any]:
    paths = {
        "actual_source_pdf": path_or_none(item.get("source_path_abs")),
        "manual_calibration": calibration_path(item),
        "image": path_or_none(item.get("image_path")),
        "native_text": path_or_none(item.get("native_text_path")),
        "vector_primitives": path_or_none(item.get("vector_primitives_path")),
        "auto_vector_bars": path_or_none(item.get("auto_vector_bars_path")),
        "digitization_report": path_or_none(item.get("digitization_report_path")),
    }
    metadata = {k: item.get(k) for k in [
        "study_id", "source_file_id", "source_file_name", "source_role", "screening_class", "ml_role",
        "figure_index", "page_no", "caption", "bbox_l", "bbox_t", "bbox_r", "bbox_b", "bbox_origin",
        "source_sha256", "source_pipeline_revision",
    ]}
    out = {"metadata": metadata, "files": {k: file_signature(v) for k, v in paths.items()}}
    out["signature_hash"] = hashlib.sha256(json.dumps(out, sort_keys=True, default=str).encode()).hexdigest()
    return out


def item_output_dir(item: Dict[str, Any]) -> Path:
    return (
        FIGURE_OUTPUT_DIR
        / safe_name(item["study_id"], 80)
        / safe_name(item.get("source_file_id") or item.get("source_file_name") or "source", 80)
        / f"FIG_{int(item['figure_index']):03d}"
    )


def item_checkpoint_path(item: Dict[str, Any]) -> Path:
    key = hashlib.sha256(item["figure_item_id"].encode()).hexdigest()[:24]
    return CHECKPOINT_DIR / f"{safe_name(item['study_id'], 60)}__{key}.json"


def manual_review_for(item_id):
    return None  # review is candidate-level and never skips unreviewed sibling series


def output_manifest(out_dir: Path) -> List[Dict[str, Any]]:
    files = []
    if not out_dir.is_dir():
        return files
    for p in sorted(out_dir.rglob("*")):
        if p.is_file():
            files.append({"path": p.relative_to(out_dir).as_posix(), "size_bytes": int(p.stat().st_size), "sha256": sha256_file(p)})
    return files


def output_manifest_current(out_dir: Path, manifest: Any) -> bool:
    if not isinstance(manifest, list) or not manifest:
        return False
    for rec in manifest:
        try:
            rel = Path(rec["path"])
            if rel.is_absolute() or ".." in rel.parts:
                return False
            p = out_dir / rel
            if not p.is_file() or p.stat().st_size != int(rec["size_bytes"]):
                return False
            if rec.get("sha256") and sha256_file(p) != rec["sha256"]:
                return False
        except Exception:
            return False
    return True


def completed_checkpoint_current(item: Dict[str, Any], cp: Optional[Dict[str, Any]] = None, *, respect_reprocess: bool = True) -> bool:
    if respect_reprocess and (FORCE_REPROCESS or item["figure_item_id"] in REPROCESS_ITEM_IDS):
        return False
    cp = cp or read_checkpoint(item_checkpoint_path(item))
    if not isinstance(cp, dict) or cp.get("_resume_state") != "COMPLETE":
        return False
    if cp.get("pipeline_fingerprint") != pipeline_fingerprint():
        return False
    current_sig = item_input_signature(item)
    if cp.get("input_signature", {}).get("signature_hash") != current_sig.get("signature_hash"):
        return False
    if not output_manifest_current(item_output_dir(item), cp.get("output_files")):
        return False
    return True


In [ ]:
# ============================================================
# 8. Load image/context and map figure pixels <-> PDF coordinates
# ============================================================

def load_image(path: Path) -> Image.Image:
    with Image.open(path) as img:
        return img.convert("RGB").copy()


def load_native_text(item: Dict[str, Any]) -> pd.DataFrame:
    p = path_or_none(item.get("native_text_path"))
    if p and p.is_file():
        try:
            df = pd.read_csv(p, keep_default_na=False)
            for c in ["x0", "y0", "x1", "y1"]:
                if c in df.columns:
                    df[c] = pd.to_numeric(df[c], errors="coerce")
            return df
        except Exception:
            pass
    return pd.DataFrame(columns=["text", "x0", "y0", "x1", "y1"])


def load_vector_primitives(item: Dict[str, Any]) -> List[Dict[str, Any]]:
    p = path_or_none(item.get("vector_primitives_path"))
    data = read_json_if_exists(p) if p else None
    return data if isinstance(data, list) else []


def load_existing_auto_bars(item: Dict[str, Any]) -> pd.DataFrame:
    p = path_or_none(item.get("auto_vector_bars_path"))
    return read_csv_if_exists(p) if p else pd.DataFrame()


def load_existing_digitization_report(item: Dict[str, Any]) -> Dict[str, Any]:
    p = path_or_none(item.get("digitization_report_path"))
    return read_json_if_exists(p) or {} if p else {}


def figure_rect_fitz(item: Dict[str, Any]) -> Optional[fitz.Rect]:
    try:
        l, t, r, b = [float(item[k]) for k in ("bbox_l", "bbox_t", "bbox_r", "bbox_b")]
    except Exception:
        return None
    source_path = path_or_none(item.get("source_path_abs"))
    page_no = int_or_none(item.get("page_no"))
    if not source_path or not source_path.is_file() or not page_no:
        return fitz.Rect(min(l, r), min(t, b), max(l, r), max(t, b))
    try:
        with fitz.open(source_path) as pdf:
            if page_no < 1 or page_no > len(pdf):
                return None
            h = float(pdf[page_no - 1].rect.height)
        origin = clean_str(item.get("bbox_origin")).upper()
        if "BOTTOM" in origin:
            y0, y1 = h - max(t, b), h - min(t, b)
        else:
            y0, y1 = min(t, b), max(t, b)
        return fitz.Rect(min(l, r), y0, max(l, r), y1)
    except Exception:
        return fitz.Rect(min(l, r), min(t, b), max(l, r), max(t, b))


def panel_to_pdf_rect(panel: Dict[str, Any], fig_rect: Optional[fitz.Rect], image_size: Tuple[int, int]) -> Optional[fitz.Rect]:
    if fig_rect is None:
        return None
    w, h = image_size
    if w <= 0 or h <= 0:
        return None
    x0 = fig_rect.x0 + (panel["x0"] / w) * fig_rect.width
    x1 = fig_rect.x0 + (panel["x1"] / w) * fig_rect.width
    y0 = fig_rect.y0 + (panel["y0"] / h) * fig_rect.height
    y1 = fig_rect.y0 + (panel["y1"] / h) * fig_rect.height
    return fitz.Rect(x0, y0, x1, y1)


def pdf_to_figure_pixel(x_pdf: float, y_pdf: float, fig_rect: fitz.Rect, image_size: Tuple[int, int]) -> Tuple[float, float]:
    w, h = image_size
    return ((x_pdf - fig_rect.x0) / fig_rect.width * w, (y_pdf - fig_rect.y0) / fig_rect.height * h)


def figure_pixel_to_pdf(x_px: float, y_px: float, fig_rect: fitz.Rect, image_size: Tuple[int, int]) -> Tuple[float, float]:
    w, h = image_size
    return (fig_rect.x0 + (x_px / w) * fig_rect.width, fig_rect.y0 + (y_px / h) * fig_rect.height)


def filter_words_to_rect(words: pd.DataFrame, rect: Optional[fitz.Rect]) -> pd.DataFrame:
    if rect is None or words.empty:
        return words.copy()
    xc = (words["x0"] + words["x1"]) / 2
    yc = (words["y0"] + words["y1"]) / 2
    return words[(xc >= rect.x0 - 2) & (xc <= rect.x1 + 2) & (yc >= rect.y0 - 2) & (yc <= rect.y1 + 2)].copy()


def drawing_rect(d: Dict[str, Any]) -> Optional[fitz.Rect]:
    try:
        x0, y0, x1, y1 = map(float, d.get("rect") or [])
        return fitz.Rect(x0, y0, x1, y1)
    except Exception:
        return None


def filter_drawings_to_rect(drawings: List[Dict[str, Any]], rect: Optional[fitz.Rect]) -> List[Dict[str, Any]]:
    if rect is None:
        return list(drawings)
    out = []
    for d in drawings:
        dr = drawing_rect(d)
        if dr is not None and rect.intersects(dr):
            out.append(d)
    return out


def extract_figure_label(caption: str, figure_index: int) -> str:
    m = FIGURE_LABEL_RE.search(caption or "")
    return f"Figure {m.group(1)}" if m else f"Docling figure {figure_index}"


def extract_unit_candidates(words: pd.DataFrame, caption: str) -> List[str]:
    text = " ".join([caption or ""] + words.get("text", pd.Series(dtype=str)).astype(str).tolist())
    chunks = re.findall(r"(?:%|°C|mM|µM|μM|mmol|µmol|μmol|mg|µg|μg|g\s*(?:FW|DW)?|cm|mm|MPa|kPa|dS\s*m[−-]?1|mol\s*CO2|m[−-]?2|s[−-]?1)", text, flags=re.I)
    seen, out = set(), []
    for c in chunks:
        key = c.strip()
        if key and key.lower() not in seen:
            seen.add(key.lower())
            out.append(key)
    return out[:20]


In [ ]:
# ============================================================
# 9. Conservative panel segmentation
# ============================================================

def _runs(mask: np.ndarray) -> List[Tuple[int, int]]:
    out, start = [], None
    for i, flag in enumerate(mask.tolist()):
        if flag and start is None:
            start = i
        elif not flag and start is not None:
            out.append((start, i))
            start = None
    if start is not None:
        out.append((start, len(mask)))
    return out


def _candidate_gutters(gray: np.ndarray, axis: int, min_frac: float) -> List[Tuple[int, int, float]]:
    # Full-height/full-width white gutters are much safer than arbitrary whitespace inside a chart.
    white_fraction = (gray >= 248).mean(axis=axis)
    dark_fraction = (gray <= 235).mean(axis=axis)
    mask = (white_fraction >= 0.992) & (dark_fraction <= 0.004)
    dim = gray.shape[1] if axis == 0 else gray.shape[0]
    min_width = max(8, int(dim * min_frac))
    gutters = []
    for a, b in _runs(mask):
        if b - a < min_width:
            continue
        if a < int(0.08 * dim) or b > int(0.92 * dim):
            continue
        gutters.append((a, b, float(white_fraction[a:b].mean())))
    return gutters


def segment_panels(image: Image.Image, caption: str) -> List[Dict[str, Any]]:
    w, h = image.size
    full = [{"panel_id": "P01", "x0": 0, "y0": 0, "x1": w, "y1": h, "split_method": "WHOLE_FIGURE", "split_confidence": 1.0}]
    if not AUTO_PANEL_SPLIT or w < 200 or h < 160:
        return full

    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    panel_labels = len(set(PANEL_LABEL_RE.findall(caption or "")))
    # Conservative by design: ordinary bar gaps can look like white gutters.
    # Without explicit multi-panel labels in the caption, keep the whole figure.
    if panel_labels < 2:
        return full
    min_frac = 0.025
    vg = _candidate_gutters(gray, axis=0, min_frac=min_frac)  # reduce rows -> columns
    hg = _candidate_gutters(gray, axis=1, min_frac=min_frac)  # reduce cols -> rows

    def choose(gutters, dim):
        # Keep only strong gutters whose adjacent regions remain large.
        chosen = []
        for a, b, score in sorted(gutters, key=lambda z: (-(z[1]-z[0]), -z[2])):
            mid = (a + b) / 2
            if mid < 0.22 * dim or mid > 0.78 * dim:
                continue
            chosen.append((a, b, score))
            if len(chosen) >= 2:
                break
        return sorted(chosen)

    vg, hg = choose(vg, w), choose(hg, h)
    # Without caption evidence, require very strong and wide whitespace before splitting.
    if panel_labels < 2:
        vg = [g for g in vg if (g[1]-g[0]) >= 0.06*w and g[2] >= 0.997]
        hg = [g for g in hg if (g[1]-g[0]) >= 0.06*h and g[2] >= 0.997]

    xcuts = [0] + [int((a+b)/2) for a,b,_ in vg] + [w]
    ycuts = [0] + [int((a+b)/2) for a,b,_ in hg] + [h]
    candidates = []
    for yi in range(len(ycuts)-1):
        for xi in range(len(xcuts)-1):
            x0, x1 = xcuts[xi], xcuts[xi+1]
            y0, y1 = ycuts[yi], ycuts[yi+1]
            if (x1-x0) < 0.20*w or (y1-y0) < 0.20*h:
                continue
            candidates.append((x0,y0,x1,y1))
    if len(candidates) <= 1 or len(candidates) > MAX_PANELS:
        return full

    result = []
    confidence = 0.92 if panel_labels >= len(candidates) else 0.80
    for i,(x0,y0,x1,y1) in enumerate(candidates,1):
        result.append({
            "panel_id": f"P{i:02d}", "x0": int(x0), "y0": int(y0), "x1": int(x1), "y1": int(y1),
            "split_method": "WHITE_GUTTER", "split_confidence": confidence,
        })
    return result


def save_panel_crops(image: Image.Image, panels: List[Dict[str, Any]], out_dir: Path):
    panel_dir = out_dir / "panels"
    panel_dir.mkdir(parents=True, exist_ok=True)
    for panel in panels:
        p = panel_dir / f"{panel['panel_id']}.png"
        crop = image.crop((panel["x0"], panel["y0"], panel["x1"], panel["y1"]))
        temp = p.with_suffix(".tmp.png")
        crop.save(temp)
        os.replace(temp, p)


In [ ]:
# ============================================================
# 10. Geometry diagnostics + figure-type candidate
# ============================================================

def normalize_hough_segments(lines) -> np.ndarray:
    """Normalize OpenCV HoughLinesP output, without changing its coordinates.

    Supports (N, 4), (N, 1, 4), (1, N, 4), a single (4,) segment,
    and None/empty results. Unexpected shapes fail explicitly.
    """
    if lines is None:
        return np.empty((0, 4), dtype=np.float64)
    arr = np.asarray(lines)
    if arr.size == 0:
        return np.empty((0, 4), dtype=np.float64)
    valid = (
        (arr.ndim == 1 and arr.shape == (4,))
        or (arr.ndim == 2 and arr.shape[1] == 4)
        or (arr.ndim == 3 and arr.shape[-1] == 4 and 1 in arr.shape[:2])
    )
    if not valid:
        raise ValueError(f"Unexpected HoughLinesP shape: {arr.shape}; expected segments of four coordinates")
    result = arr.reshape(-1, 4)
    if not np.isfinite(result).all():
        raise ValueError("HoughLinesP returned non-finite coordinates")
    return result


def image_geometry_features(image: Image.Image) -> Dict[str, Any]:
    arr = np.array(image.convert("RGB"))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 80, 180)
    h, w = gray.shape
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=max(30, int(min(w,h)*0.12)),
                            minLineLength=max(20, int(min(w,h)*0.18)), maxLineGap=max(4, int(min(w,h)*0.02)))
    horizontal = vertical = diagonal = 0
    if lines is not None:
        for line in normalize_hough_segments(lines):
            x1,y1,x2,y2 = map(float,line)
            dx,dy = abs(x2-x1),abs(y2-y1)
            if dx > 4*max(dy,1): horizontal += 1
            elif dy > 4*max(dx,1): vertical += 1
            else: diagonal += 1
    contours,_ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    rect_like = 0
    for c in contours:
        x,y,cw,ch = cv2.boundingRect(c)
        if cw < 5 or ch < 5: continue
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.03*peri, True)
        if len(approx) == 4 and 0.002*w*h <= cw*ch <= 0.25*w*h:
            rect_like += 1
    small = cv2.resize(arr, (min(160,w), min(160,h)), interpolation=cv2.INTER_AREA)
    quant = (small // 32).reshape(-1,3)
    color_bins = int(len(np.unique(quant, axis=0)))
    return {
        "width_px": w, "height_px": h,
        "edge_density": float((edges>0).mean()),
        "hough_horizontal": horizontal, "hough_vertical": vertical, "hough_diagonal": diagonal,
        "rect_like_contours": rect_like, "quantized_color_bins": color_bins,
    }


def classify_figure(item: Dict[str, Any], features: Dict[str, Any], panels: List[Dict[str, Any]],
                    words: pd.DataFrame, drawings: List[Dict[str, Any]], existing_bars: pd.DataFrame) -> Dict[str, Any]:
    caption = item.get("caption", "") or ""
    scores = {k: 0.0 for k in ["bar", "line_or_timecourse", "scatter", "box_violin", "heatmap", "ordination", "photograph", "mixed_or_unknown"]}
    if BAR_HINT_RE.search(caption): scores["bar"] += 3
    if LINE_HINT_RE.search(caption): scores["line_or_timecourse"] += 3
    if SCATTER_HINT_RE.search(caption): scores["scatter"] += 3
    if BOX_HINT_RE.search(caption): scores["box_violin"] += 4
    if HEATMAP_HINT_RE.search(caption): scores["heatmap"] += 4
    if ORDINATION_HINT_RE.search(caption): scores["ordination"] += 4
    if PHOTO_HINT_RE.search(caption): scores["photograph"] += 4
    if not existing_bars.empty: scores["bar"] += 6
    if features.get("rect_like_contours", 0) >= 4 and features.get("hough_horizontal",0) and features.get("hough_vertical",0): scores["bar"] += 2
    if features.get("hough_diagonal",0) >= 3 and features.get("hough_horizontal",0) and features.get("hough_vertical",0): scores["line_or_timecourse"] += 1.5
    if features.get("quantized_color_bins",0) > 120 and features.get("rect_like_contours",0) > 20: scores["heatmap"] += 1
    if len(panels) > 1:
        scores["mixed_or_unknown"] += 1
    # Native vector paths provide strong evidence for a chart, but not a safe subtype alone.
    if drawings and scores["bar"] == 0 and scores["line_or_timecourse"] == 0:
        scores["mixed_or_unknown"] += 1
    best = max(scores, key=scores.get)
    if scores[best] <= 0:
        best = "mixed_or_unknown"
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    margin = ranked[0][1] - ranked[1][1] if len(ranked)>1 else ranked[0][1]
    confidence = max(0.2, min(0.98, 0.50 + 0.10*margin + 0.03*ranked[0][1]))
    return {"figure_type_candidate": best, "classification_scores": scores, "classification_confidence": confidence,
            "panel_count": len(panels), "native_text_tokens": int(len(words)), "vector_primitive_count": int(len(drawings))}


In [ ]:
def _candidate_base(item, panel_id, method, figure_type):
    return {
        "figure_item_id": item["figure_item_id"], "study_id": item["study_id"], "source_file_id": item.get("source_file_id",""),
        "figure_index": int(item["figure_index"]), "figure_label": extract_figure_label(item.get("caption", ""), int(item["figure_index"])),
        "page_no": item.get("page_no"), "panel_id": panel_id, "digitization_method": method,
        "figure_type_candidate": figure_type, "verification_status": "UNVERIFIED", "eligible_for_database": False,
        "uncertainty_candidate": None, "uncertainty_type_candidate": "", "outcome_raw_candidate": "",
        "unit_raw_candidate": "", "target_entity_candidate": "", "possible_duplicate": False,
        "duplicate_evidence": "", "notes": "",
    }

def semantic_target_hint(caption: str) -> str:
    if CONTRAST_HINT_RE.search(caption or ""):
        return "CONTRAST_OR_TRANSFORMED_QUANTITY"
    if ORDINATION_HINT_RE.search(caption or "") or HEATMAP_HINT_RE.search(caption or ""):
        return "DATA_ASSET_OR_MECHANISTIC_ONLY"
    return "ARM_MEASUREMENT_POSSIBLE"


In [ ]:
# ============================================================
# Enhanced context: read-only PDF recovery, one-pass local OCR,
# immutable input binding, and bounded working images.
# All coordinates in the new digitizer are WORKING-IMAGE PIXELS.
# ============================================================
import shutil
import sys
import subprocess
from PIL import ImageDraw

WORD_COLUMNS = ["text", "x0", "y0", "x1", "y1", "origin", "confidence", "axis_hint", "panel_hint", "tick_position_px"]


def _plain_words(df):
    if df is None or df.empty:
        return pd.DataFrame(columns=WORD_COLUMNS)
    df = df.copy()
    for c in WORD_COLUMNS:
        if c not in df:
            df[c] = 1.0 if c == "confidence" else ("NATIVE" if c == "origin" else "")
    for c in ["x0", "y0", "x1", "y1", "confidence", "tick_position_px"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["x0", "y0", "x1", "y1"])
    df["text"] = df["text"].map(clean_str)
    return df[WORD_COLUMNS].reset_index(drop=True)


def working_image_path(item):
    return item_output_dir(item) / "context" / "working.png"


def context_meta(item):
    return read_json_if_exists(item_output_dir(item) / "context" / "context.json") or {}


def load_working_image(item):
    return load_image(working_image_path(item))


def effective_words(item):
    folder = item_output_dir(item) / "context"
    p = folder / "words_effective.csv"
    if not p.is_file():
        p = folder / "words_native.csv"
    return _plain_words(read_csv_if_exists(p))


def effective_drawings(item):
    data = read_json_if_exists(item_output_dir(item) / "context" / "drawings_pixels.json")
    return data if isinstance(data, list) else []


def calibration_path(item):
    key = hashlib.sha256(item["figure_item_id"].encode()).hexdigest()[:24]
    return CALIBRATION_DIR / (safe_name(item["study_id"], 60) + "__" + key + ".json")


def load_calibration(item, validate_binding=True):
    p = calibration_path(item)
    if not p.exists():
        return {}
    try:
        data = json.loads(p.read_text(encoding="utf-8-sig"))
    except Exception as exc:
        raise ValueError(f"Invalid manual calibration file: {p}") from exc
    if not isinstance(data, dict) or data.get("figure_item_id") != item["figure_item_id"]:
        raise ValueError("Manual calibration belongs to a different figure")
    if validate_binding:
        meta = context_meta(item)
        if data.get("working_image_sha256") != meta.get("working_image_sha256"):
            raise ValueError("STALE_MANUAL_CALIBRATION: working image changed; re-open the editor")
    return data


def _save_png_atomic(image, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".writing.png")
    try:
        image.save(tmp, format="PNG")
        os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)


def _bounded_image(image, max_pixels=MAX_WORKING_PIXELS):
    scale = min(1.0, math.sqrt(max_pixels / max(1, image.width * image.height)))
    if scale < 1.0:
        return image.resize((max(1, round(image.width * scale)), max(1, round(image.height * scale))), Image.Resampling.LANCZOS)
    return image


def _drawing_as_pixels(d, origin_x, origin_y, scale):
    def point(p):
        return [(float(p[0]) - origin_x) * scale, (float(p[1]) - origin_y) * scale]
    out = {k: d.get(k) for k in ["type", "fill", "color", "width", "fill_opacity", "stroke_opacity", "closePath"]}
    r = d.get("rect")
    out["rect"] = [*point((r.x0, r.y0)), *point((r.x1, r.y1))] if r is not None else []
    out["width"] = float(d.get("width") or 0) * scale
    out["items"] = []
    for it in d.get("items", []):
        op = it[0]
        if op == "l":
            out["items"].append(["l", point(it[1]), point(it[2])])
        elif op == "c":
            out["items"].append(["c", *[point(p) for p in it[1:5]]])
        elif op == "re":
            rr = it[1]
            out["items"].append(["re", [*point((rr.x0, rr.y0)), *point((rr.x1, rr.y1))]])
        elif op == "qu":
            q = it[1]
            out["items"].append(["qu", *[point(p) for p in [q.ul, q.ur, q.lr, q.ll]]])
    return out


def prepare_context(item, out_dir):
    folder = out_dir / "context"
    folder.mkdir(parents=True, exist_ok=True)
    raw = path_or_none(item.get("image_path"))
    pdf_path = path_or_none(item.get("source_path_abs"))
    meta = {"figure_item_id": item["figure_item_id"], "coordinate_system": "WORKING_IMAGE_PIXELS_TOPLEFT", "warnings": [], "pdf_mapping": None}
    rows, drawings, image = [], [], None
    if RECOVER_FROM_SOURCE_PDF and pdf_path and pdf_path.is_file() and pdf_path.suffix.lower() == ".pdf":
        actual_hash = cached_sha256(pdf_path)
        expected_hash = clean_str(item.get("source_sha256"))
        if expected_hash and actual_hash and expected_hash != actual_hash:
            raise ValueError("SOURCE_PDF_HASH_MISMATCH: source PDF no longer matches its Docling generation")
        try:
            l, t, r, b = [float(item[k]) for k in ("bbox_l", "bbox_t", "bbox_r", "bbox_b")]
            if not all(math.isfinite(x) for x in [l, t, r, b]):
                raise ValueError("non-finite source bounding box")
            page_no = int(item["page_no"])
            with fitz.open(pdf_path) as doc:
                if not 1 <= page_no <= len(doc):
                    raise ValueError("page index outside source PDF")
                page = doc[page_no - 1]
                original_rotation = int(page.rotation)
                if "BOTTOM" in clean_str(item.get("bbox_origin")).upper():
                    y0, y1 = page.rect.height - max(t, b), page.rect.height - min(t, b)
                else:
                    y0, y1 = min(t, b), max(t, b)
                clip = fitz.Rect(min(l, r), y0, max(l, r), y1)
                if original_rotation:
                    clip = clip * page.derotation_matrix
                    page.set_rotation(0)  # In-memory only: the source PDF is NEVER saved.
                clip = clip & page.rect
                if clip.is_empty or clip.width < 12 or clip.height < 12:
                    raise ValueError("missing/invalid figure bounding box")
                zoom = min(PDF_RENDER_DPI / 72.0, math.sqrt(MAX_WORKING_PIXELS / (clip.width * clip.height)))
                pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), clip=clip, colorspace=fitz.csRGB, alpha=False, annots=False)
                image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                # Account for integer rounding of a clipped pixmap, not just clip.width.
                ox, oy = pix.x / zoom, pix.y / zoom
                for word in page.get_text("words", sort=False):
                    x0, y0, x1, y1, text = word[:5]
                    if clip.contains(fitz.Point((x0 + x1) / 2, (y0 + y1) / 2)):
                        rows.append({"text": text, "x0": (x0 - ox)*zoom, "y0": (y0 - oy)*zoom,
                                     "x1": (x1 - ox)*zoom, "y1": (y1 - oy)*zoom, "origin": "NATIVE_PDF", "confidence": 1.0})
                for d in page.get_drawings():
                    dr = d.get("rect")
                    # Zero-area line rectangles need an expanded intersection test.
                    if dr is not None and clip.intersects(fitz.Rect(dr.x0-1, dr.y0-1, dr.x1+1, dr.y1+1)):
                        drawings.append(_drawing_as_pixels(d, ox, oy, zoom))
                meta.update({"image_source": "SOURCE_PDF_RENDER", "source_pdf_sha256": actual_hash,
                             "original_page_rotation": original_rotation,
                             "pdf_mapping": {"origin_x": ox, "origin_y": oy, "scale": zoom, "page_no": page_no},
                             "source_bbox_pdf": list(clip)})
        except Exception as exc:
            image, rows, drawings = None, [], []
            meta["warnings"].append("PDF_CONTEXT_RECOVERY_FAILED: " + repr(exc))
    if image is None:
        if not raw or not raw.is_file():
            raise FileNotFoundError("No usable original figure image and no renderable source PDF crop")
        image = _bounded_image(load_image(raw))
        meta["image_source"] = "UPSTREAM_RASTER_IMAGE"
        # No invented PDF mapping when the source's coordinate origin is unknown.
        meta["warnings"].append("RASTER_ONLY: native sidecars are not assumed to use image coordinates")
    meta.update({"width_px": image.width, "height_px": image.height,
                 "native_word_count": len(rows), "vector_drawing_count": len(drawings)})
    _save_png_atomic(image, working_image_path(item))
    meta["working_image_sha256"] = sha256_file(working_image_path(item))
    _atomic_write_text(folder/"words_native.csv", pd.DataFrame(rows, columns=WORD_COLUMNS).to_csv(index=False))
    write_json_atomic(folder/"drawings_pixels.json", drawings)
    write_json_atomic(folder/"context.json", meta)
    write_json_atomic(out_dir/"asset_validation.json", {**meta, "valid": True})
    return meta


# A real local OCR adapter. It runs in a separate process with a timeout.
# Exactly ONE image_to_data / engine call is made per needed figure generation.
OCR_WORKER_SOURCE = r'''
import json, sys, os
from pathlib import Path
import numpy as np
from PIL import Image
request = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
rows = []
if request["backend"] == "TESSERACT":
    import pytesseract
    pytesseract.pytesseract.tesseract_cmd = request["tesseract_cmd"]
    result = pytesseract.image_to_data(request["image_path"], lang=request["language"],
        config="--psm 11", output_type=pytesseract.Output.DICT, timeout=request["timeout_seconds"])
    for i, text in enumerate(result.get("text", [])):
        confidence = float(result["conf"][i]) / 100.0
        if str(text).strip() and confidence >= request["min_confidence"]:
            x, y, w, h = [float(result[k][i]) for k in ["left", "top", "width", "height"]]
            rows.append(dict(text=str(text), x0=x, y0=y, x1=x+w, y1=y+h, confidence=confidence, origin="OCR_TESSERACT"))
elif request["backend"] == "RAPIDOCR":
    from rapidocr import RapidOCR
    engine = RapidOCR()
    result = engine(request["image_path"])
    boxes, texts, scores = result.boxes, result.txts, result.scores
    if boxes is not None and texts is not None:
        for box, text, confidence in zip(boxes, texts, scores):
            if str(text).strip() and float(confidence) >= request["min_confidence"]:
                a = np.asarray(box, dtype=float).reshape(-1, 2)
                rows.append(dict(text=str(text), x0=float(a[:,0].min()), y0=float(a[:,1].min()),
                    x1=float(a[:,0].max()), y1=float(a[:,1].max()), confidence=float(confidence), origin="OCR_RAPIDOCR"))
else:
    raise ValueError("Unknown OCR backend")
Path(request["output_path"]).write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
'''


def _run_local_ocr_uncached(item):
    if OCR_EFFECTIVE_BACKEND == "OFF":
        return pd.DataFrame(columns=WORD_COLUMNS), {"status": "OCR_UNAVAILABLE", "backend": "OFF"}
    worker = LOCAL_RESUME_DIR / "runtime" / "ocr_worker_v1_1.py"
    _atomic_write_text(worker, OCR_WORKER_SOURCE)
    with tempfile.TemporaryDirectory(prefix="synerplant_ocr_") as tmp:
        tmp = Path(tmp)
        local_image = tmp / "input.png"
        # Tesseract / cv2 paths do not need to support the Drive's extended Unicode path.
        atlas,tiles=build_axis_ocr_atlas(item)
        atlas.save(local_image)
        _save_png_atomic(atlas,item_output_dir(item)/"context"/"ocr_atlas.png")
        write_json_atomic(item_output_dir(item)/"context"/"ocr_atlas_map.json",tiles)
        request = {"backend": OCR_EFFECTIVE_BACKEND, "tesseract_cmd": TESSERACT_COMMAND,
                   "language": OCR_LANGUAGE, "timeout_seconds": OCR_TIMEOUT_SECONDS,
                   "min_confidence": OCR_MIN_CONFIDENCE, "image_path": str(local_image),
                   "output_path": str(tmp/"result.json")}
        (tmp/"request.json").write_text(json.dumps(request), encoding="utf-8")
        env = os.environ.copy()
        env.update({"OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "2"})
        try:
            with (tmp/"worker.log").open("w", encoding="utf-8") as log:
                proc = subprocess.run([sys.executable, str(worker), str(tmp/"request.json")],
                    stdout=log, stderr=subprocess.STDOUT, timeout=OCR_TIMEOUT_SECONDS+30, env=env, check=False)
            if proc.returncode != 0:
                error = (tmp/"worker.log").read_text(encoding="utf-8", errors="replace")[-6000:]
                raise RuntimeError(f"OCR worker exit {proc.returncode}: {error}")
            data = json.loads((tmp/"result.json").read_text(encoding="utf-8"))
            mapped=atlas_words_to_figure(_plain_words(pd.DataFrame(data)),tiles)
            return mapped, {"status": "OCR_COMPLETED", "backend": OCR_EFFECTIVE_BACKEND, "token_count": len(mapped), "passes": 1, "input": "AXIS_CONTACT_SHEET"}
        except Exception as exc:
            return pd.DataFrame(columns=WORD_COLUMNS), {"status": "OCR_FAILED", "backend": OCR_EFFECTIVE_BACKEND, "error": repr(exc), "passes": 1}


def merge_native_and_ocr(native, ocr):
    native, ocr = _plain_words(native), _plain_words(ocr)
    rows = native.to_dict("records")
    for row in ocr.to_dict("records"):
        rr = fitz.Rect(row["x0"], row["y0"], row["x1"], row["y1"])
        duplicate = False
        for nr in rows:
            nn = fitz.Rect(nr["x0"], nr["y0"], nr["x1"], nr["y1"])
            inter = rr & nn
            if not inter.is_empty and inter.get_area() >= 0.6 * min(rr.get_area(), nn.get_area()):
                duplicate = True
                break
        if not duplicate:
            rows.append(row)
    return pd.DataFrame(rows, columns=WORD_COLUMNS)


def recover_axis_text(item, out_dir):
    native = _plain_words(read_csv_if_exists(out_dir/"context"/"words_native.csv"))
    panels = _read_panels(out_dir)
    manual = load_calibration(item)
    failures = []
    for p in panels:
        cfg = next((m for m in manual.get("panels", []) if m.get("panel_id") == p["panel_id"]), {})
        if cfg.get("kind") == "skip":
            continue
        ycal = axis_calibration(native, p, "y", cfg.get("y_axis"))
        xcal = axis_calibration(native, p, "x", cfg.get("x_axis"))
        if not ycal.get("ok") and not xcal.get("ok"):
            failures.append(p["panel_id"])
        elif cfg.get("kind") in {"line", "scatter"} and not (ycal.get("ok") and xcal.get("ok")):
            failures.append(p["panel_id"])
    need_ocr = bool(failures)
    report = {"status": "NATIVE_TEXT_SUFFICIENT", "native_tokens": len(native), "panels_needing_text": failures, "passes": 0}
    ocr = pd.DataFrame(columns=WORD_COLUMNS)
    if need_ocr and ENABLE_LOCAL_OCR:
        ocr, ocr_report = run_local_ocr(item)
        report.update(ocr_report)
    elif need_ocr:
        report["status"] = "OCR_DISABLED_MANUAL_CALIBRATION_REQUIRED"
    combined = merge_native_and_ocr(native, ocr)
    report["effective_tokens"] = len(combined)
    _atomic_write_text(out_dir/"context"/"words_ocr.csv", ocr.to_csv(index=False))
    _atomic_write_text(out_dir/"context"/"words_effective.csv", combined.to_csv(index=False))
    write_json_atomic(out_dir/"context"/"ocr_report.json", report)
    return report


def build_axis_ocr_atlas(item):
    """One bounded contact sheet of axis-label strips. Never repair OCR digits by guessing."""
    image=load_working_image(item); gray=np.asarray(image.convert("L")); tiles=[]
    for panel in _read_panels(item_output_dir(item)):
        if not panel.get("plot_bbox_px"):continue
        left,top,right,bottom=panel["plot_bbox_px"];pw,ph=right-left,bottom-top
        for axis in ["y","x"]:
            if axis=="y":
                x0=max(0,int(left-max(42,.22*pw)));x1=max(x0+1,int(left)-3)
                y0=max(0,int(top-max(12,.06*ph)));y1=min(image.height,int(bottom+max(12,.06*ph)))
                patch=gray[y0:y1,x0:x1]
                if patch.size:
                    gutters=_runs((patch<190).mean(axis=0)<.012)
                    eligible=[(a,b) for a,b in gutters if b-a>=3 and b<patch.shape[1]-24 and b>4]
                    if eligible:
                        _,end=max(eligible,key=lambda ab:ab[1]);x0+=max(0,end-10)
            else:
                x0=max(0,int(left-12));x1=min(image.width,int(right+12))
                y0=min(image.height-1,int(bottom+3));y1=min(image.height,int(bottom+max(25,.18*ph)))
            if x1>x0 and y1>y0:
                tiles.append({"panel_id":panel["panel_id"],"axis":axis,"crop_bbox":[x0,y0,x1,y1],"scale":2.0})
    if not tiles:
        return image,[{"panel_id":"P01","axis":"whole","crop_bbox":[0,0,image.width,image.height],"scale":1.,"atlas_bbox":[0,0,image.width,image.height]}]
    gap=40;max_row_width=2400;x=y=row_h=0
    for tile in tiles:
        crop=image.crop(tuple(tile["crop_bbox"]));scale=tile["scale"]
        crop=crop.resize((max(1,round(crop.width*scale)),max(1,round(crop.height*scale))),Image.Resampling.LANCZOS)
        tile["image"]=crop
        if x and x+crop.width+gap>max_row_width:x=0;y+=row_h+2*gap;row_h=0
        tile["atlas_bbox"]=[x+gap,y+gap,x+gap+crop.width,y+gap+crop.height]
        x+=crop.width+2*gap;row_h=max(row_h,crop.height)
    width=max(t["atlas_bbox"][2] for t in tiles)+gap;height=max(t["atlas_bbox"][3] for t in tiles)+gap
    ratio=min(1.0,math.sqrt(MAX_OCR_ATLAS_PIXELS/(width*height)))
    atlas=Image.new("RGB",(width,height),"white")
    for t in tiles:atlas.paste(t["image"],tuple(t["atlas_bbox"][:2]));del t["image"]
    if ratio<1:
        atlas=atlas.resize((round(width*ratio),round(height*ratio)),Image.Resampling.LANCZOS)
        for t in tiles:t["atlas_bbox"]=[v*ratio for v in t["atlas_bbox"]];t["scale"]*=ratio
    return atlas,tiles


def atlas_words_to_figure(words,tiles):
    rows=[]
    for r in words.to_dict("records"):
        xc=(r["x0"]+r["x1"])/2;yc=(r["y0"]+r["y1"])/2
        for t in tiles:
            a,b,c,d=t["atlas_bbox"]
            if a<=xc<=c and b<=yc<=d:
                row=dict(r)
                # Reject tokens cut off by tile edges; losing a minus sign is unsafe.
                if r["x0"]<=a+1 or r["x1"]>=c-1 or r["y0"]<=b+1 or r["y1"]>=d-1:break
                x0,y0,_,_=t["crop_bbox"];scale=t["scale"]
                row.update(x0=x0+(r["x0"]-a)/scale,x1=x0+(r["x1"]-a)/scale,
                           y0=y0+(r["y0"]-b)/scale,y1=y0+(r["y1"]-b)/scale)
                row.update(axis_hint=t["axis"],panel_hint=t["panel_id"],tick_position_px=t.get("tick_position_px"))
                rows.append(row);break
    return _plain_words(pd.DataFrame(rows))


def run_local_ocr(item):
    # Cache survives a manual calibration change. A process failure is cached as a
    # diagnostic too; use clear_ocr_cache(item_id) to request one explicit retry.
    request_identity={"image":context_meta(item).get("working_image_sha256"),
        "backend":OCR_EFFECTIVE_BACKEND,"versions":OCR_DEPENDENCY_VERSIONS,
        "language":OCR_LANGUAGE,"confidence":OCR_MIN_CONFIDENCE,"timeout":OCR_TIMEOUT_SECONDS,
        "preprocessing":OCR_PREPROCESSING,
        "panels":[{"panel_id":p["panel_id"],"box":p.get("plot_bbox_px")} for p in _read_panels(item_output_dir(item))]}
    key=hashlib.sha256(json.dumps(request_identity,sort_keys=True).encode()).hexdigest()
    cp=OCR_CACHE_DIR/(key+".json")
    old=read_json_if_exists(cp)
    if isinstance(old,dict) and old.get("identity")==request_identity:
        report={**old["report"],"cache_hit":True,"passes_this_run":0}
        return _plain_words(pd.DataFrame(old["words"])),report
    words,report=_run_local_ocr_uncached(item)
    report.update(cache_hit=False,passes_this_run=report.get("passes",0),cache_key=key)
    write_json_atomic(cp,{"identity":request_identity,"report":report,"words":words.to_dict("records")})
    return words,report


def clear_ocr_cache(figure_item_id):
    item=_item_by_id(figure_item_id)
    if item is None:raise KeyError("Unknown figure")
    image_hash=context_meta(item).get("working_image_sha256")
    count=0
    for p in OCR_CACHE_DIR.glob("*.json"):
        obj=read_json_if_exists(p) or {}
        if obj.get("identity",{}).get("image")==image_hash:
            p.unlink();count+=1
    # Explicit opt-in retry, limited to this figure.
    if figure_item_id not in REPROCESS_ITEM_IDS:REPROCESS_ITEM_IDS.append(figure_item_id)
    print("Removed OCR cache records:",count,". Run reprocess_one(figure_item_id).")


In [ ]:
# ============================================================
# Multi-panel axes and calibrated coordinate transforms.
# Automatic calibration needs >= 3 actual numeric ticks.
# Two manually supplied points are allowed, but not called a fit.
# ============================================================
import itertools
import unicodedata


def parse_number(text):
    s = clean_str(text).replace("\u2212", "-").replace("\u2013", "-")
    s = s.replace("\u00a0", "").replace(" ", "")
    # Explicit superscript exponents are kept, not confused with ordinary digits.
    supers = str.maketrans("\u2070\u00b9\u00b2\u00b3\u2074\u2075\u2076\u2077\u2078\u2079\u207b\u207a", "0123456789-+")
    m = re.fullmatch(r"10([\u2070\u00b9\u00b2\u00b3\u2074-\u2079\u207b\u207a]+)", s)
    if m:
        try:
            value = 10.0 ** int(m.group(1).translate(supers))
            return value if math.isfinite(value) else None
        except (ValueError, OverflowError):
            return None
    m = re.fullmatch(r"([+\-]?(?:\d+(?:[.,]\d*)?|[.,]\d+))?[x\u00d7*]10\^?([+\-]?\d+)", s)
    if m:
        try:
            value = float((m.group(1) or "1").replace(",", ".")) * 10.0 ** int(m.group(2))
            return value if math.isfinite(value) else None
        except (ValueError, OverflowError):
            return None
    m = re.fullmatch(r"10\^([+\-]?\d+)", s)
    if m:
        try:
            value = 10.0 ** int(m.group(1))
            return value if math.isfinite(value) else None
        except (ValueError, OverflowError):
            return None
    if "," in s:
        # 1,234 can mean 1.234 or 1234. Never silently multiply it by 1000.
        if "." in s or s.count(",") != 1:
            return None
        left, right = s.split(",")
        if len(right) == 3 and left.lstrip("+-") not in {"", "0"} and DECIMAL_COMMA != "DECIMAL":
            return None
        s = s.replace(",", ".")
    if re.fullmatch(r"[+\-]?0[0-9]+", s):
        return None  # e.g. OCR 04 may have lost a decimal point.
    if not re.fullmatch(r"[+\-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+\-]?\d+)?", s):
        return None
    try:
        value = float(s)
        return value if math.isfinite(value) else None
    except ValueError:
        return None


def _rect_iou(a, b):
    inter = max(0., min(a[2],b[2])-max(a[0],b[0])) * max(0., min(a[3],b[3])-max(a[1],b[1]))
    aa = max(0.,a[2]-a[0])*max(0.,a[3]-a[1]); bb = max(0.,b[2]-b[0])*max(0.,b[3]-b[1])
    return inter / max(aa + bb - inter, 1e-12)


def _merge_axis_segments(segs, direction, tolerance):
    # Collapse the two edges of an axis stroke and small collinear fragments.
    records = []
    for x0, y0, x1, y1 in segs:
        if direction == "h":
            records.append([(y0+y1)/2, min(x0,x1), max(x0,x1)])
        else:
            records.append([(x0+x1)/2, min(y0,y1), max(y0,y1)])
    merged = []
    for level, start, end in sorted(records):
        match = None
        for j, (lv, a, b) in enumerate(merged):
            if abs(lv-level) <= tolerance and start <= b+tolerance*3 and end >= a-tolerance*3:
                match = j; break
        if match is None:
            merged.append([level, start, end])
        else:
            old = merged[match]
            merged[match] = [(old[0]+level)/2, min(old[1],start), max(old[2],end)]
    return merged


def detect_plot_boxes(image):
    rgb = np.asarray(image.convert("RGB"))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    edges = cv2.Canny(gray, 60, 160)
    min_len = max(24, round(min(w,h)*0.11))
    lines = normalize_hough_segments(cv2.HoughLinesP(edges,1,np.pi/180,threshold=max(18,min_len//3),
                        minLineLength=min_len,maxLineGap=max(3,round(min(w,h)*0.008))))
    hs, vs = [], []
    tol = max(2., min(w,h)*0.0025)
    for x0,y0,x1,y1 in lines:
        if abs(y1-y0) <= tol and abs(x1-x0) >= max(30,w*.10): hs.append((x0,y0,x1,y1))
        if abs(x1-x0) <= tol and abs(y1-y0) >= max(30,h*.10): vs.append((x0,y0,x1,y1))
    # Morphological line masks recover strokes which Canny splits at filled bars.
    ink=(gray < 205).astype(np.uint8)*255
    horizontal=cv2.morphologyEx(ink,cv2.MORPH_OPEN,np.ones((1,max(35,int(w*.10))),np.uint8))
    vertical=cv2.morphologyEx(ink,cv2.MORPH_OPEN,np.ones((max(35,int(h*.10)),1),np.uint8))
    for mask,direction in [(horizontal,"h"),(vertical,"v")]:
        n,labels,stats,centroids=cv2.connectedComponentsWithStats(mask,8)
        for x,y,ww,hh,area in stats[1:]:
            if direction=="h" and ww>max(35,w*.10) and hh<max(12,h*.035):
                hs.append((x,y+hh/2,x+ww,y+hh/2))
            elif direction=="v" and hh>max(35,h*.10) and ww<max(12,w*.025):
                vs.append((x+ww/2,y,x+ww/2,y+hh))
    hs = _merge_axis_segments(hs,"h",tol*1.5)
    vs = _merge_axis_segments(vs,"v",tol*1.5)
    boxes = []
    for vx, vy0, vy1 in vs:
        for hy, hx0, hx1 in hs:
            # A left y-axis and bottom x-axis must meet at their ends.
            if not hx0-tol*4 <= vx <= hx0+max(tol*5,.12*(hx1-hx0)) or abs(vy1-hy) > tol*5:
                continue
            bw,bh = hx1-vx,hy-vy0
            if bw < max(40,w*.10) or bh < max(35,h*.10):
                continue
            if not .12 <= bw/max(bh,1) <= 12:
                continue
            boxes.append([float(vx),float(vy0),float(hx1),float(hy)])
    kept = []
    for box in sorted(boxes,key=lambda a: -(a[2]-a[0])*(a[3]-a[1])):
        if any(_rect_iou(box,k)>.55 or
               (abs(box[0]-k[0])<max(tol*5,.10*(k[2]-k[0])) and abs(box[3]-k[3])<tol*5) for k in kept):
            continue
        kept.append(box)
    if len(kept)>MAX_PANELS:
        raise ValueError(f"Too many axis regions ({len(kept)}); panel split needs review, not silent truncation")
    return sorted(kept,key=lambda b:(round(b[1]/max(25,h*.08)),b[0]))


def enhanced_segment_panels(item, image):
    h,w=image.height,image.width
    manual=load_calibration(item)
    overrides=manual.get("panels",[])
    if overrides:
        panels=[]
        for i,p in enumerate(overrides,1):
            box=p.get("plot_bbox_px")
            if not isinstance(box,list) or len(box)!=4:
                raise ValueError("Each manual panel needs plot_bbox_px=[left,top,right,bottom]")
            left,top,right,bottom=map(float,box)
            if not (0<=left<right<=w and 0<=top<bottom<=h):
                raise ValueError("Manual plot rectangle lies outside the current working image")
            panels.append({"panel_id":p.get("panel_id",f"P{i:02d}"),"x0":max(0,int(left-.30*(right-left))),
                "y0":max(0,int(top-.12*(bottom-top))),"x1":min(w,int(right+.12*(right-left))),
                "y1":min(h,int(bottom+.30*(bottom-top))),"plot_bbox_px":box,"split_method":"MANUAL",
                "split_confidence":None,"kind":p.get("kind","auto")})
        return panels
    boxes=detect_plot_boxes(image)
    if not boxes:
        return [{"panel_id":"P01","x0":0,"y0":0,"x1":w,"y1":h,"plot_bbox_px":None,
                 "split_method":"NO_AXES_WHOLE_FIGURE","split_confidence":0.,"kind":"auto"}]
    panels=[]
    for i,(left,top,right,bottom) in enumerate(boxes,1):
        pw,ph=right-left,bottom-top
        panels.append({"panel_id":f"P{i:02d}","x0":max(0,int(left-.32*pw)),"y0":max(0,int(top-.12*ph)),
            "x1":min(w,int(right+.10*pw)),"y1":min(h,int(bottom+.30*ph)),
            "plot_bbox_px":[left,top,right,bottom],"split_method":"AXIS_PAIR","split_confidence":None,"kind":"auto"})
    return panels


def _fit_scale(ticks, scale, manual=False):
    if len(ticks)<(2 if manual else MIN_AUTO_TICKS):
        return {"ok":False,"reason":"INSUFFICIENT_NUMERIC_TICKS","tick_count":len(ticks),"ticks":ticks}
    positions=np.array([float(t["pixel"]) for t in ticks]); values=np.array([float(t["value"]) for t in ticks])
    if not np.isfinite(positions).all() or not np.isfinite(values).all() or np.ptp(positions)<3 or np.ptp(values)==0:
        return {"ok":False,"reason":"DEGENERATE_CALIBRATION","ticks":ticks}
    if len(set(positions))<len(positions) or len(set(values))<len(values):
        return {"ok":False,"reason":"CONFLICTING_OR_REPEATED_TICKS","ticks":ticks}
    if scale=="log10" and np.any(values<=0):
        return {"ok":False,"reason":"NONPOSITIVE_LOG_TICK","ticks":ticks}
    transformed=np.log10(values) if scale=="log10" else values
    order=np.argsort(positions); diff=np.diff(transformed[order])
    if not (np.all(diff>0) or np.all(diff<0)):
        return {"ok":False,"reason":"NONMONOTONIC_TICKS","ticks":ticks}
    coef=np.polyfit(positions,transformed,1)
    predicted=np.polyval(coef,positions)
    ss=float(np.sum((transformed-transformed.mean())**2))
    r2=1-float(np.sum((transformed-predicted)**2))/ss if ss else 0.
    pixel_residual=float(np.max(np.abs((transformed-predicted)/coef[0])))
    tolerance=max(2.5,.009*float(np.ptp(positions)))
    ok=(manual and len(ticks)==2) or (r2>=MIN_AXIS_CALIBRATION_R2 and pixel_residual<=tolerance)
    return {"ok":bool(ok),"reason":"OK" if ok else "NONLINEAR_OR_INCONSISTENT_TICKS",
            "scale":scale,"coef":[float(x) for x in coef],"r2":None if manual and len(ticks)==2 else r2,
            "max_residual_px":pixel_residual,"tick_count":len(ticks),"ticks":ticks,
            "pixel_min":float(positions.min()),"pixel_max":float(positions.max()),
            "source":"MANUAL_CALIBRATION" if manual else "AUTO_TICK_FIT","multiplier":1.0}


def manual_axis_calibration(spec):
    if not isinstance(spec,dict):
        return {"ok":False,"reason":"NO_MANUAL_AXIS"}
    scale=spec.get("scale","linear")
    if scale=="piecewise":
        segments=[manual_axis_calibration(s) for s in spec.get("segments",[])]
        if not segments or any(not s.get("ok") for s in segments):
            return {"ok":False,"reason":"INVALID_PIECEWISE_SEGMENTS"}
        intervals=sorted((s["pixel_min"],s["pixel_max"]) for s in segments)
        if any(intervals[i][1]>intervals[i+1][0] for i in range(len(intervals)-1)):
            return {"ok":False,"reason":"OVERLAPPING_PIECEWISE_SEGMENTS"}
        return {"ok":True,"scale":"piecewise","segments":segments,"source":"MANUAL_CALIBRATION","r2":None,
                "reason":"OK","multiplier":float(spec.get("multiplier",1.0))}
    if scale not in {"linear","log10"}:
        return {"ok":False,"reason":"UNSUPPORTED_SCALE"}
    try:
        ticks=[{"pixel":float(t["pixel"]),"value":float(t["value"]),"text":str(t["value"])} for t in spec.get("ticks",[])]
        result=_fit_scale(ticks,scale,manual=True)
        mult=float(spec.get("multiplier",1.0))
        if not math.isfinite(mult) or mult==0: raise ValueError("invalid multiplier")
        result["multiplier"]=mult
        return result
    except (TypeError,ValueError,KeyError) as exc:
        return {"ok":False,"reason":"INVALID_MANUAL_CALIBRATION","error":repr(exc)}


def calibrated_value(cal, pixel, extrapolate=False):
    if not cal.get("ok") or pixel is None or not math.isfinite(float(pixel)):
        return None
    pixel=float(pixel)
    if cal["scale"]=="piecewise":
        for segment in cal["segments"]:
            if segment["pixel_min"]<=pixel<=segment["pixel_max"]:
                value=calibrated_value(segment,pixel)
                return value*cal.get("multiplier",1.0) if value is not None else None
        return None  # An axis break is a gap, not a missing value to interpolate.
    margin=.20*(cal["pixel_max"]-cal["pixel_min"])
    if not extrapolate and not cal["pixel_min"]-margin<=pixel<=cal["pixel_max"]+margin:
        return None
    z=float(np.polyval(cal["coef"],pixel))
    try:
        value=(10.0**z if cal["scale"]=="log10" else z)*cal.get("multiplier",1.0)
        return value if math.isfinite(value) else None
    except OverflowError:
        return None


def calibrated_pixel(cal,value):
    if not cal.get("ok") or cal["scale"]=="piecewise": return None
    value=float(value)/cal.get("multiplier",1.0)
    if cal["scale"]=="log10":
        if value<=0:return None
        value=math.log10(value)
    a,b=cal["coef"]
    return (value-b)/a if a else None


def axis_calibration(words,panel,orientation,spec=None,side="left"):
    if spec:
        return manual_axis_calibration(spec)
    box=panel.get("plot_bbox_px")
    if not box:
        return {"ok":False,"reason":"AXES_NOT_DETECTED","tick_count":0}
    left,top,right,bottom=box; pw,ph=right-left,bottom-top
    candidates=[]
    for r in _plain_words(words).to_dict("records"):
        value=parse_number(r.get("text"))
        if value is None:continue
        xc=(r["x0"]+r["x1"])/2; yc=(r["y0"]+r["y1"])/2
        if orientation=="y":
            if side=="right":
                use=right-3<=r["x0"]<=right+max(22,.20*pw)
                align=r["x0"]
            else:
                use=left-max(35,.22*pw)<=r["x1"]<=left+max(4,.012*pw)
                align=r["x1"]
            use=use and top-.04*ph<=yc<=bottom+.04*ph
            pixel=yc
        else:
            use=left-.04*pw<=xc<=right+.04*pw and bottom-3<=r["y0"]<=bottom+max(20,.17*ph)
            align=yc; pixel=xc
        if use:
            anchor=r.get("tick_position_px")
            if r.get("axis_hint")==orientation and r.get("panel_hint")==panel["panel_id"] and anchor is not None and pd.notna(anchor):
                pixel=float(anchor)
            candidates.append({"pixel":float(pixel),"value":float(value),"text":r["text"],"alignment":align,"origin":r["origin"],"confidence":r["confidence"]})
    if not candidates:return {"ok":False,"reason":"NO_NUMERIC_TICKS","tick_count":0}
    tolerance=max(6,min(pw,ph)*.028)
    groups=[]
    for t in sorted(candidates,key=lambda r:r["alignment"]):
        target=next((g for g in groups if abs(np.median([v["alignment"] for v in g])-t["alignment"])<=tolerance),None)
        if target is None:groups.append([t])
        else:target.append(t)
    proposals=[]
    for group in groups:
        ticks=[]
        for t in sorted(group,key=lambda r:(r["pixel"],-r["confidence"])):
            near=next((q for q in ticks if abs(q["pixel"]-t["pixel"])<=max(2,min(pw,ph)*.005)),None)
            if near is not None:
                if near["value"]!=t["value"]:
                    # Conflicting OCR values at one pixel cannot be used for calibration.
                    near["conflict"]=True
                continue
            ticks.append(t)
        ticks=[t for t in ticks if not t.get("conflict")]
        linear=_fit_scale(ticks,"linear")
        log=_fit_scale(ticks,"log10")
        if linear.get("ok"):
            proposals.append(linear)
        elif log.get("ok") and max(t["value"] for t in ticks)/min(t["value"] for t in ticks)>=3:
            proposals.append(log)
    if not proposals:
        return {"ok":False,"reason":"INSUFFICIENT_OR_INCONSISTENT_TICKS","tick_count":len(candidates),"ticks":candidates}
    proposals.sort(key=lambda c:(-c["tick_count"],c["max_residual_px"]))
    cal=proposals[0]; cal["orientation"]=orientation;cal["axis_side"]=side
    cal["text_sources"]=sorted(set(t["origin"] for t in cal["ticks"]))
    # Explicit axis multipliers only. Fragmented multipliers are never guessed.
    region=[]
    for r in _plain_words(words).to_dict("records"):
        xc=(r["x0"]+r["x1"])/2;yc=(r["y0"]+r["y1"])/2
        near=(left-.25*pw<=xc<=left+.35*pw and top-.20*ph<=yc<=top+.06*ph) if orientation=="y" else (right-.20*pw<=xc<=right+.15*pw and bottom<=yc<=bottom+.22*ph)
        if near and re.search(r"[x\u00d7*]\s*10",r["text"]):region.append(r["text"])
    if region:
        factors=[parse_number(r) for r in region]
        if any(f is None for f in factors) or len(set(factors))!=1:
            return {"ok":False,"reason":"UNRESOLVED_AXIS_MULTIPLIER","texts":region}
        cal["multiplier"]=factors[0]
    return cal


In [ ]:
# ============================================================
# Candidate extraction: vector/raster bars, signed/horizontal bars,
# geometric error-bar endpoints, vector/color curves, scatter marks,
# and explicit human calibration / external digitized values.
# Nothing in this cell marks a measurement VERIFIED.
# ============================================================


def _panel_override(item,panel_id):
    return next((p for p in load_calibration(item).get("panels",[]) if p.get("panel_id")==panel_id),{})


def _inside_plot(x,y,box,margin=0):
    return box[0]-margin<=x<=box[2]+margin and box[1]-margin<=y<=box[3]+margin


def _make_candidate(item,panel,method,kind,x,y,value,xvalue=None,series="",point_role="PLOTTED_VALUE_UNSPECIFIED",xcal=None,ycal=None):
    row=_candidate_base(item,panel["panel_id"],method,kind)
    meta=context_meta(item);mapping=meta.get("pdf_mapping")
    row.update({"x_position_px":float(x) if x is not None else None,"y_position_px":float(y) if y is not None else None,
        "value_candidate":float(value),"x_value_candidate":xvalue,"series_candidate":series,"point_role":point_role,
        "value_axis":"y","axis_scale_x":(xcal or {}).get("scale",""),"axis_scale_y":(ycal or {}).get("scale",""),
        "axis_calibration_r2_x":(xcal or {}).get("r2"),"axis_calibration_r2_y":(ycal or {}).get("r2"),
        "calibration_source_x":(xcal or {}).get("source",""),"calibration_source_y":(ycal or {}).get("source",""),
        "working_image_sha256":meta.get("working_image_sha256",""),"source_pdf_sha256":meta.get("source_pdf_sha256",""),
        "coordinate_system":"WORKING_IMAGE_PIXELS_TOPLEFT","confidence_geometry":None,
        "error_lower_candidate":None,"error_upper_candidate":None,"error_low_endpoint_candidate":None,"error_high_endpoint_candidate":None,
        "digitization_precision_candidate":None,"evidence_path":str(item_output_dir(item)/"digitization"/"panel_reports.json"),
        "possible_duplicate":False,"eligible_for_database":False,"verification_status":"UNVERIFIED"})
    if mapping and x is not None and y is not None:
        row["x_position_pdf"]=mapping["origin_x"]+x/mapping["scale"]
        row["y_position_pdf"]=mapping["origin_y"]+y/mapping["scale"]
    cal=xcal if kind=="horizontal_bar" else ycal
    pixel=x if kind=="horizontal_bar" else y
    if cal and cal.get("ok") and pixel is not None:
        ends=[calibrated_value(cal,pixel+d) for d in [-2.,2.]]
        if all(v is not None for v in ends):
            row["digitization_precision_candidate"]=max(abs(v-value) for v in ends)
    return row


def _bar_baseline(cal,box,orientation):
    zero=calibrated_pixel(cal,0)
    low,high=(box[1],box[3]) if orientation=="vertical" else (box[0],box[2])
    if zero is not None and low-8<=zero<=high+8:return float(zero)
    return float(box[3] if orientation=="vertical" else box[0])


def _bar_plausible(rect,baseline,plot,orientation):
    x0,y0,x1,y1=rect;pw,ph=plot[2]-plot[0],plot[3]-plot[1]
    if not _inside_plot(x0,y0,plot,4) or not _inside_plot(x1,y1,plot,4):return False
    if orientation=="vertical":
        width,length=x1-x0,y1-y0
        return max(4,.007*pw)<=width<=.34*pw and length>=3 and min(abs(y0-baseline),abs(y1-baseline))<=max(6,.025*ph)
    width,length=y1-y0,x1-x0
    return max(4,.007*ph)<=width<=.34*ph and length>=3 and min(abs(x0-baseline),abs(x1-baseline))<=max(6,.025*pw)


def _component_rects(image,plot,orientation):
    l,t,r,b=[int(round(v)) for v in plot]
    l=max(0,l);t=max(0,t);r=min(image.width,r);b=min(image.height,b)
    arr=np.asarray(image.convert("RGB"))[t:b,l:r]
    if arr.size==0:return []
    hsv=cv2.cvtColor(arr,cv2.COLOR_RGB2HSV);gray=cv2.cvtColor(arr,cv2.COLOR_RGB2GRAY)
    masks=[("COLOUR",((hsv[:,:,1]>38)&(hsv[:,:,2]>30)).astype(np.uint8)*255)]
    for lo,hi in [(0,95),(95,180),(180,242)]:
        masks.append((f"GRAY_{lo}_{hi}",((gray>=lo)&(gray<hi)&(hsv[:,:,1]<=38)).astype(np.uint8)*255))
    result=[]
    for name,mask in masks:
        mask=cv2.morphologyEx(mask,cv2.MORPH_OPEN,np.ones((3,3),np.uint8))
        # Remove axis/grid connections without erasing the filled bar interiors.
        if orientation=="vertical":
            grid=cv2.morphologyEx(mask,cv2.MORPH_OPEN,np.ones((1,max(20,int(mask.shape[1]*.55))),np.uint8))
        else:
            grid=cv2.morphologyEx(mask,cv2.MORPH_OPEN,np.ones((max(20,int(mask.shape[0]*.55)),1),np.uint8))
        mask=cv2.subtract(mask,grid)
        count,labels,stats,_=cv2.connectedComponentsWithStats(mask,8)
        for label,(x,y,w,h,area) in enumerate(stats[1:],1):
            if w<4 or h<4 or area<20 or area/(w*h)<.45:continue
            local=(labels[y:y+h,x:x+w]==label)
            # A filled bar has a wide supported cross-section. Error caps can
            # span > half of the central columns, so their median top is unsafe.
            # Require 80% of the full component width/height for the bar body.
            if orientation=="vertical":
                spans=_runs(local.sum(axis=1)>=max(3,.80*w))
                if not spans:continue
                start,end=max(spans,key=lambda ab:ab[1]-ab[0])
                if end-start<3:continue
                y0=y+float(start);y1=y+float(end)
                box=[l+x,t+y0,l+x+w,t+y1]
            else:
                spans=_runs(local.sum(axis=0)>=max(3,.80*h))
                if not spans:continue
                start,end=max(spans,key=lambda ab:ab[1]-ab[0])
                if end-start<3:continue
                x0=x+float(start);x1=x+float(end)
                box=[l+x0,t+y,l+x1,t+y+h]
            pixels=arr[y:y+h,x:x+w][local]
            rgb=np.median(pixels,axis=0).astype(int).tolist() if len(pixels) else []
            result.append({"rect":box,"method":"RASTER_FILLED_BAR","rgb":rgb,"component_area":int(area)})
    edges=cv2.Canny(gray,55,155)
    contours,_=cv2.findContours(edges,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)
    for c in contours:
        x,y,w,h=cv2.boundingRect(c)
        if w<5 or h<5 or cv2.contourArea(c)/max(1,w*h)<.65:continue
        peri=cv2.arcLength(c,True);approx=cv2.approxPolyDP(c,.018*peri,True)
        if len(approx)>7:continue
        result.append({"rect":[l+x,t+y,l+x+w,t+y+h],"method":"RASTER_OUTLINE_BAR","rgb":[],"component_area":int(w*h)})
    return result


def detect_bars(image,panel,cal,orientation,drawings):
    plot=panel["plot_bbox_px"];baseline=_bar_baseline(cal,plot,orientation)
    found=[]
    if AUTO_VECTOR_BAR_DIGITIZATION:
        for drawing in drawings:
            for it in drawing.get("items",[]):
                if it[0]!="re":continue
                box=list(map(float,it[1]))
                if _bar_plausible(box,baseline,plot,orientation):
                    color=drawing.get("fill") or drawing.get("color") or []
                    found.append({"rect":box,"method":"VECTOR_RECT_BAR","rgb":[int(v*255) for v in color]})
    if AUTO_RASTER_BAR_DIGITIZATION:
        found.extend(b for b in _component_rects(image,plot,orientation) if _bar_plausible(b["rect"],baseline,plot,orientation))
    rank={"VECTOR_RECT_BAR":0,"RASTER_FILLED_BAR":1,"RASTER_OUTLINE_BAR":2}
    result=[]
    for b in sorted(found,key=lambda x:(rank[x["method"]],-(x["rect"][2]-x["rect"][0])*(x["rect"][3]-x["rect"][1]))):
        rect=b["rect"]
        if any(_rect_iou(rect,v["rect"])>.58 or max(abs(rect[j]-v["rect"][j]) for j in range(4))<5 for v in result):continue
        result.append(b)
    result.sort(key=lambda v:(v["rect"][0]+v["rect"][2]) if orientation=="vertical" else (v["rect"][1]+v["rect"][3]))
    if not MIN_BAR_COUNT<=len(result)<=MAX_BAR_COUNT:return [],{"reason":f"BAR_COUNT_{len(result)}","baseline_px":baseline}
    return result,{"reason":"CALIBRATED_BAR_GEOMETRY","baseline_px":baseline}


def _error_segments(image,panel):
    box=panel["plot_bbox_px"];l,t,r,b=[int(v) for v in box]
    gray=cv2.cvtColor(np.asarray(image.convert("RGB"))[max(t,0):b,max(l,0):r],cv2.COLOR_RGB2GRAY)
    if not gray.size:return []
    lines=normalize_hough_segments(cv2.HoughLinesP(cv2.Canny(gray,60,160),1,np.pi/180,threshold=8,minLineLength=5,maxLineGap=3))
    return [(float(x0+l),float(y0+t),float(x1+l),float(y1+t)) for x0,y0,x1,y1 in lines]


def attach_error_bounds(row,bar,panel,cal,orientation,segments):
    if not AUTO_ERRORBAR_ENDPOINTS:return
    x0,y0,x1,y1=bar["rect"]
    center=(x0+x1)/2 if orientation=="vertical" else (y0+y1)/2
    endpoint=row["y_position_px"] if orientation=="vertical" else row["x_position_px"]
    width=(x1-x0) if orientation=="vertical" else (y1-y0)
    extent=(panel["plot_bbox_px"][3]-panel["plot_bbox_px"][1]) if orientation=="vertical" else (panel["plot_bbox_px"][2]-panel["plot_bbox_px"][0])
    if orientation=="horizontal":segments=[(b,a,d,c) for a,b,c,d in segments]
    stems=[];caps=[]
    for a,b,c,d in segments:
        if abs(a-c)<=2.5 and abs((a+c)/2-center)<=max(3,width*.12):
            lo,hi=sorted([b,d])
            if lo-5<=endpoint<=hi+5 and hi-lo<.30*extent:stems.append((lo,hi))
        if abs(b-d)<=2.5 and 3<=abs(c-a)<=width*1.25 and abs((a+c)/2-center)<=max(3,width*.18):
            y=(b+d)/2
            if 2<abs(y-endpoint)<.30*extent:caps.append(y)
    endpoints=[]
    for y in caps:
        if any(lo-4<=y<=hi+4 for lo,hi in stems):
            v=calibrated_value(cal,y)
            if v is not None:endpoints.append(v)
    value=row["value_candidate"]
    lower=[v for v in endpoints if v<value];upper=[v for v in endpoints if v>value]
    # Keep the nearest supported cap on each side; no forced symmetry or SE/SD inference.
    if lower:
        row["error_low_endpoint_candidate"]=max(lower);row["error_lower_candidate"]=value-max(lower)
    if upper:
        row["error_high_endpoint_candidate"]=min(upper);row["error_upper_candidate"]=min(upper)-value
    if lower or upper:
        row["notes"]+=" Geometric error-bar endpoint(s); SD/SE/CI definition remains unspecified."


def bar_candidates(item,panel,image,drawings,cal,xcal,ycal,orientation):
    bars,report=detect_bars(image,panel,cal,orientation,drawings)
    if not bars:return [],report
    segments=_error_segments(image,panel);rows=[]
    baseline=report["baseline_px"]
    for bar in bars:
        l,t,r,b=bar["rect"]
        if orientation=="vertical":
            x=(l+r)/2;y=t if abs(t-baseline)>abs(b-baseline) else b;pixel=y;kind="bar"
        else:
            y=(t+b)/2;x=l if abs(l-baseline)>abs(r-baseline) else r;pixel=x;kind="horizontal_bar"
        value=calibrated_value(cal,pixel)
        if value is None:continue
        rgb=bar.get("rgb",[])
        series=("rgb("+",".join(map(str,rgb))+")") if rgb else "UNASSIGNED_SERIES"
        row=_make_candidate(item,panel,bar["method"],kind,x,y,value,series=series,point_role="BAR_ENDPOINT_UNSPECIFIED_STATISTIC",xcal=xcal,ycal=ycal)
        row["value_axis"]="x" if orientation=="horizontal" else "y"
        row["geometry_bbox_px"]=json.dumps(jsonable(bar["rect"]))
        row["notes"]="Bar endpoint, not an asserted mean. Colour is a series cue, not a treatment identity."
        attach_error_bounds(row,bar,panel,cal,orientation,segments)
        rows.append(row)
    return rows,{**report,"bar_count":len(rows),"orientation":orientation,"boxes":bars}


def _vector_curve_points(drawing):
    points=[]
    for it in drawing.get("items",[]):
        if it[0]=="l":points.extend([it[1],it[2]])
        elif it[0]=="c":
            p0,p1,p2,p3=[np.array(p,dtype=float) for p in it[1:5]]
            for t in np.linspace(0,1,17):points.append(((1-t)**3*p0+3*(1-t)**2*t*p1+3*(1-t)*t*t*p2+t**3*p3).tolist())
    return points


def vector_curve_candidates(item,panel,drawings,xcal,ycal):
    box=panel["plot_bbox_px"];pw,ph=box[2]-box[0],box[3]-box[1]
    rows=[];paths=[]
    for n,drawing in enumerate(drawings):
        if drawing.get("fill") is not None:continue
        points=_vector_curve_points(drawing)
        pts=np.array([p for p in points if _inside_plot(p[0],p[1],box,-2)],dtype=float)
        if pts.size==0 or len(pts)<3 or np.ptp(pts[:,0])<pw*.15:continue
        if np.ptp(pts[:,1])<2 and np.ptp(pts[:,0])>pw*.75:continue # grid/frame ambiguity
        # A single right-angle bracket is not a curve.
        if len(set(np.round(pts[:,0],1)))<3:continue
        key="vector_rgb="+str(drawing.get("color"))
        ded=[]
        for p in pts:
            if not ded or math.hypot(p[0]-ded[-1][0],p[1]-ded[-1][1])>1:ded.append(p)
        if len(ded)>MAX_CURVE_POINTS_PER_SERIES:
            indices=np.linspace(0,len(ded)-1,MAX_CURVE_POINTS_PER_SERIES).round().astype(int);ded=[ded[i] for i in indices]
        path_rows=0
        for x,y in ded:
            xx=calibrated_value(xcal,x);yy=calibrated_value(ycal,y)
            if xx is None or yy is None:continue
            row=_make_candidate(item,panel,"VECTOR_CURVE_TRACE","line",x,y,yy,xvalue=xx,series=key,
                                point_role="CURVE_GEOMETRY_NOT_OBSERVATION",xcal=xcal,ycal=ycal)
            row["notes"]="Sampled drawing path. Points may belong to a fitted curve; never use the point count as biological n."
            rows.append(row);path_rows+=1
        paths.append({"drawing_index":n,"series_cue":key,"exported_trace_points":path_rows})
    return rows,{"reason":"VECTOR_CURVE_TRACE" if rows else "NO_VECTOR_CURVES","paths":paths}


def _colour_masks(image,panel,words):
    l,t,r,b=[int(round(v)) for v in panel["plot_bbox_px"]]
    crop=np.asarray(image.convert("RGB"))[max(0,t):min(image.height,b),max(0,l):min(image.width,r)]
    if not crop.size:return []
    hsv=cv2.cvtColor(crop,cv2.COLOR_RGB2HSV);gray=cv2.cvtColor(crop,cv2.COLOR_RGB2GRAY)
    sat=(hsv[:,:,1]>55)&(hsv[:,:,2]>35)
    masks=[]
    for hue in range(0,180,15):
        mask=(sat&(hsv[:,:,0]>=hue)&(hsv[:,:,0]<hue+15)).astype(np.uint8)*255
        if np.count_nonzero(mask)<20:continue
        rgb=np.median(crop[mask>0],axis=0).astype(int).tolist()
        masks.append((f"rgb({','.join(map(str,rgb))})",mask))
    dark=((gray<110)&~sat).astype(np.uint8)*255
    masks.append(("GRAY_UNASSIGNED_SERIES",dark))
    out=[]
    for series,mask in masks:
        mask[:3,:]=0;mask[-3:,:]=0;mask[:,:3]=0;mask[:,-3:]=0
        # Remove detected text inside the plotting area, including p-value annotations.
        for w in words.to_dict("records"):
            x0,y0,x1,y1=w["x0"]-l,w["y0"]-t,w["x1"]-l,w["y1"]-t
            if 0<x0<x1<mask.shape[1] and 0<y0<y1<mask.shape[0]:
                mask[max(0,int(y0)-2):min(mask.shape[0],int(y1)+2),max(0,int(x0)-2):min(mask.shape[1],int(x1)+2)]=0
        if series.startswith("GRAY"):
            hg=cv2.morphologyEx(mask,cv2.MORPH_OPEN,np.ones((1,max(20,int(mask.shape[1]*.70))),np.uint8))
            vg=cv2.morphologyEx(mask,cv2.MORPH_OPEN,np.ones((max(20,int(mask.shape[0]*.70)),1),np.uint8))
            mask=cv2.subtract(mask,cv2.bitwise_or(hg,vg))
        out.append((series,mask,l,t))
    return out


def raster_curve_candidates(item,panel,image,words,xcal,ycal):
    rows=[];reports=[]
    for series,mask,ox,oy in _colour_masks(image,panel,words):
        h,w=mask.shape
        n,labels,stats,_=cv2.connectedComponentsWithStats(mask,8)
        component_ids=[i+1 for i,(x,y,ww,hh,area) in enumerate(stats[1:]) if ww>=w*.20 and area>=max(25,ww*.5)]
        for component in component_ids:
            component_mask=(labels==component)
            xvalues=np.flatnonzero(component_mask.any(axis=0))
            if len(xvalues)<.20*w:continue
            ambiguous=0;trace=[]
            for x in xvalues:
                ys=np.flatnonzero(component_mask[:,x]);runs=_runs(component_mask[:,x])
                if len(runs)>1 and max(ys)-min(ys)>max(6,h*.025):
                    ambiguous+=1;continue
                if len(ys)>max(14,h*.10):ambiguous+=1;continue
                trace.append((float(x+ox),float(np.median(ys)+oy)))
            if ambiguous>len(xvalues)*.15 or len(trace)<8:
                reports.append({"series":series,"reason":"MULTIPLE_BRANCHES_OR_ERRORBAR_AMBIGUITY"});continue
            if len(trace)>MAX_CURVE_POINTS_PER_SERIES:
                indices=np.linspace(0,len(trace)-1,MAX_CURVE_POINTS_PER_SERIES).round().astype(int);trace=[trace[i] for i in indices]
            for x,y in trace:
                xx=calibrated_value(xcal,x);yy=calibrated_value(ycal,y)
                if xx is None or yy is None:continue
                row=_make_candidate(item,panel,"RASTER_COLOUR_CURVE_TRACE","line",x,y,yy,xvalue=xx,series=series,
                    point_role="CURVE_GEOMETRY_NOT_OBSERVATION",xcal=xcal,ycal=ycal)
                row["notes"]="Raster trace coordinates, not independent observations. Overlapping/same-colour curves may need manual separation."
                rows.append(row)
            reports.append({"series":series,"reason":"TRACE_CANDIDATES","sample_count":len(trace)})
    return rows,{"reason":"RASTER_CURVE_TRACE" if rows else "NO_UNAMBIGUOUS_RASTER_TRACE","series_reports":reports}


def scatter_candidates(item,panel,image,words,xcal,ycal):
    rows=[]
    for series,mask,ox,oy in _colour_masks(image,panel,words):
        h,w=mask.shape
        contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            area=cv2.contourArea(contour);x,y,ww,hh=cv2.boundingRect(contour)
            if not (4<=area<=max(500,w*h*.003) and 2<=ww<=w*.065 and 2<=hh<=h*.065):continue
            if not .35<=ww/max(hh,1)<=2.85:continue
            perimeter=cv2.arcLength(contour,True)
            if not perimeter or 4*math.pi*area/(perimeter**2)<.22:continue
            moment=cv2.moments(contour)
            if not moment["m00"]:continue
            xxp=ox+moment["m10"]/moment["m00"];yyp=oy+moment["m01"]/moment["m00"]
            xx=calibrated_value(xcal,xxp);yy=calibrated_value(ycal,yyp)
            if xx is None or yy is None:continue
            row=_make_candidate(item,panel,"RASTER_MARKER_CENTROID","scatter",xxp,yyp,yy,xvalue=xx,series=series,
                                point_role="PLOTTED_MARKER_NOT_ASSERTED_REPLICATE",xcal=xcal,ycal=ycal)
            row["notes"]="Marker centroid. An individual dot is not automatically an independent biological replicate."
            rows.append(row)
    return rows,{"reason":"SCATTER_MARKERS" if rows else "NO_SEPARABLE_SCATTER_MARKERS","marker_count":len(rows)}


def manual_point_candidates(item,panel,spec,xcal,ycal):
    rows=[]
    horizontal=spec.get("kind")=="horizontal_bar"
    for p in spec.get("points",[]):
        x,y=float(p["x_px"]),float(p["y_px"])
        if not _inside_plot(x,y,panel["plot_bbox_px"],1):raise ValueError("Manual point outside its declared plot rectangle")
        cal=xcal if horizontal else ycal
        value=calibrated_value(cal,x if horizontal else y)
        if value is None:raise ValueError("Manual point cannot be calibrated (including a gap in a broken axis)")
        xx=calibrated_value(xcal,x) if not horizontal and xcal.get("ok") else None
        row=_make_candidate(item,panel,"MANUAL_CALIBRATED_POINT",spec.get("kind","manual"),x,y,value,xvalue=xx,
            series=clean_str(p.get("series")),point_role=clean_str(p.get("statistic")) or "HUMAN_SELECTED_POINT_UNSPECIFIED",
            xcal=xcal,ycal=ycal)
        row["value_axis"]="x" if horizontal else "y"
        row["unit_raw_candidate"]=clean_str(spec.get("unit"));row["outcome_raw_candidate"]=clean_str(spec.get("outcome"))
        row["category_raw_candidate"]=clean_str(p.get("label"));row["notes"]="Human-picked geometry; still unverified until row-level review."
        for endpoint_name in ["low","high"]:
            point_key=endpoint_name+("_x_px" if horizontal else "_y_px")
            raw=p.get(point_key)
            if raw is None or raw=="":continue
            endpoint=calibrated_value(cal,float(raw))
            if endpoint is None:raise ValueError("Error-bar endpoint is outside calibrated range")
            if endpoint<value:
                row["error_low_endpoint_candidate"]=endpoint;row["error_lower_candidate"]=value-endpoint
            elif endpoint>value:
                row["error_high_endpoint_candidate"]=endpoint;row["error_upper_candidate"]=endpoint-value
        row["uncertainty_type_candidate"]=clean_str(p.get("uncertainty_type"))
        rows.append(row)
    return rows


def _dedupe_same_geometry(rows):
    # Same geometry only. Equal numbers alone NEVER imply a duplicate.
    result=[];keys=set()
    for row in rows:
        key=(row["panel_id"],row["digitization_method"],row.get("series_candidate"),row.get("point_role"),row.get("external_file_sha256"),
             round(row["x_position_px"],2) if row.get("x_position_px") is not None else row.get("external_row"),
             round(row["y_position_px"],2) if row.get("y_position_px") is not None else row.get("value_candidate"))
        if key not in keys:result.append(row);keys.add(key)
    return result


def enhanced_digitize_item(item,panels,analysis):
    image=load_working_image(item);words=effective_words(item);drawings=effective_drawings(item)
    manual=load_calibration(item);rows=[];reports=[]
    caption=item.get("caption","")
    sensitive=bool(ORDINATION_HINT_RE.search(caption) or HEATMAP_HINT_RE.search(caption) or BOX_HINT_RE.search(caption))
    for panel in panels:
        cfg=next((p for p in manual.get("panels",[]) if p.get("panel_id")==panel["panel_id"]),{})
        kind=cfg.get("kind","auto")
        rep={"panel_id":panel["panel_id"],"kind_requested":kind,"plot_bbox_px":panel.get("plot_bbox_px"),"candidate_count":0,"attempts":[]}
        if kind=="skip":
            rep.update(status="HUMAN_EXCLUDED",reason=cfg.get("notes","Explicitly excluded panel"));reports.append(rep);continue
        if not panel.get("plot_bbox_px"):
            rep.update(status="NEEDS_MANUAL",reason="NO_AXES_MANUAL_PANEL_REQUIRED");reports.append(rep);continue
        ycal=axis_calibration(words,panel,"y",cfg.get("y_axis"),side=cfg.get("y_axis_side","left"))
        xcal=axis_calibration(words,panel,"x",cfg.get("x_axis"))
        rep.update(x_calibration=xcal,y_calibration=ycal)
        rightcal=axis_calibration(words,panel,"y",side="right")
        if kind=="auto" and ycal.get("ok") and rightcal.get("ok"):
            leftvalues=[calibrated_value(ycal,panel["plot_bbox_px"][i]) for i in [1,3]]
            rightvalues=[calibrated_value(rightcal,panel["plot_bbox_px"][i]) for i in [1,3]]
            if any(v is None for v in leftvalues+rightvalues) or not np.allclose(leftvalues,rightvalues,rtol=.03,atol=.01):
                rep.update(status="NEEDS_MANUAL",reason="POSSIBLE_DUAL_Y_AXIS_SELECT_AXIS_AND_SERIES");reports.append(rep);continue
        got=[]
        if cfg.get("points"):
            got=manual_point_candidates(item,panel,cfg,xcal,ycal)
            rep["attempts"].append({"reason":"MANUAL_POINTS_REPLACE_AUTOMATIC_PANEL"})
        elif sensitive and kind=="auto":
            rep.update(status="NEEDS_MANUAL",reason="BOX_HEATMAP_OR_ORDINATION_NOT_ASSUMED_TO_BE_BAR_MEANS");reports.append(rep);continue
        elif kind in {"boxplot","heatmap","manual"}:
            rep.update(status="NEEDS_MANUAL",reason="PICK_POINTS_OR_IMPORT_EXPLICIT_DIGITIZED_CSV");reports.append(rep);continue
        else:
            if kind in {"auto","bar"} and ycal.get("ok"):
                got,attempt=bar_candidates(item,panel,image,drawings,ycal,xcal,ycal,"vertical");rep["attempts"].append(attempt)
            if not got and kind in {"auto","horizontal_bar"} and xcal.get("ok"):
                got,attempt=bar_candidates(item,panel,image,drawings,xcal,xcal,ycal,"horizontal");rep["attempts"].append(attempt)
            if not got and kind in {"auto","line"} and xcal.get("ok") and ycal.get("ok"):
                if AUTO_VECTOR_LINE_DIGITIZATION:
                    got,attempt=vector_curve_candidates(item,panel,drawings,xcal,ycal);rep["attempts"].append(attempt)
                if not got and AUTO_RASTER_LINE_DIGITIZATION:
                    got,attempt=raster_curve_candidates(item,panel,image,words,xcal,ycal);rep["attempts"].append(attempt)
            if not got and kind in {"auto","scatter"} and xcal.get("ok") and ycal.get("ok") and AUTO_SCATTER_DIGITIZATION:
                got,attempt=scatter_candidates(item,panel,image,words,xcal,ycal);rep["attempts"].append(attempt)
        if got:
            for row in got:
                if cfg.get("unit"):row["unit_raw_candidate"]=cfg["unit"]
                if cfg.get("outcome"):row["outcome_raw_candidate"]=cfg["outcome"]
            rows.extend(got);rep.update(status="CANDIDATES_REVIEW_REQUIRED",reason="CANDIDATES_NOT_A_COMPLETENESS_GUARANTEE",candidate_count=len(got))
        else:
            rep.update(status="NEEDS_MANUAL",reason="NO_CALIBRATED_GEOMETRY" if ycal.get("ok") or xcal.get("ok") else "AXIS_CALIBRATION_REQUIRED")
        reports.append(rep)
    for p in manual.get("imported_rows",[]):
        value=float(p["value_candidate"])
        if not math.isfinite(value):raise ValueError("Non-finite value in explicitly imported CSV")
        panel={"panel_id":p.get("panel_id","EXTERNAL")}
        method=clean_str(p.get("digitization_method")) or "EXPLICIT_EXTERNAL_CSV"
        row=_make_candidate(item,panel,method,"external",None,None,value,
            xvalue=p.get("x_value_candidate"),series=p.get("series_candidate",""),point_role=p.get("statistic","UNSPECIFIED"))
        row.update(unit_raw_candidate=p.get("unit",""),notes="Explicitly mapped external digitization; not verified.",
                   external_file_sha256=p.get("file_sha256",""),external_row=p.get("external_row"),evidence_path=p.get("source_file",""),
                   external_tool=p.get("external_tool",""),external_tool_version=p.get("external_tool_version",""),
                   external_project_sha256=p.get("external_project_sha256",""),external_operator=p.get("external_operator",""),
                   external_axis_type=p.get("external_axis_type",""),external_calibration_notes=p.get("external_calibration_notes",""))
        rows.append(row)
    rows=_dedupe_same_geometry(rows)
    for index,row in enumerate(rows,1):
        row["candidate_index"]=index
        payload={k:row.get(k) for k in ["figure_item_id","panel_id","digitization_method","x_position_px","y_position_px","value_candidate","series_candidate","external_row","external_file_sha256","external_project_sha256","external_tool","external_tool_version","point_role","working_image_sha256"]}
        row["candidate_id"]=hashlib.sha256(json.dumps(payload,sort_keys=True,default=str).encode()).hexdigest()[:24]
    frame=pd.DataFrame(rows).reindex(columns=CANDIDATE_COLUMNS) if rows else pd.DataFrame(columns=CANDIDATE_COLUMNS)
    summary={"candidate_count":len(frame),"methods":sorted(set(frame.get("digitization_method",pd.Series(dtype=str)).astype(str))),
             "panel_count":len(panels),"panels_with_candidates":sum(r["candidate_count"]>0 for r in reports),
             "panels_needing_manual":sum(r.get("status")=="NEEDS_MANUAL" for r in reports),
             "reports":reports,"blocking_reasons":[r["panel_id"]+":"+r["reason"] for r in reports if r.get("status")=="NEEDS_MANUAL"],
             "coverage_note":"Counts refer to detected panels and candidate coordinates, not a validated fraction of all scientific measurements."}
    return frame,summary


def save_candidate_overlay(item,panels,candidates):
    image=load_working_image(item);draw=ImageDraw.Draw(image)
    for panel in panels:
        if panel.get("plot_bbox_px"):
            draw.rectangle(panel["plot_bbox_px"],outline=(30,90,210),width=2)
            draw.text((panel["plot_bbox_px"][0]+4,panel["plot_bbox_px"][1]+3),panel["panel_id"],fill=(30,90,210))
    for row in candidates.to_dict("records"):
        x,y=row.get("x_position_px"),row.get("y_position_px")
        try:x,y=float(x),float(y)
        except (ValueError,TypeError):continue
        if not math.isfinite(x) or not math.isfinite(y):continue
        draw.ellipse((x-4,y-4,x+4,y+4),outline=(230,40,30),width=2)
    _save_png_atomic(image,item_output_dir(item)/"qa"/"candidate_overlay.png")


In [ ]:
# ============================================================
# 12. Duplicate screening against text / native table values
# ============================================================

_STUDY_NUMERIC_CACHE = {}

def numeric_values_from_text(text: str) -> List[float]:
    vals=[]
    for token in re.findall(r"(?<![A-Za-z0-9])[+\-−]?\d+(?:[.,]\d+)?(?:[eE][+\-]?\d+)?", text or ""):
        v=parse_number(token)
        if v is not None and math.isfinite(v): vals.append(v)
    return vals


def build_study_numeric_index(study_dir: Path) -> Dict[str,Any]:
    key=str(study_dir)
    if key in _STUDY_NUMERIC_CACHE: return _STUDY_NUMERIC_CACHE[key]
    records=[]; scanned_cells=0
    # Explicit prose values. This is only a duplicate warning index, not a semantic source of truth.
    for txt in study_dir.rglob("document.txt"):
        if "_history" in txt.parts: continue
        try:
            content=txt.read_text(encoding="utf-8",errors="replace")
            for v in numeric_values_from_text(content)[:20_000]:
                records.append((v,f"TEXT:{txt.relative_to(study_dir).as_posix()}"))
                if len(records)>=MAX_DUPCHECK_VALUES_PER_STUDY: break
        except Exception: pass
        if len(records)>=MAX_DUPCHECK_VALUES_PER_STUDY: break
    # Native extracted tables; skip very large high-dimensional tables for this heuristic.
    if len(records)<MAX_DUPCHECK_VALUES_PER_STUDY:
        for table in study_dir.rglob("table_*.csv"):
            if "_history" in table.parts: continue
            try:
                df=pd.read_csv(table,keep_default_na=False)
                cells=int(df.shape[0]*max(df.shape[1],1))
                if cells>MAX_DUPCHECK_CELLS_PER_FILE: continue
                scanned_cells+=cells
                for col in df.columns:
                    series=pd.to_numeric(df[col].map(parse_number),errors="coerce").dropna()
                    for v in series.tolist():
                        records.append((float(v),f"TABLE:{table.relative_to(study_dir).as_posix()}"))
                        if len(records)>=MAX_DUPCHECK_VALUES_PER_STUDY: break
                    if len(records)>=MAX_DUPCHECK_VALUES_PER_STUDY: break
            except Exception: pass
            if len(records)>=MAX_DUPCHECK_VALUES_PER_STUDY: break
    records.sort(key=lambda x:x[0])
    index={"values":[x[0] for x in records],"records":records,"scanned_cells":scanned_cells,"truncated":len(records)>=MAX_DUPCHECK_VALUES_PER_STUDY}
    _STUDY_NUMERIC_CACHE[key]=index
    return index


def close_duplicate_matches(value: float, index: Dict[str,Any], limit=6) -> List[Dict[str,Any]]:
    vals=index.get("values") or []; rec=index.get("records") or []
    if not vals or value is None or not math.isfinite(float(value)): return []
    tol=max(DUPLICATE_ABS_TOL,DUPLICATE_REL_TOL*max(abs(float(value)),1.0))
    lo=bisect.bisect_left(vals,float(value)-tol); hi=bisect.bisect_right(vals,float(value)+tol)
    out=[]
    for i in range(lo,min(hi,lo+limit)):
        v,loc=rec[i]
        out.append({"matched_value":v,"difference":abs(v-float(value)),"locator":loc})
    return out


def reconcile_candidates(item: Dict[str,Any], candidates: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str,Any]]:
    if candidates.empty or not ENABLE_DUPLICATE_SCREEN:
        return candidates.copy(), {"enabled":ENABLE_DUPLICATE_SCREEN,"candidate_count":int(len(candidates)),"possible_duplicate_candidates":0}
    index=build_study_numeric_index(Path(item["study_dir"]))
    out=candidates.copy(); duplicate_count=0; evidence=[]
    for i,row in out.iterrows():
        try: value=float(row.get("value_candidate"))
        except Exception: continue
        matches=close_duplicate_matches(value,index)
        # Small integer matches are too nonspecific to be useful duplicate evidence.
        if abs(value-round(value))<1e-10 and abs(value)<=10: matches=[]
        if matches:
            duplicate_count+=1
            ev=json.dumps(matches,ensure_ascii=False)
            out.at[i,"possible_duplicate"]=True; out.at[i,"duplicate_evidence"]=ev
            evidence.append({"candidate_index":row.get("candidate_index"),"value":value,"matches":matches})
    report={"enabled":True,"candidate_count":int(len(out)),"possible_duplicate_candidates":duplicate_count,
            "numeric_index_size":len(index.get("values") or []),"numeric_index_truncated":index.get("truncated"),"evidence":evidence[:100]}
    return out,report


In [ ]:
# ============================================================
# Enhanced stage implementations; writes only to the new figure run.
# ============================================================


def stage_artifact_paths(out_dir):
    return {
        "VALIDATE_ASSET":[out_dir/"asset_validation.json",out_dir/"context"/"context.json",out_dir/"context"/"working.png",out_dir/"context"/"words_native.csv",out_dir/"context"/"drawings_pixels.json"],
        "PANEL_SEGMENTATION":[out_dir/"panels.json"],
        "AXIS_TEXT_RECOVERY":[out_dir/"context"/"words_effective.csv",out_dir/"context"/"ocr_report.json"],
        "GEOMETRY_ANALYSIS":[out_dir/"geometry_analysis.json"],
        "DIGITIZATION":[out_dir/"digitization"/"digitized_candidates.csv",out_dir/"digitization"/"digitization_summary.json",out_dir/"digitization"/"panel_reports.json"],
        "RECONCILIATION":[out_dir/"reconciliation"/"candidates_reconciled.csv",out_dir/"reconciliation"/"duplicate_check.json"],
        "SEMANTIC_STAGING":[out_dir/"staging"/"figure_staging.json",out_dir/"staging"/"candidates_staged.csv"],
        "QA_FINAL":[out_dir/"qa"/"figure_qa.json",out_dir/"qa"/"candidate_overlay.png"],
    }


def stage_is_complete(stage,out_dir):
    return all(p.is_file() and p.stat().st_size>0 for p in stage_artifact_paths(out_dir)[stage])


def _read_panels(out_dir):
    return (read_json_if_exists(out_dir/"panels.json") or {}).get("panels",[])


def run_stage(stage,item,out_dir):
    out_dir.mkdir(parents=True,exist_ok=True)
    if stage=="VALIDATE_ASSET":
        return prepare_context(item,out_dir)
    if stage=="PANEL_SEGMENTATION":
        image=load_working_image(item);panels=enhanced_segment_panels(item,image)
        save_panel_crops(image,panels,out_dir)
        payload={"figure_item_id":item["figure_item_id"],"panels":panels,"panel_count":len(panels)}
        write_json_atomic(out_dir/"panels.json",payload);return payload
    if stage=="AXIS_TEXT_RECOVERY":
        return recover_axis_text(item,out_dir)
    if stage=="GEOMETRY_ANALYSIS":
        image=load_working_image(item);panels=_read_panels(out_dir);words=effective_words(item);drawings=effective_drawings(item)
        features=image_geometry_features(image)
        classification=classify_figure(item,features,panels,words,drawings,pd.DataFrame())
        payload={"figure_item_id":item["figure_item_id"],**features,**classification,
            "ocr_report":read_json_if_exists(out_dir/"context"/"ocr_report.json"),"context":context_meta(item),
            "unit_text_candidates":extract_unit_candidates(words,item.get("caption",""))}
        write_json_atomic(out_dir/"geometry_analysis.json",payload);return payload
    if stage=="DIGITIZATION":
        analysis=read_json_if_exists(out_dir/"geometry_analysis.json") or {}
        candidates,summary=enhanced_digitize_item(item,_read_panels(out_dir),analysis)
        _atomic_write_text(out_dir/"digitization"/"digitized_candidates.csv",candidates.to_csv(index=False))
        write_json_atomic(out_dir/"digitization"/"digitization_summary.json",summary)
        write_json_atomic(out_dir/"digitization"/"panel_reports.json",summary["reports"])
        return summary
    if stage=="RECONCILIATION":
        _STUDY_NUMERIC_CACHE.clear()  # bounded memory, not an ever-growing corpus cache
        candidates=read_csv_if_exists(out_dir/"digitization"/"digitized_candidates.csv")
        candidates,report=reconcile_candidates(item,candidates)
        _atomic_write_text(out_dir/"reconciliation"/"candidates_reconciled.csv",candidates.to_csv(index=False))
        write_json_atomic(out_dir/"reconciliation"/"duplicate_check.json",report);return report
    if stage=="SEMANTIC_STAGING":
        staged=read_csv_if_exists(out_dir/"reconciliation"/"candidates_reconciled.csv")
        if not staged.empty:
            staged["eligible_for_database"]=False;staged["verification_status"]="UNVERIFIED"
            for i,row in staged.iterrows():
                role=clean_str(row.get("point_role"))
                if "CURVE_GEOMETRY" in role:target="CURVE_GEOMETRY_NOT_MEASUREMENT"
                elif "MARKER" in role:target="PLOTTED_POINT_REQUIRES_REPLICATE_DEFINITION"
                else:target=semantic_target_hint(item.get("caption",""))
                staged.at[i,"target_entity_candidate"]=target
        meta={"figure_item_id":item["figure_item_id"],"study_id":item["study_id"],"caption":item.get("caption",""),
              "candidate_count":len(staged),"eligible_for_database":False,"source_priority":"FIGURE_DIGITIZED",
              "semantic_rule":"No inferred biological n, unit, arm, mean, SD/SE/CI or outcome ontology. Explicit manual fields are preserved."}
        _atomic_write_text(out_dir/"staging"/"candidates_staged.csv",staged.reindex(columns=CANDIDATE_COLUMNS).to_csv(index=False))
        write_json_atomic(out_dir/"staging"/"figure_staging.json",meta);return meta
    if stage=="QA_FINAL":
        cand=read_csv_if_exists(out_dir/"staging"/"candidates_staged.csv")
        summary=read_json_if_exists(out_dir/"digitization"/"digitization_summary.json") or {}
        status="CANDIDATES_REVIEW_REQUIRED" if len(cand) else "NO_VALUES_MANUAL_STEP_REQUIRED"
        if len(cand) and summary.get("panels_needing_manual",0):status="PARTIAL_CANDIDATES_WITH_UNRESOLVED_PANELS"
        qa={"figure_item_id":item["figure_item_id"],"study_id":item["study_id"],"candidate_count":len(cand),
            "qa_status":status,"panels_with_candidates":summary.get("panels_with_candidates",0),
            "panels_needing_manual":summary.get("panels_needing_manual",0),"review_required":True,"eligible_for_database":False,
            "completed_at_utc":datetime.now(timezone.utc).isoformat(),"coverage_note":summary.get("coverage_note","")}
        save_candidate_overlay(item,_read_panels(out_dir),cand)
        write_json_atomic(out_dir/"qa"/"figure_qa.json",qa);return qa
    raise ValueError(f"Unknown stage: {stage}")


In [ ]:
# ============================================================
# 15. Per-figure resume from first incomplete stage
# ============================================================

def normalized_completed_stages(cp: Dict[str,Any], out_dir: Path) -> List[str]:
    completed=[]
    old=cp.get("completed_stages") if isinstance(cp,dict) else []
    if not isinstance(old,list): old=[]
    for stage in STAGE_ORDER:
        records=cp.get("stage_artifacts",{}).get(stage,[])
        intact=all((out_dir/r["path"]).is_file() and sha256_file(out_dir/r["path"])==r["sha256"] for r in records)
        if stage in old and stage_is_complete(stage,out_dir) and records and intact:
            completed.append(stage)
        else:
            break
    return completed


def process_figure_item(item: Dict[str,Any]) -> Dict[str,Any]:
    out_dir=item_output_dir(item); cp_path=item_checkpoint_path(item); sig=item_input_signature(item); cp=read_checkpoint(cp_path) or {}
    review=manual_review_for(item["figure_item_id"])

    if completed_checkpoint_current(item,cp):
        if review and clean_str(review.get("review_status")) in FINAL_REVIEW_STATUSES and not REPROCESS_VERIFIED:
            return {"figure_item_id":item["figure_item_id"],"study_id":item["study_id"],"action":"SKIP_VERIFIED_CURRENT",
                    "qa_status":clean_str((read_json_if_exists(out_dir/"qa"/"figure_qa.json") or {}).get("qa_status")),
                    "candidate_count":int((read_json_if_exists(out_dir/"qa"/"figure_qa.json") or {}).get("candidate_count") or 0),
                    "review_status":review.get("review_status"),
                    "completed_at_utc":datetime.now(timezone.utc).isoformat()}
        return {"figure_item_id":item["figure_item_id"],"study_id":item["study_id"],"action":"SKIP_CURRENT",
                "qa_status":clean_str((read_json_if_exists(out_dir/"qa"/"figure_qa.json") or {}).get("qa_status")),
                "candidate_count":int((read_json_if_exists(out_dir/"qa"/"figure_qa.json") or {}).get("candidate_count") or 0),
                "review_status":clean_str(review.get("review_status")) if review else "",
                "completed_at_utc":datetime.now(timezone.utc).isoformat()}

    if EXECUTION_MODE=="AUDIT_EXISTING":
        return {"figure_item_id":item["figure_item_id"],"study_id":item["study_id"],"action":"AUDIT_NEEDS_PROCESSING",
                "qa_status":"", "review_status":clean_str(review.get("review_status")) if review else "", "completed_at_utc":datetime.now(timezone.utc).isoformat()}

    same_generation=(cp.get("pipeline_fingerprint")==pipeline_fingerprint() and cp.get("input_signature",{}).get("signature_hash")==sig.get("signature_hash"))
    completed=normalized_completed_stages(cp,out_dir) if same_generation and not FORCE_REPROCESS and item["figure_item_id"] not in REPROCESS_ITEM_IDS else []
    if cp.get("_resume_state")=="COMPLETE" and not output_manifest_current(out_dir,cp.get("output_files")):
        completed=[]  # tampered/corrupt artifacts must be recomputed, not stamped COMPLETE again
    if review and clean_str(review.get("review_status")) in FINAL_REVIEW_STATUSES and not same_generation:
        # Do not silently trust old manual review after the underlying figure/source changed.
        stale_review=True
    else:
        stale_review=False

    state={"figure_item_id":item["figure_item_id"],"study_id":item["study_id"],"pipeline_version":FIGURE_PIPELINE_VERSION,
           "extractor_revision":FIGURE_EXTRACTOR_REVISION,"pipeline_fingerprint":pipeline_fingerprint(),"input_signature":sig,
           "completed_stages":completed,"stage_artifacts":cp.get("stage_artifacts",{}) if same_generation else {},"_resume_state":"PARTIAL","manual_review_stale_due_to_input_change":stale_review}
    write_checkpoint(cp_path,state)

    for stage in STAGE_ORDER[len(completed):]:
        last_exc=None
        for attempt in range(MAX_STAGE_RETRIES+1):
            try:
                state["current_stage"]=stage; state["stage_attempt"]=attempt+1; write_checkpoint(cp_path,state)
                run_stage(stage,item,out_dir)
                state["stage_artifacts"][stage]=[{"path":p.relative_to(out_dir).as_posix(),"sha256":sha256_file(p)} for p in stage_artifact_paths(out_dir)[stage]]
                completed.append(stage); state["completed_stages"]=completed.copy(); state["current_stage"]=""; state["stage_attempt"]=0
                write_checkpoint(cp_path,state)
                last_exc=None; break
            except KeyboardInterrupt:
                print(f"Interrupted during {item['figure_item_id']} / {stage}. Completed earlier stages are checkpointed; rerun from the top to resume.")
                raise
            except Exception as exc:
                last_exc=exc; state.update({"failed_stage":stage,"error":repr(exc),"traceback":traceback.format_exc(),"_resume_state":"PARTIAL"}); write_checkpoint(cp_path,state)
                if attempt<MAX_STAGE_RETRIES:
                    time.sleep(0.5)
        if last_exc is not None:
            print(f"FIGURE_ERROR | {item['figure_item_id']} | {stage} | {last_exc!r}")
            if FAIL_FAST: raise last_exc
            return {"figure_item_id":item["figure_item_id"],"study_id":item["study_id"],"action":"FIGURE_ERROR","qa_status":"FAIL",
                    "failed_stage":stage,"error":repr(last_exc),"review_status":clean_str(review.get("review_status")) if review else "",
                    "completed_at_utc":datetime.now(timezone.utc).isoformat()}

    qa=read_json_if_exists(out_dir/"qa"/"figure_qa.json") or {}
    state.update({"_resume_state":"COMPLETE","completed_stages":STAGE_ORDER.copy(),"current_stage":"","failed_stage":"","error":"",
                  "qa":qa,"output_files":output_manifest(out_dir),"completed_at_utc":datetime.now(timezone.utc).isoformat()})
    write_checkpoint(cp_path,state)
    return {"figure_item_id":item["figure_item_id"],"study_id":item["study_id"],"action":"PROCESSED",
            "qa_status":qa.get("qa_status",""),"candidate_count":qa.get("candidate_count",0),"review_status":clean_str(review.get("review_status")) if review else "",
            "completed_at_utc":datetime.now(timezone.utc).isoformat()}


In [ ]:
# ============================================================
# 16. Live progress reports (restored after restart)
# ============================================================

REPORT_STEM = ("pilot" if RUN_MODE.upper()=="PILOT" else "queue") + "__v1_2"
LIVE_PROGRESS_PATH = BATCH_REPORT_DIR / f"{REPORT_STEM}__LIVE_PROGRESS.csv"
REVIEW_QUEUE_PATH = BATCH_REPORT_DIR / f"{REPORT_STEM}__REVIEW_QUEUE.csv"


def restore_progress(path: Path, selected_ids: set) -> List[Dict[str,Any]]:
    latest={}
    for candidate in (path,_resume_local_path(path)):
        try:
            df=pd.read_csv(candidate,dtype=str,keep_default_na=False)
            for row in df.to_dict("records"):
                key=clean_str(row.get("figure_item_id"))
                if key in selected_ids and (key not in latest or _resume_timestamp(row)>=_resume_timestamp(latest[key])):
                    latest[key]=row
        except Exception: pass
    return list(latest.values())


def upsert_progress(rows: List[Dict[str,Any]], row: Dict[str,Any]):
    key=row.get("figure_item_id")
    for i,old in enumerate(rows):
        if old.get("figure_item_id")==key:
            rows[i]=row; return
    rows.append(row)


def write_live_progress(rows: List[Dict[str,Any]]):
    if EXECUTION_MODE!="AUDIT_EXISTING":
        _write_resume_text(LIVE_PROGRESS_PATH,pd.DataFrame(rows).to_csv(index=False))

selected_ids=set(SELECTED_ITEMS["figure_item_id"].tolist()) if not SELECTED_ITEMS.empty else set()
progress_rows=restore_progress(LIVE_PROGRESS_PATH,selected_ids)
print("Restored live progress rows:",len(progress_rows))


In [ ]:
# ============================================================
# Offline calibration editor + explicit external CSV import.
# No server, API key, ipympl or external image upload is required.
# ============================================================
import base64
import html
from IPython.display import FileLink


def _item_by_id(item_id):
    if DISCOVERED_ITEMS.empty:return None
    hit=DISCOVERED_ITEMS[DISCOVERED_ITEMS["figure_item_id"].eq(item_id)]
    return hit.iloc[0].to_dict() if not hit.empty else None


EDITOR_HTML = r'''<!doctype html><html><head><meta charset="utf-8"><title>SynerPlant calibration</title>
<style>body{font:15px system-ui,sans-serif;margin:0;background:#f4f6f8;color:#18212b}header{padding:18px 24px;background:#172b3a;color:white}main{padding:18px;display:grid;grid-template-columns:320px 1fr;gap:18px}.box{background:white;padding:16px;border:1px solid #d4dce4;border-radius:8px}button,input,select{font:inherit;margin:3px 0;padding:6px;max-width:100%;box-sizing:border-box}button{cursor:pointer}input{width:130px}label{display:block;margin-top:7px}canvas{width:100%;height:auto;border:1px solid #bac6d2;cursor:crosshair}small{display:block;color:#5a6570;margin:8px 0}.row{display:flex;gap:5px;flex-wrap:wrap}#status{white-space:pre-wrap;font-size:13px}h2{margin:0 0 8px}pre{white-space:pre-wrap;word-break:break-word}</style></head>
<body><header><b>SynerPlant | calibration and point selection</b><div>__TITLE__</div></header>
<main><div class="box"><h2>1. Panel</h2><select id="panel"></select><button id="add">Add panel</button>
<label>Kind <select id="kind"><option>auto</option><option>bar</option><option>horizontal_bar</option><option>line</option><option>scatter</option><option>manual</option><option>boxplot</option><option>heatmap</option><option>skip</option></select></label>
<div class="row"><button data-mode="tl">Plot: top-left</button><button data-mode="br">Plot: bottom-right</button></div>
<h2>2. Axis calibration</h2><small>Enter the two values printed on the axis, then click their tick marks. Y calibration uses vertical pixels; X uses horizontal pixels.</small>
<label>Y scale <select id="yscale"><option>linear</option><option>log10</option></select></label>
<div class="row"><input id="yv0" type="number" step="any" value="0"><button data-mode="y0">Click Y tick 1</button></div>
<div class="row"><input id="yv1" type="number" step="any" value="1"><button data-mode="y1">Click Y tick 2</button></div>
<label>X scale <select id="xscale"><option>linear</option><option>log10</option></select></label>
<div class="row"><input id="xv0" type="number" step="any" value="0"><button data-mode="x0">Click X tick 1</button></div>
<div class="row"><input id="xv1" type="number" step="any" value="1"><button data-mode="x1">Click X tick 2</button></div>
<h2>3. Points / endpoints</h2><label>Series <input id="series" value=""></label><label>Category <input id="label" value=""></label>
<label>Statistic <select id="stat"><option>UNSPECIFIED</option><option>MEAN</option><option>MEDIAN</option><option>Q1</option><option>Q3</option><option>WHISKER</option><option>REPLICATE_NOT_VALIDATED</option><option>CURVE_GEOMETRY_NOT_OBSERVATION</option></select></label>
<div class="row"><button data-mode="point">Add point</button><button data-mode="low">Error end 1</button><button data-mode="high">Error end 2</button><button id="undo">Undo point</button></div>
<label>Unit <input id="unit" value=""></label><label>Outcome <input id="outcome" value=""></label><label>Operator <input id="operator" value=""></label>
<small>Points replace automatic candidates for this panel. A panel with no points uses its calibration for automatic extraction. Use kind=skip for photographs/non-data panels. Error ends remain unspecified SD/SE/CI.</small>
<button id="save">Save calibration JSON</button><small>Put the downloaded JSON into manual_calibrations, or call import_calibration(path) in the notebook. This does NOT verify the numbers.</small><pre id="status"></pre></div>
<div class="box"><canvas id="canvas"></canvas><div id="info"></div><small>Blue: plot rectangle. Red: selected points. Coordinates refer to the embedded working image, not the resized browser view. Nothing is uploaded.</small></div></main>
<script>
const data=__DATA__;const image=new Image();image.src='data:image/png;base64,__IMAGE__';
const $=id=>document.getElementById(id);const canvas=$('canvas'),ctx=canvas.getContext('2d');let active=0,mode='point';
function p(){return data.panels[active]}
function status(s){$('status').textContent=s}
function refreshList(){let sel=$('panel');sel.innerHTML='';data.panels.forEach((v,i)=>{let o=document.createElement('option');o.value=i;o.textContent=v.panel_id;sel.appendChild(o)});sel.value=active;loadFields()}
function loadFields(){let a=p();$('kind').value=a.kind||'auto';$('unit').value=a.unit||'';$('outcome').value=a.outcome||'';for(let k of ['x','y']){let z=a[k+'_axis'];if(z&&z.scale!=='piecewise'){$(k+'scale').value=z.scale||'linear';if(z.ticks&&z.ticks.length){$(k+'v0').value=z.ticks[0].value;$(k+'v1').value=z.ticks[z.ticks.length-1].value}}}draw()}
function fields(){p().kind=$('kind').value;p().unit=$('unit').value;p().outcome=$('outcome').value}
function draw(){if(!image.complete||!image.naturalWidth)return;canvas.width=image.naturalWidth;canvas.height=image.naturalHeight;ctx.drawImage(image,0,0);let a=p(),b=a.plot_bbox_px;if(b){ctx.strokeStyle='#195fc0';ctx.lineWidth=2;ctx.strokeRect(b[0],b[1],b[2]-b[0],b[3]-b[1])}ctx.strokeStyle='#dc3025';ctx.fillStyle='#dc3025';for(let q of a.points||[]){ctx.beginPath();ctx.arc(q.x_px,q.y_px,4,0,Math.PI*2);ctx.stroke()}$('info').textContent=a.panel_id+' | mode: '+mode+' | selected points: '+(a.points||[]).length}
image.onload=draw;$('panel').onchange=()=>{fields();active=Number($('panel').value);loadFields()};$('add').onclick=()=>{fields();active=data.panels.length;data.panels.push({panel_id:'P'+String(active+1).padStart(2,'0'),kind:'auto',plot_bbox_px:[0,0,image.naturalWidth,image.naturalHeight],points:[]});refreshList()};
document.querySelectorAll('[data-mode]').forEach(b=>b.onclick=()=>{mode=b.dataset.mode;draw()});$('undo').onclick=()=>{p().points=(p().points||[]);p().points.pop();draw()};
canvas.onclick=e=>{fields();let rect=canvas.getBoundingClientRect(),x=(e.clientX-rect.left)*canvas.width/rect.width,y=(e.clientY-rect.top)*canvas.height/rect.height,a=p();if(!a.plot_bbox_px)a.plot_bbox_px=[0,0,canvas.width,canvas.height];
if(mode==='tl'){a.plot_bbox_px[0]=x;a.plot_bbox_px[1]=y}else if(mode==='br'){a.plot_bbox_px[2]=x;a.plot_bbox_px[3]=y}else if(['y0','y1','x0','x1'].includes(mode)){let ax=mode[0],i=Number(mode[1]);let val=Number($(ax+'v'+i).value);if(!Number.isFinite(val)){status('Enter a finite axis value');return};if(!a[ax+'_axis']||!Array.isArray(a[ax+'_axis'].ticks)||a[ax+'_axis'].ticks.length!==2)a[ax+'_axis']={scale:$(ax+'scale').value,ticks:[null,null]};a[ax+'_axis'].scale=$(ax+'scale').value;a[ax+'_axis'].ticks[i]={pixel:ax==='y'?y:x,value:val}}else if(mode==='point'){a.points=a.points||[];a.points.push({x_px:x,y_px:y,series:$('series').value,label:$('label').value,statistic:$('stat').value})}else{let q=(a.points||[]).at(-1);if(!q){status('Select the central point first');return};q[mode+(a.kind==='horizontal_bar'?'_x_px':'_y_px')]=a.kind==='horizontal_bar'?x:y}status('Last click: '+x.toFixed(2)+', '+y.toFixed(2));draw()};
$('save').onclick=()=>{fields();if(!$('operator').value.trim()){status('Enter the operator name');return};for(let a of data.panels){let b=a.plot_bbox_px;if(!b||!(b[0]<b[2]&&b[1]<b[3])){status('Invalid plot rectangle: '+a.panel_id);return};for(let k of ['x_axis','y_axis'])if(a[k]&&(a[k].ticks||[]).some(v=>v===null)){status('Complete both tick clicks for '+a.panel_id+' '+k);return}}data.operator=$('operator').value.trim();data.saved_at=new Date().toISOString();const blob=new Blob([JSON.stringify(data,null,2)],{type:'application/json'});const a=document.createElement('a');a.href=URL.createObjectURL(blob);a.download='__FILENAME__';a.click();URL.revokeObjectURL(a.href);status('Calibration saved locally. Import it in the notebook and reprocess this figure.')};refreshList();
</script></body></html>'''


def export_manual_editor(figure_item_id):
    item=_item_by_id(figure_item_id)
    if item is None:raise KeyError("Unknown figure_item_id")
    meta=context_meta(item)
    if not meta or not working_image_path(item).exists():raise RuntimeError("Run asset validation for this figure first")
    existing=load_calibration(item)
    if existing:data=existing
    else:
        reports=(read_json_if_exists(item_output_dir(item)/"digitization"/"digitization_summary.json") or {}).get("reports",[])
        data={"schema_version":1,"figure_item_id":figure_item_id,"working_image_sha256":meta["working_image_sha256"],"panels":[]}
        for panel in _read_panels(item_output_dir(item)):
            cfg={"panel_id":panel["panel_id"],"plot_bbox_px":panel.get("plot_bbox_px") or [0,0,meta["width_px"],meta["height_px"]],"kind":"auto","points":[]}
            report=next((r for r in reports if r.get("panel_id")==panel["panel_id"]),{})
            for axis in ["x","y"]:
                cal=report.get(axis+"_calibration",{})
                if cal.get("ok") and cal.get("scale") in {"linear","log10"}:
                    cfg[axis+"_axis"]={"scale":cal["scale"],"multiplier":cal.get("multiplier",1.),"ticks":[{"pixel":t["pixel"],"value":t["value"]} for t in cal["ticks"]]}
            data["panels"].append(cfg)
    if not data.get("panels"):
        data["panels"]=[{"panel_id":"P01","plot_bbox_px":[0,0,meta["width_px"],meta["height_px"]],"kind":"manual","points":[]}]
    encoded=base64.b64encode(working_image_path(item).read_bytes()).decode("ascii")
    title=figure_item_id+" | "+clean_str(item.get("caption"))[:260]
    data_json=json.dumps(data,ensure_ascii=True).replace("</","<\\/")
    content=EDITOR_HTML.replace("__TITLE__",html.escape(title)).replace("__DATA__",data_json).replace("__IMAGE__",encoded).replace("__FILENAME__",calibration_path(item).name)
    path=EDITOR_DIR/(calibration_path(item).stem+".html")
    _atomic_write_text(path,content)
    print("Open this local HTML file in your browser:",path)
    display(FileLink(str(path)))
    return path


def import_calibration(path):
    source=Path(path)
    data=json.loads(source.read_text(encoding="utf-8-sig"))
    item=_item_by_id(data.get("figure_item_id",""))
    if item is None:raise ValueError("Calibration refers to an undiscovered figure")
    if data.get("working_image_sha256")!=context_meta(item).get("working_image_sha256"):
        raise ValueError("Calibration image hash does not match the current working image")
    ids=[]
    for p in data.get("panels",[]):
        if not isinstance(p.get("panel_id"),str) or not p["panel_id"]:raise ValueError("Each panel needs a stable panel_id")
        ids.append(p["panel_id"])
        box=p.get("plot_bbox_px",[]);meta=context_meta(item)
        if len(box)!=4 or not all(math.isfinite(float(v)) for v in box):raise ValueError("Invalid plot rectangle")
        if not (0<=box[0]<box[2]<=meta["width_px"] and 0<=box[1]<box[3]<=meta["height_px"]):raise ValueError("Rectangle outside working image")
        for axis in ["x_axis","y_axis"]:
            if p.get(axis) and not manual_axis_calibration(p[axis]).get("ok"):raise ValueError(f"Invalid {axis} for {p['panel_id']}")
        if p.get("points") and not p.get("x_axis" if p.get("kind")=="horizontal_bar" else "y_axis"):
            raise ValueError("Selected points require a value-axis calibration")
        for q in p.get("points",[]):
            if not _inside_plot(float(q["x_px"]),float(q["y_px"]),box):raise ValueError("Selected point outside plot rectangle")
    if len(ids)!=len(set(ids)):raise ValueError("Duplicate panel IDs")
    destination=calibration_path(item)
    if destination.exists():
        backup=CALIBRATION_DIR/"history"/(destination.stem+"__"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")+".json")
        _atomic_write_text(backup,destination.read_text(encoding="utf-8"))
    write_json_atomic(destination,data)
    print("Imported:",destination,"\nReprocess only:",item["figure_item_id"])
    return item["figure_item_id"]


def import_digitized_csv(figure_item_id,path,*,value_column,x_column=None,series_column=None,panel_id="EXTERNAL",unit="",statistic="UNSPECIFIED",operator="",
                         digitization_method="EXPLICIT_EXTERNAL_CSV",external_tool="",external_tool_version="",project_file="",axis_type="",calibration_notes=""):
    if not operator.strip():raise ValueError("Record the operator for an external import")
    item=_item_by_id(figure_item_id)
    if item is None:raise KeyError("Unknown figure")
    source=Path(path);df=pd.read_csv(source,keep_default_na=False)
    for col in [value_column,x_column,series_column]:
        if col and col not in df:raise ValueError(f"CSV column not found: {col}")
    digest=sha256_file(source);project_digest=""
    if project_file:
        p=Path(project_file)
        if not p.is_file():raise ValueError("Saved digitizer project file does not exist")
        project_digest=sha256_file(p)
    rows=[]
    for i,r in df.iterrows():
        value=parse_number(r[value_column]);xvalue=parse_number(r[x_column]) if x_column else None
        if value is None or (x_column and xvalue is None):raise ValueError(f"Ambiguous/non-numeric CSV value at data row {i+1}")
        rows.append({"value_candidate":value,"x_value_candidate":xvalue,"series_candidate":clean_str(r[series_column]) if series_column else "",
            "panel_id":panel_id,"unit":unit,"statistic":statistic,"file_sha256":digest,"source_file":str(source),"external_row":int(i+1),
            "digitization_method":digitization_method,"external_tool":external_tool,"external_tool_version":external_tool_version,
            "external_project_sha256":project_digest,"external_operator":operator,"external_axis_type":axis_type,
            "external_calibration_notes":calibration_notes})
    data=load_calibration(item) or {"schema_version":1,"figure_item_id":figure_item_id,"working_image_sha256":context_meta(item)["working_image_sha256"],"panels":[]}
    # Idempotent replacement by source-file hash/path; do not append duplicates after restart.
    data["imported_rows"]=[r for r in data.get("imported_rows",[]) if r.get("source_file")!=str(source)]+rows
    data["operator"]=operator;data["saved_at"]=datetime.now(timezone.utc).isoformat()
    write_json_atomic(calibration_path(item),data)
    print("Imported unverified CSV rows:",len(rows),"; method:",digitization_method,"; reprocess figure:",figure_item_id)
    return len(rows)


def _copy_evidence_file(source, destination_dir, stem):
    source=Path(source)
    if not source.is_file():raise ValueError(f"Evidence file does not exist: {source}")
    destination_dir.mkdir(parents=True,exist_ok=True)
    digest=sha256_file(source)
    dest=destination_dir/(stem+"__"+digest[:12]+source.suffix.lower())
    if not dest.exists():shutil.copy2(source,dest)
    if sha256_file(dest)!=digest:raise RuntimeError("Evidence copy hash mismatch")
    return dest,digest


def import_webplotdigitizer_csv(figure_item_id,csv_path,*,value_column,x_column=None,series_column=None,panel_id="EXTERNAL",
                                unit="",statistic="UNSPECIFIED",operator,wpd_version,project_path,axis_type="",
                                calibration_notes=""):
    """Import WebPlotDigitizer output as UNVERIFIED candidates and preserve the WPD project as audit evidence."""
    if not clean_str(wpd_version):raise ValueError("Record the WebPlotDigitizer version")
    if REQUIRE_WPD_PROJECT_FILE and not project_path:raise ValueError("A saved WebPlotDigitizer project file is required")
    item=_item_by_id(figure_item_id)
    if item is None:raise KeyError("Unknown figure")
    evidence_dir=WPD_RESULTS_DIR/safe_name(figure_item_id,120)
    csv_copy,csv_sha=_copy_evidence_file(csv_path,evidence_dir,"digitized_values")
    project_copy=None;project_sha=""
    if project_path:
        project_copy,project_sha=_copy_evidence_file(project_path,evidence_dir,"webplotdigitizer_project")
    meta={"figure_item_id":figure_item_id,"tool":WPD_TOOL_NAME,"tool_version":wpd_version,"operator":operator,
          "csv_sha256":csv_sha,"project_sha256":project_sha,"axis_type":axis_type,"calibration_notes":calibration_notes,
          "working_image_sha256":context_meta(item).get("working_image_sha256",""),"imported_at_utc":datetime.now(timezone.utc).isoformat()}
    write_json_atomic(evidence_dir/"IMPORT_METADATA.json",meta)
    return import_digitized_csv(figure_item_id,csv_copy,value_column=value_column,x_column=x_column,series_column=series_column,
        panel_id=panel_id,unit=unit,statistic=statistic,operator=operator,digitization_method="FIGURE_DIGITIZATION_WEBPLOTDIGITIZER",
        external_tool=WPD_TOOL_NAME,external_tool_version=wpd_version,project_file=project_copy or "",axis_type=axis_type,
        calibration_notes=calibration_notes)


def _wpd_item_unresolved(item):
    out=item_output_dir(item);cp=read_checkpoint(item_checkpoint_path(item)) or {}
    if completed_checkpoint_current(item,cp,respect_reprocess=False):
        summary=read_json_if_exists(out/"digitization"/"digitization_summary.json") or {}
        return bool(summary.get("panels_needing_manual",0) or summary.get("candidate_count",0)==0)
    return True


def export_webplotdigitizer_queue(*,only_unresolved=True,max_items=None,include_reconcile_first=False):
    """Create a deterministic WPD work package; no values are accepted by this step."""
    rows=[]
    if include_reconcile_first:
        items=DISCOVERED_ITEMS.copy()
        if RUN_MODE.upper()=="PILOT":
            wanted={clean_str(x) for x in PILOT_STUDY_IDS if clean_str(x)}
            items=items[items["study_id"].isin(wanted)].copy()
    else:
        items=SELECTED_ITEMS.copy()
    if only_unresolved:
        items=items[[ _wpd_item_unresolved(r) for r in items.to_dict("records") ]].copy()
    if max_items is not None:items=items.head(int(max_items))
    for item in items.to_dict("records"):
        source=working_image_path(item) if working_image_path(item).is_file() else Path(item.get("image_path",""))
        if not source.is_file():continue
        item_dir=WPD_INPUT_DIR/safe_name(item["figure_item_id"],120);item_dir.mkdir(parents=True,exist_ok=True)
        image_dest=item_dir/("working_image"+source.suffix.lower())
        if not image_dest.exists() or sha256_file(image_dest)!=sha256_file(source):shutil.copy2(source,image_dest)
        panel_dir=item_output_dir(item)/"panels"
        copied_panels=[]
        if panel_dir.is_dir():
            dest_panels=item_dir/"panels";dest_panels.mkdir(exist_ok=True)
            for p in sorted(panel_dir.glob("*.png")):
                d=dest_panels/p.name
                if not d.exists() or sha256_file(d)!=sha256_file(p):shutil.copy2(p,d)
                copied_panels.append(str(d))
        meta={k:item.get(k,"") for k in ["figure_item_id","study_id","screening_class","source_file_id","source_file_name","source_role",
             "figure_index","source_figure_label","page_no","caption","queue_source","queue_reason","master_gap_status","digitization_decision",
             "priority_tier","existing_supported_measurement_count","existing_supported_measurement_ids"]}
        meta.update({"working_image":str(image_dest),"working_image_sha256":sha256_file(image_dest),"panel_images":copied_panels,
                     "wpd_project_required":bool(REQUIRE_WPD_PROJECT_FILE),"candidate_only":True})
        write_json_atomic(item_dir/"SOURCE_METADATA.json",meta)
        (item_dir/"caption.txt").write_text(clean_str(item.get("caption")),encoding="utf-8")
        rows.append({**meta,"wpd_input_dir":str(item_dir),"wpd_image":str(image_dest),
                     "save_project_as":str(WPD_RESULTS_DIR/safe_name(item["figure_item_id"],120)/"project.tar"),
                     "export_csv_to":str(WPD_RESULTS_DIR/safe_name(item["figure_item_id"],120)/"values.csv")})
    manifest=pd.DataFrame(rows)
    _write_resume_text(WPD_ROOT/"WPD_WORK_QUEUE.csv",manifest.to_csv(index=False))
    readme="""SynerPlant WebPlotDigitizer work package\n\n1. Work only on rows in WPD_WORK_QUEUE.csv.\n2. Prefer panel crops where available; otherwise use working_image.\n3. Calibrate the axes from printed tick labels. Record linear/log axis choice.\n4. Save the WebPlotDigitizer project (.tar) AND export the digitized values CSV.\n5. Do not infer SD/SE/CI from error-bar geometry unless the caption/text defines it.\n6. Import with import_webplotdigitizer_csv(...). The import remains UNVERIFIED until candidate review.\n7. Keep the project and CSV hashes: they are part of the provenance trail.\n"""
    _write_resume_text(WPD_ROOT/"README_WEBPLOTDIGITIZER.txt",readme)
    print("WPD queue rows:",len(manifest),"|",WPD_ROOT/"WPD_WORK_QUEUE.csv")
    return manifest


def reprocess_one(figure_item_id):
    item=_item_by_id(figure_item_id)
    if item is None:raise KeyError("Unknown figure")
    with project_run_lock():
        result=process_figure_item(item)
        upsert_progress(progress_rows,result);write_live_progress(progress_rows)
    print(result)
    print("Rerun the diagnostics/export cells to refresh aggregate CSV files.")
    return result


In [ ]:
# Explicit coverage diagnostics. COMPLETE means execution completed, not that
# every scientific datum in an image was recovered.
ALL_CANDIDATES_PATH=FIGURE_OUTPUT_DIR/"ALL_FIGURE_CANDIDATES.csv"
DIAGNOSTICS_PATH=FIGURE_OUTPUT_DIR/"FIGURE_DIAGNOSTICS.csv"
PANEL_DIAGNOSTICS_PATH=FIGURE_OUTPUT_DIR/"PANEL_DIAGNOSTICS.csv"
CANDIDATE_REVIEWS_PATH=FIGURE_OUTPUT_DIR/"CANDIDATE_REVIEWS.csv"
REPORT_COLUMNS=["figure_item_id","study_id","screening_class","source_file_id","figure_index","source_figure_label","page_no",
    "queue_source","queue_reason","master_gap_status","digitization_decision","priority_tier",
    "existing_supported_measurement_count","existing_supported_measurement_ids","status","failed_stage","error",
    "candidate_count","panel_count","panels_with_candidates","panels_needing_manual","ocr_status","ocr_backend",
    "native_tokens","ocr_tokens","image_source","blocking_reasons","figure_output_dir","working_image_path",
    "overlay_path","manual_calibration_path"]
PANEL_REPORT_COLUMNS=["figure_item_id","study_id","panel_id","status","reason","candidate_count","x_calibration","y_calibration","attempts"]


def collect_current_candidates():
    rows=[]
    for item in SELECTED_ITEMS.to_dict("records"):
        cp=read_checkpoint(item_checkpoint_path(item)) or {}
        if not completed_checkpoint_current(item,cp,respect_reprocess=False):continue
        frame=read_csv_if_exists(item_output_dir(item)/"staging"/"candidates_staged.csv")
        for rec in frame.to_dict("records"):
            rec.update(generation_signature=cp["input_signature"]["signature_hash"],
                generation_fingerprint=cp["pipeline_fingerprint"],generation_completed_at=cp["completed_at_utc"])
            rows.append(rec)
    return pd.DataFrame(rows).reindex(columns=CANDIDATE_COLUMNS+["generation_signature","generation_fingerprint","generation_completed_at"])


def build_run_reports():
    candidates=collect_current_candidates();diagnostics=[];panel_rows=[]
    for item in SELECTED_ITEMS.to_dict("records"):
        out=item_output_dir(item);cp=read_checkpoint(item_checkpoint_path(item)) or {}
        current=completed_checkpoint_current(item,cp,respect_reprocess=False)
        meta=context_meta(item);ocr=read_json_if_exists(out/"context"/"ocr_report.json") or {}
        summary=read_json_if_exists(out/"digitization"/"digitization_summary.json") or {}
        count=len(candidates[candidates["figure_item_id"].eq(item["figure_item_id"])])
        if current:state="CURRENT_WITH_CANDIDATES" if count else "CURRENT_ZERO_CANDIDATES"
        elif cp.get("failed_stage") or cp.get("error"):state="FAILED"
        elif cp:state="INCOMPLETE"
        else:state="NOT_RUN"
        diagnostics.append(dict(figure_item_id=item["figure_item_id"],study_id=item["study_id"],screening_class=item.get("screening_class",""),
            source_file_id=item.get("source_file_id",""),figure_index=item["figure_index"],source_figure_label=item.get("source_figure_label",""),
            page_no=item.get("page_no"),queue_source=item.get("queue_source",""),queue_reason=item.get("queue_reason",""),
            master_gap_status=item.get("master_gap_status",""),digitization_decision=item.get("digitization_decision",""),priority_tier=item.get("priority_tier",""),
            existing_supported_measurement_count=item.get("existing_supported_measurement_count",0),
            existing_supported_measurement_ids=item.get("existing_supported_measurement_ids",""),status=state,
            failed_stage=cp.get("failed_stage",""),error=cp.get("error",""),candidate_count=count,
            panel_count=summary.get("panel_count",0) if current else 0,panels_with_candidates=summary.get("panels_with_candidates",0) if current else 0,
            panels_needing_manual=summary.get("panels_needing_manual",0) if current else 0,ocr_status=ocr.get("status","NOT_REACHED"),
            ocr_backend=ocr.get("backend",OCR_EFFECTIVE_BACKEND),native_tokens=meta.get("native_word_count",0),
            ocr_tokens=ocr.get("token_count",0),image_source=meta.get("image_source",""),
            blocking_reasons="; ".join(summary.get("blocking_reasons",[])) if current else ("EXECUTION_NOT_CURRENT" if cp else "NOT_RUN"),
            figure_output_dir=str(out),working_image_path=str(working_image_path(item)),overlay_path=str(out/"qa"/"candidate_overlay.png"),
            manual_calibration_path=str(calibration_path(item))))
        if current:
            for rec in summary.get("reports",[]):panel_rows.append({"figure_item_id":item["figure_item_id"],"study_id":item["study_id"],**rec})
    figures=pd.DataFrame(diagnostics,columns=REPORT_COLUMNS)
    panels=pd.DataFrame(panel_rows).reindex(columns=PANEL_REPORT_COLUMNS)
    for column in ["x_calibration","y_calibration","attempts"]:
        if column in panels:panels[column]=panels[column].map(lambda x:json.dumps(jsonable(x),ensure_ascii=False))
    if EXECUTION_MODE!="AUDIT_EXISTING":
        _write_resume_text(ALL_CANDIDATES_PATH,candidates.to_csv(index=False))
        _write_resume_text(DIAGNOSTICS_PATH,figures.to_csv(index=False))
        _write_resume_text(PANEL_DIAGNOSTICS_PATH,panels.to_csv(index=False))
        _write_resume_text(REVIEW_QUEUE_PATH,figures.to_csv(index=False))
    print("Selected queue items:",len(figures))
    print("Figures with current candidates:",int((figures['candidate_count']>0).sum()))
    print("Failed:",int(figures['status'].eq('FAILED').sum()),"| Incomplete:",int(figures['status'].eq('INCOMPLETE').sum()),"| Not run:",int(figures['status'].eq('NOT_RUN').sum()))
    print("ALL unverified candidate rows:",len(candidates))
    print("Detected panels needing manual work:",int(pd.to_numeric(figures['panels_needing_manual'],errors='coerce').fillna(0).sum()))
    print("These are coverage indicators, not a claim of complete or validated scientific extraction.")
    print("Candidates:",ALL_CANDIDATES_PATH,"\nFigures:",DIAGNOSTICS_PATH,"\nPanels:",PANEL_DIAGNOSTICS_PATH)
    return candidates,figures,panels


REVIEW_COLUMNS=["candidate_id","figure_item_id","review_status","reviewer","reviewed_at","generation_signature",
    "generation_fingerprint","generation_completed_at","experiment_id","arm_id","outcome_standard_name","unit",
    "statistic","uncertainty_type","target_entity","notes"]


def load_candidate_reviews():
    if not CANDIDATE_REVIEWS_PATH.is_file():return pd.DataFrame(columns=REVIEW_COLUMNS)
    frame=pd.read_csv(CANDIDATE_REVIEWS_PATH,dtype=str,keep_default_na=False)
    return frame.reindex(columns=REVIEW_COLUMNS).fillna("")


def review_candidate(candidate_id,*,reviewer,status="ACCEPTED",experiment_id="",arm_id="",outcome_standard_name="",
                     unit="",statistic="",uncertainty_type="",target_entity="",notes=""):
    if EXECUTION_MODE=="AUDIT_EXISTING":raise RuntimeError("Read-only mode")
    if not reviewer.strip():raise ValueError("A named reviewer is required")
    if status not in {"ACCEPTED","VERIFIED","REJECTED","NEEDS_WORK"}:raise ValueError("Invalid review status")
    current=collect_current_candidates();hits=current[current["candidate_id"].eq(candidate_id)]
    if len(hits)!=1:raise ValueError("Candidate is absent, stale or non-unique; rerun reports and use its current candidate_id")
    row=hits.iloc[0].to_dict()
    rec=dict(candidate_id=candidate_id,figure_item_id=row["figure_item_id"],review_status=status,reviewer=reviewer,
        reviewed_at=datetime.now(timezone.utc).isoformat(),generation_signature=row["generation_signature"],
        generation_fingerprint=row["generation_fingerprint"],generation_completed_at=row["generation_completed_at"],
        experiment_id=experiment_id,arm_id=arm_id,outcome_standard_name=outcome_standard_name,unit=unit,
        statistic=statistic,uncertainty_type=uncertainty_type,target_entity=target_entity,notes=notes)
    reviews=load_candidate_reviews();reviews=reviews[~reviews['candidate_id'].eq(candidate_id)]
    reviews=pd.concat([reviews,pd.DataFrame([rec])],ignore_index=True).reindex(columns=REVIEW_COLUMNS)
    with project_run_lock():_write_resume_text(CANDIDATE_REVIEWS_PATH,reviews.to_csv(index=False))
    print("Recorded review of this candidate only:",candidate_id)
    return rec


def build_verified_export():
    candidates=collect_current_candidates();reviews=load_candidate_reviews();rows=[]
    for review in reviews.to_dict("records"):
        if review["review_status"] not in {"ACCEPTED","VERIFIED"}:continue
        hit=candidates[candidates["candidate_id"].eq(review["candidate_id"])]
        if len(hit)!=1:continue
        candidate=hit.iloc[0].to_dict()
        if any(clean_str(candidate[k])!=review[k] for k in ["generation_signature","generation_fingerprint","generation_completed_at"]):continue
        geometry_only="CURVE_GEOMETRY" in clean_str(candidate.get("point_role"))
        mapped=all(clean_str(review[k]) for k in ["experiment_id","arm_id","outcome_standard_name","unit","statistic","target_entity"])
        is_wpd=clean_str(candidate.get("digitization_method"))=="FIGURE_DIGITIZATION_WEBPLOTDIGITIZER"
        wpd_trace_ok=(not is_wpd) or (bool(clean_str(candidate.get("external_file_sha256"))) and
            bool(clean_str(candidate.get("external_project_sha256"))) and bool(clean_str(candidate.get("external_operator"))) and
            bool(clean_str(candidate.get("external_tool_version"))) and bool(clean_str(candidate.get("external_axis_type"))))
        eligible=bool(mapped and not geometry_only and wpd_trace_ok and review["target_entity"] in {"measurements","raw_observations","contrasts"})
        rows.append({**candidate,**{"review_"+k:v for k,v in review.items() if k not in {"candidate_id","figure_item_id"}},
                     "eligible_for_database":eligible})
    cols=list(candidates.columns)+["review_"+k for k in REVIEW_COLUMNS if k not in {"candidate_id","figure_item_id"}]
    result=pd.DataFrame(rows).reindex(columns=list(dict.fromkeys(cols)))
    if EXECUTION_MODE!="AUDIT_EXISTING":_write_resume_text(VERIFIED_EXPORT_PATH,result.to_csv(index=False))
    print("Verified/accepted candidate rows:",len(result))
    print("No master database was written. Numeric acceptance and semantic mapping are separate checks.")
    print("Verified local export:",VERIFIED_EXPORT_PATH)
    return result


## Run queue-aligned figures

Only upstream digitization/review-queue items selected by the master-reference reconciliation are processed. `RECONCILE_FIRST` items are excluded from automatic digitization unless explicitly enabled. Stop the cell to interrupt; restart from the top to resume.


In [ ]:
# ============================================================
# 17. Run selected figures; preserve progress after every figure
# ============================================================

if SELECTED_ITEMS.empty:
    print("No figure candidates selected.")
else:
    try:
        with project_run_lock():
            for pos,item in enumerate(SELECTED_ITEMS.to_dict("records"),1):
                print(f"\n{EXECUTION_MODE}: {pos}/{len(SELECTED_ITEMS)} | {item['figure_item_id']}")
                try:
                    result=process_figure_item(item)
                except KeyboardInterrupt:
                    print("Run interrupted. Restart this notebook from the top; the current figure resumes from its first incomplete stage.")
                    raise
                except Exception as exc:
                    traceback.print_exc()
                    result={"figure_item_id":item["figure_item_id"],"study_id":item["study_id"],"action":"FATAL_EXCEPTION","qa_status":"FAIL",
                            "error":repr(exc),"completed_at_utc":datetime.now(timezone.utc).isoformat()}
                    if FAIL_FAST: raise
                upsert_progress(progress_rows,result); write_live_progress(progress_rows); gc.collect()
    except KeyboardInterrupt:
        pass

print("Queue-aligned figure loop ended. This does NOT imply successful digitization.")
print("Read the diagnostics below: failures, zero-candidate figures, and extracted rows are reported separately.")
print("Live progress:", LIVE_PROGRESS_PATH)


## Coverage, queue alignment and candidates

Inspect `QUEUE_ALIGNMENT.csv`, figure diagnostics and candidate overlays before accepting anything. `NOT_RUN` after interruption is not an extraction failure.


In [ ]:
all_candidates_df, figure_diagnostics_df, panel_diagnostics_df = build_run_reports()
if not figure_diagnostics_df.empty:
    display(figure_diagnostics_df.head(30))
if not all_candidates_df.empty:
    display(all_candidates_df.head(30))

## Export unresolved work to WebPlotDigitizer

This creates a deterministic work package only for unresolved queue items. It does **not** accept or write any values to the master database.

In [ ]:
wpd_queue_df = export_webplotdigitizer_queue(only_unresolved=True)
if not wpd_queue_df.empty:
    display(wpd_queue_df[["figure_item_id","study_id","source_figure_label","priority_tier","master_gap_status","wpd_image"]].head(30))


## WebPlotDigitizer / manual recovery of unresolved panels

The preferred manual fallback is WebPlotDigitizer with a saved project file. The work-package cell below copies the exact working image/panel crops and records their hashes. Imported WPD CSV values remain unverified candidates.


In [ ]:
# Local calibration editor for a panel that only needs axis/point clicks:
# figure_id = figure_diagnostics_df.iloc[0]["figure_item_id"]
# export_manual_editor(figure_id)
# import_calibration(r"C:/Users/lenovo/Downloads/<saved_calibration>.json")
# reprocess_one(figure_id)

# Preferred external fallback: WebPlotDigitizer. Save BOTH the WPD project and exported CSV.
# import_webplotdigitizer_csv(
#     figure_id,
#     r"C:/Users/lenovo/Downloads/values.csv",
#     value_column="Y", x_column="X", series_column="series", panel_id="P01",
#     unit="<unit printed in source>", statistic="MEAN",
#     operator="<name>", wpd_version="<WebPlotDigitizer version>",
#     project_path=r"C:/Users/lenovo/Downloads/project.tar",
#     axis_type="2D_XY_LINEAR", calibration_notes="Ticks checked against source figure"
# )
# reprocess_one(figure_id)
# all_candidates_df, figure_diagnostics_df, panel_diagnostics_df = build_run_reports()


## Candidate-level acceptance

`VERIFIED_FIGURE_CANDIDATES.csv` stays empty until explicit candidate review. For WebPlotDigitizer candidates, the saved WPD project hash, CSV hash, operator and version are required for database eligibility. Never bulk-accept unknown series, units or uncertainty definitions.


In [ ]:
# Review one exact candidate from all_candidates_df:
# review_candidate("<candidate_id>", reviewer="<name>", status="ACCEPTED",
#     experiment_id="...", arm_id="...", outcome_standard_name="...",
#     unit="...", statistic="MEAN", target_entity="measurements", notes="Checked against source")
verified_df = build_verified_export()
if not verified_df.empty:
    display(verified_df.head(30))


## Method limits

The queue is provenance-first: exact machine-readable/table/text values outrank digitized pixels. OCR and geometry extraction remain conservative and can miss valid data. Automatic candidates and WebPlotDigitizer imports are both reviewable evidence, not ground truth. Error-bar geometry must not be labelled SD/SE/CI unless supported by caption/text. Dense curve pixels are not independent biological observations.
